# Final-Boss DPS-C Primacy Experiment — AGNews + TREC + Yahoo

This notebook performs a **single cross-task DPS-C analysis** using the same Gemma2-9B-IT / Gemma Scope layer-20 residual SAE across:

- **AGNews**: 4 labels
- **TREC**: 6 coarse labels
- **Yahoo Answers Topics**: 10 labels

Hence each SAE feature is evaluated against **20 task/label-specific DPS-C equations**.

For task \(d\), query label \(\ell\), feature \(f\), and varied-demo position \(p\),

\[
\Delta_{d,\ell,f}^{(p)}
=
A_{d,f}^{(p)}(\ell \rightarrow \ell)
-
\max_{c\neq \ell} A_{d,f}^{(p)}(c\rightarrow \ell).
\]

The equation is satisfied iff

\[
\Delta_{d,\ell,f}^{(p)} > \varepsilon,
\]

with a strict tolerance `DPS_TIE_TOL`. This excludes exact ties.

## Experimental discipline

1. **The same fixed context and same variable example for each label are reused across positions for each query.**
2. Feature discovery and evaluation are separated:
   - **discovery split**: used to define a feature set.
   - **holdout split**: used for position-specificity / primacy evaluation.
3. No top-200 cap is used. Aggregate analyses always use the **entire selected feature population**.
4. The main primacy plot reports:
   - how many position-1-derived features still satisfy at least the requested number of equations at each position;
   - the **mean signed activation gap of the fixed position-1 feature set** at each position, avoiding survivorship bias.
5. Per-task counts/gaps are also reported because the 20 equations are not statistically independent and Yahoo contributes more equations than the other tasks.

The default representation is **SAE encoder preactivation**, matching the supplied TREC/Yahoo experiment notebooks. Set `USE_SAE_PREACTIVATIONS=False` to analyze thresholded sparse activations instead.


### TREC query-source note

The official TREC test split contains only 9 Abbreviation examples. To avoid
making the six-class TREC analysis depend on this small-class bottleneck, the
camera-ready version samples TREC final queries from the original training
split. All selected TREC final-query examples are globally reserved and
excluded from every demonstration pool, so query and demonstration examples
remain strictly disjoint.

### Gemma Scope configuration

This version uses `google/gemma-2-9b-it` together with the official
**Gemma Scope 9B instruction-tuned residual-stream SAE** at layer
20, width 16K, `average_l0_91` from
`google/gemma-scope-9b-it-res`.

### DPS-C criterion

For DPS-C features, the matched condition is the **minimum**:

\[
A_f(\mathcal{S}^{N}_{\ell\to\ell})
<
\min_{k\neq\ell}
A_f(\mathcal{S}^{N}_{k\to\ell}).
\]

Internally, this notebook stores the positive DPS-C gap

\[
\min_{k\neq\ell} A_f(k\to\ell)
-
A_f(\ell\to\ell),
\]

so positive values continue to mean that the desired DPS condition is
satisfied. The existing \(1\%\) JumpReLU firing-frequency filter is applied
after this preactivation-based DPS-C selection exactly as in the final DPS-A
notebook.

## 1. Imports and user-editable configuration


In [1]:
import os
import re
import json
import math
import random
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional, Sequence

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from IPython.display import display

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "notebook"

# ============================================================
# Reproducibility
# ============================================================
SEED = 42

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

# ============================================================
# Model / SAE
# ============================================================
MODEL_NAME = "google/gemma-2-9b-it"
MODEL_SHORT_NAME = "Gemma2-9B-IT"

# Middle-depth residual-stream site used for this model.
TARGET_LAYER = 20
EXPECTED_NUM_HIDDEN_LAYERS = 42
EXPECTED_D_MODEL = 3584

SAE_REPO_ID = "google/gemma-scope-9b-it-res"
SAE_WIDTH = "width_16k"
SAE_L0 = "average_l0_91"
SAE_FILENAME = (
    f"layer_{TARGET_LAYER}/"
    f"{SAE_WIDTH}/"
    f"{SAE_L0}/"
    "params.npz"
)
# Official Gemma Scope instruction-tuned residual SAE.
SAE_SOURCE_MODEL = "Gemma2-9B-IT"

DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available()
    else torch.float32
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# ============================================================
# Core experiment settings
# ============================================================
# Same requested final-query count per class across tasks
# -> comparable equation reliability.
#
# For AGNews/Yahoo, final queries come from the original test split.
# For TREC, final queries come from the original train split because the
# official test split contains only 9 ABBR examples. The selected TREC
# query examples are globally reserved and NEVER used as demonstrations.
N_QUERIES_PER_LABEL = 30
DISCOVERY_FRACTION = 0.50
AUTO_REDUCE_N_QUERIES_PER_LABEL = True

# Balanced fixed contexts:
# AGNews: 2/class -> 8 fixed + 1 variable = 9 demos
# TREC:   2/class -> 12 fixed + 1 variable = 13 demos
# Yahoo:  1/class -> 10 fixed + 1 variable = 11 demos
FIXED_DEMOS_PER_LABEL = {
    "agnews": 2,
    "trec": 2,
    "yahoo": 1,
}

# Every requested position must exist in every task.
POSITIONS_TO_TEST = list(range(1, 2))

# Default matches the main AGNews system-demonstration setup.
# Switch to "format2_wrapped_user_assistant_demos" to reproduce wrapped-turn prompts.
PROMPT_FORMAT_NAME = "format1_system_demos_user_query_assistant_label"

# ============================================================
# Measurement token convention
# ============================================================
# LAST_K is anchored to the FIRST TOKEN OF THE FINAL GOLD LABEL,
# NOT to the physical end of the rendered chat sequence.
#
# Therefore, for every task and every label (including multi-token labels):
#
#   LAST_K = [-1]
#       -> first token of the final gold label
#
#   LAST_K = [-2]
#       -> token immediately BEFORE the first label token
#       -> this is the position whose next-token prediction is the
#          first token of the answer label
#
#   LAST_K = [-3]
#       -> one token further back, etc.
#
# The main experiment uses [-2] so that we measure the residual state
# at the exact point where the model predicts the first label token.
LAST_K = [-1]

# Included in cache fingerprints so caches produced under the old
# sequence-end-relative LAST_K convention can never be reused here.
MEASUREMENT_ANCHOR = "first_final_label_token_v1"

# The supplied TREC/Yahoo notebooks use preactivations.
USE_SAE_PREACTIVATIONS = True
DPS_TIE_TOL = 1e-9

# ============================================================
# SAE activation-frequency filter
# ============================================================
# DPS-C is still defined using SAE PREACTIVATION.
#
# This is an additional activity filter using the TRUE sparse SAE
# post-activation (Gemma Scope JumpReLU):
#
#     feature fired on a prompt
#         <=> post-activation > 0 on at least one selected
#             measurement token for that prompt.
#
# With the current single-token LAST_K setting this is exactly the
# fraction of prompts on which the feature is active.
#
# Default requested threshold:
#     retain DPS-C features firing on >= 1% of all prompts.
ACTIVATION_FREQUENCY_THRESHOLD = 0.01

# "Across all prompts we used":
# aggregate AGNews + TREC + Yahoo and both partitions.
#
# For a strictly selection-only activity filter, change to:
#     ACTIVATION_FREQUENCY_SPLITS = ("discovery",)
ACTIVATION_FREQUENCY_SPLITS = (
    "discovery",
    "holdout",
)

# Version string included in the scan cache fingerprint so old caches
# that did not store prompt-level firing counts are never reused.
PROMPT_FIRING_MODE = "jump_relu_postactivation_any_selected_token_v1"

MAX_DEMO_CHARS = 1000
MAX_QUERY_CHARS = None

# Batching can substantially reduce runtime. 1 is safest; try 2 or 4 on a large GPU.
BATCH_SIZE = 16
PRINT_EVERY_BATCHES = 100

USE_CACHE = True
FORCE_RECOMPUTE = False
CACHE_ROOT = Path("./dps_3task_20eq_cache_gemma2_9b_it_gemmascope_it")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT = Path("./dps_3task_20eq_outputs_gemma2_9b_it_gemmascope_it")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Plot styling
PLOT_FONT = "Times New Roman, Times, serif"
COUNT_COLOR = "#636EFA"
PRIMACY_COLOR = "#EF553B"

print("MODEL:", MODEL_NAME)
print("TARGET_LAYER:", TARGET_LAYER)
print("SAE_REPO_ID:", SAE_REPO_ID)
print("SAE_FILENAME:", SAE_FILENAME)
print("SAE_SOURCE_MODEL:", SAE_SOURCE_MODEL)
print("POSITIONS_TO_TEST:", POSITIONS_TO_TEST)
print("PROMPT_FORMAT_NAME:", PROMPT_FORMAT_NAME)
print("LAST_K:", LAST_K)
print("MEASUREMENT_ANCHOR:", MEASUREMENT_ANCHOR)
print("USE_SAE_PREACTIVATIONS:", USE_SAE_PREACTIVATIONS)
print("ACTIVATION_FREQUENCY_THRESHOLD:", ACTIVATION_FREQUENCY_THRESHOLD)
print("ACTIVATION_FREQUENCY_SPLITS:", ACTIVATION_FREQUENCY_SPLITS)
print("PROMPT_FIRING_MODE:", PROMPT_FIRING_MODE)
print("N_QUERIES_PER_LABEL:", N_QUERIES_PER_LABEL)


/home/ikhyuncho23/ml-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MODEL: google/gemma-2-9b-it
TARGET_LAYER: 20
SAE_REPO_ID: google/gemma-scope-9b-it-res
SAE_FILENAME: layer_20/width_16k/average_l0_91/params.npz
SAE_SOURCE_MODEL: Gemma2-9B-IT
POSITIONS_TO_TEST: [1]
PROMPT_FORMAT_NAME: format1_system_demos_user_query_assistant_label
LAST_K: [-1]
MEASUREMENT_ANCHOR: first_final_label_token_v1
USE_SAE_PREACTIVATIONS: True
ACTIVATION_FREQUENCY_THRESHOLD: 0.01
ACTIVATION_FREQUENCY_SPLITS: ('discovery', 'holdout')
PROMPT_FIRING_MODE: jump_relu_postactivation_any_selected_token_v1
N_QUERIES_PER_LABEL: 30


## 2. Task specifications and robust dataset loaders


In [ ]:
# ============================================================
# Task metadata
# ============================================================
TASK_SPECS = {
    "agnews": {
        "display_name": "AGNews",
        "label_to_word": {
            0: "World",
            1: "Sports",
            2: "Business",
            3: "Technology",
        },
        "instruction": (
            "Pretend that you are an expert in news topic classification. "
            "For a given news article, classify its topic as one of: "
            "World, Sports, Business, or Technology."
        ),
        "input_field_name": "Article",
        "output_field_name": "Topic",
        "dataset_candidates": [
            ("fancyzhx/ag_news", None),
            ("SetFit/ag_news", None),
        ],
        "label_fields": ["label"],
        "text_fields": ["text"],
    },
    "trec": {
        "display_name": "TREC",
        "label_to_word": {
            0: "Abbreviation",
            1: "Entity",
            2: "Description",
            3: "Human",
            4: "Location",
            5: "Numeric",
        },
        "instruction": (
            "Pretend that you are an expert in question classification. "
            "For a given question, classify its coarse question type as one of: "
            "abbreviation, entity, description, human, location, or numeric."
        ),
        "input_field_name": "Question",
        "output_field_name": "Question Type",
        "dataset_candidates": [
            ("SetFit/TREC-QC", None),
            ("CogComp/trec", None),
        ],
        "label_fields": ["coarse_label", "label", "label_coarse", "coarse", "label-coarse"],
        "text_fields": ["question", "text"],
    },
    "yahoo": {
        "display_name": "Yahoo",
        "label_to_word": {
            0: "Society & Culture",
            1: "Science & Mathematics",
            2: "Health",
            3: "Education & Reference",
            4: "Computers & Internet",
            5: "Sports",
            6: "Business & Finance",
            7: "Entertainment & Music",
            8: "Family & Relationships",
            9: "Politics & Government",
        },
        "instruction": (
            "Pretend that you are an expert in Yahoo Answers topic classification. "
            "For a given question and answer text, classify the topic as one of the "
            "provided Yahoo Answers categories."
        ),
        "input_field_name": "Question / Answer",
        "output_field_name": "Topic",
        "dataset_candidates": [
            ("mteb/yahoo_answers_topics", None),
            ("community-datasets/yahoo_answers_topics", None),
        ],
        "label_fields": ["topic", "label"],
        "text_fields": ["question_title", "question_content", "best_answer", "text"],
    },
}

TASK_ORDER = ["agnews", "trec", "yahoo"]
TASK_SEED_OFFSET = {"agnews": 0, "trec": 100_000, "yahoo": 200_000}

for task_key in TASK_ORDER:
    spec = TASK_SPECS[task_key]
    spec["label_ids"] = list(spec["label_to_word"].keys())
    spec["num_labels"] = len(spec["label_ids"])
    spec["num_demos_total"] = (
        FIXED_DEMOS_PER_LABEL[task_key] * spec["num_labels"] + 1
    )
    assert max(POSITIONS_TO_TEST) <= spec["num_demos_total"], (
        task_key, POSITIONS_TO_TEST, spec["num_demos_total"]
    )

print("Task summary:")
display(pd.DataFrame([
    {
        "task": TASK_SPECS[k]["display_name"],
        "n_labels": TASK_SPECS[k]["num_labels"],
        "fixed_per_label": FIXED_DEMOS_PER_LABEL[k],
        "n_fixed": FIXED_DEMOS_PER_LABEL[k] * TASK_SPECS[k]["num_labels"],
        "n_total_demos": TASK_SPECS[k]["num_demos_total"],
    }
    for k in TASK_ORDER
]))

print("Total DPS-C equations:", sum(TASK_SPECS[k]["num_labels"] for k in TASK_ORDER))


In [3]:
# ============================================================
# Robust Hugging Face DATASET loading helpers
# ============================================================
#
# IMPORTANT (datasets >= 4.x):
# Remote dataset loading scripts / trust_remote_code are no longer supported.
# This notebook therefore uses namespaced Hub repositories that expose
# standard data formats (typically Parquet) and NEVER passes
# trust_remote_code to datasets.load_dataset().
#
# This restriction applies to Hugging Face *Datasets* loading below.
# The Transformers model/tokenizer loader later in the notebook may still
# use trust_remote_code, which is a separate API.
# ============================================================

HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
    or None
)


def _load_dataset_standard_format(path, config_name=None, split=None):
    """
    Load a Hub dataset without remote-code execution.

    Designed for modern `datasets` versions (including datasets >= 4.x).
    The dataset repository must expose a supported standard format
    such as Parquet/Arrow/CSV/JSON.
    """
    base_kwargs = {"path": path}

    if config_name is not None:
        base_kwargs["name"] = config_name

    if split is not None:
        base_kwargs["split"] = split

    attempts = []

    # Preferred modern authenticated form.
    if HF_TOKEN is not None:
        kw = dict(base_kwargs)
        kw["token"] = HF_TOKEN
        attempts.append(kw)

    # Public/no-token form. This also avoids incompatibilities in older
    # installations whose `load_dataset` may not accept `token=`.
    attempts.append(dict(base_kwargs))

    last_error = None

    for kwargs in attempts:
        try:
            return load_dataset(**kwargs)
        except TypeError as e:
            last_error = e

            # Compatibility fallback for older datasets versions that do not
            # accept token=. Importantly, we still do NOT add trust_remote_code.
            kwargs2 = dict(kwargs)
            kwargs2.pop("token", None)
            try:
                return load_dataset(**kwargs2)
            except Exception as e2:
                last_error = e2
        except Exception as e:
            last_error = e

    raise last_error


def _load_train_test(path, config_name=None):
    """
    Return `(train, test)` from either:
      1) split-list loading, or
      2) DatasetDict loading.

    If a repository has validation but no test split, validation is used as test.
    """
    split_error = None

    try:
        out = _load_dataset_standard_format(
            path=path,
            config_name=config_name,
            split=["train", "test"],
        )
        if isinstance(out, (list, tuple)) and len(out) == 2:
            return out[0], out[1]
    except Exception as e:
        split_error = e

    try:
        ds = _load_dataset_standard_format(
            path=path,
            config_name=config_name,
            split=None,
        )
    except Exception as e:
        raise RuntimeError(
            f"Could not load dataset {path!r}. "
            f"split-list error={split_error!r}; DatasetDict error={e!r}"
        ) from e

    if "train" not in ds:
        raise KeyError(
            f"{path!r} has no train split. Available splits={list(ds.keys())}"
        )

    if "test" in ds:
        return ds["train"], ds["test"]

    if "validation" in ds:
        print(f"Warning: {path!r} has no test split; using validation.")
        return ds["train"], ds["validation"]

    raise KeyError(
        f"{path!r} has neither test nor validation. "
        f"Available splits={list(ds.keys())}"
    )


# Canonical TREC order used by the supplied TREC notebook.
TREC_SHORT_TO_ID = {
    "ABBR": 0,
    "ENTY": 1,
    "DESC": 2,
    "HUM": 3,
    "LOC": 4,
    "NUM": 5,
}

SETFIT_TREC_COARSE_ID_TO_CANONICAL_ID = {
    0: 2,  # DESC -> Description
    1: 1,  # ENTY -> Entity
    2: 0,  # ABBR -> Abbreviation
    3: 3,  # HUM  -> Human
    4: 5,  # NUM  -> Numeric
    5: 4,  # LOC  -> Location
}

SETFIT_TREC_TEXT_TO_SHORT = {
    "description and abstract concepts": "DESC",
    "entities": "ENTY",
    "abbreviation": "ABBR",
    "human beings": "HUM",
    "locations": "LOC",
    "numeric values": "NUM",
}


def _canonicalize_setfit_trec_example(example):
    """
    Normalize SetFit/TREC-QC to the six-class coarse-label ordering used
    throughout this experiment.
    """
    canonical_id = None

    raw_short = example.get("label_coarse_original", None)
    if raw_short is not None:
        raw_short = str(raw_short).strip().upper()
        if raw_short in TREC_SHORT_TO_ID:
            canonical_id = TREC_SHORT_TO_ID[raw_short]

    if canonical_id is None:
        raw_text = example.get("label_coarse_text", None)
        if raw_text is not None:
            short = SETFIT_TREC_TEXT_TO_SHORT.get(
                str(raw_text).strip().lower()
            )
            if short is not None:
                canonical_id = TREC_SHORT_TO_ID[short]

    if canonical_id is None:
        raw_id = example.get("label_coarse", None)
        if (
            raw_id is not None
            and int(raw_id) in SETFIT_TREC_COARSE_ID_TO_CANONICAL_ID
        ):
            canonical_id = SETFIT_TREC_COARSE_ID_TO_CANONICAL_ID[int(raw_id)]

    if canonical_id is None:
        raise ValueError(
            "Could not canonicalize TREC row with keys="
            f"{list(example.keys())}"
        )

    text = example.get("text", example.get("question", None))
    if text is None:
        raise ValueError(
            f"Could not find TREC text in keys={list(example.keys())}"
        )

    return {
        "question": str(text),
        "text": str(text),
        "coarse_label": int(canonical_id),
        "label": int(canonical_id),
    }


def load_task_dataset(task_key: str):
    """
    Load one of AGNews/TREC/Yahoo using only standard-format Hub datasets.
    """
    spec = TASK_SPECS[task_key]
    last_error = None

    for path, config_name in spec["dataset_candidates"]:
        try:
            print(
                f"[{task_key}] trying standard-format dataset "
                f"{path!r} config={config_name!r}"
            )

            train, test = _load_train_test(path, config_name)

            # Preserve the canonical TREC mapping used by the supplied notebook.
            if task_key == "trec" and path == "SetFit/TREC-QC":
                train = train.map(
                    _canonicalize_setfit_trec_example,
                    desc="Canonicalizing TREC train labels",
                )
                test = test.map(
                    _canonicalize_setfit_trec_example,
                    desc="Canonicalizing TREC test labels",
                )

            print(
                f"[{task_key}] loaded {path}; "
                f"train={len(train)}, test={len(test)}"
            )

            return (
                train,
                test,
                path if config_name is None else f"{path}:{config_name}",
            )

        except Exception as e:
            last_error = e
            print(f"  failed: {e!r}")

    raise RuntimeError(
        f"Failed to load {task_key}. Last error: {last_error!r}"
    )

### Hugging Face `datasets >= 4.x` compatibility fix

This version intentionally **does not pass `trust_remote_code` to `datasets.load_dataset()`**.
It uses namespaced, standard-format dataset repositories instead:

- AGNews: `fancyzhx/ag_news` (fallback: `SetFit/ag_news`)
- TREC: `SetFit/TREC-QC` with the supplied notebook's canonical six-class remapping
- Yahoo: `mteb/yahoo_answers_topics` (standard Parquet representation)

The model/tokenizer loading code is separate and is unchanged.


## 3. Load all three datasets


In [ ]:
TASK_DATA = {}

for task_key in TASK_ORDER:
    train, test, dataset_name = load_task_dataset(task_key)

    # --------------------------------------------------------
    # Demo/query source policy
    # --------------------------------------------------------
    # AGNews / Yahoo:
    #     demonstrations -> original train split
    #     final queries  -> original test split
    #
    # TREC:
    #     demonstrations -> original train split
    #     final queries  -> original train split
    #
    # For TREC, the selected final-query indices are globally reserved
    # later and excluded from ALL demonstration sampling. Thus there is
    # no query/demo overlap despite using the same underlying HF split.
    # --------------------------------------------------------
    if task_key == "trec":
        query_dataset = train
        query_source = "train"
        query_demo_same_dataset = True
    else:
        query_dataset = test
        query_source = "test"
        query_demo_same_dataset = False

    TASK_DATA[task_key] = {
        # Preserve original HF splits for transparency.
        "train": train,
        "test": test,

        # Explicit experiment pools.
        "demo": train,
        "query": query_dataset,

        "demo_source": "train",
        "query_source": query_source,
        "query_demo_same_dataset": bool(query_demo_same_dataset),

        "dataset_name": dataset_name,
    }


display(pd.DataFrame([
    {
        "task": TASK_SPECS[k]["display_name"],
        "dataset": TASK_DATA[k]["dataset_name"],
        "original_train_n": len(TASK_DATA[k]["train"]),
        "original_test_n": len(TASK_DATA[k]["test"]),
        "demo_source": TASK_DATA[k]["demo_source"],
        "query_source": TASK_DATA[k]["query_source"],
        "demo_pool_n": len(TASK_DATA[k]["demo"]),
        "query_pool_n": len(TASK_DATA[k]["query"]),
        "same_underlying_pool": TASK_DATA[k]["query_demo_same_dataset"],
    }
    for k in TASK_ORDER
]))

## 4. Load Gemma2-9B-IT and the Gemma Scope IT residual SAE

Model:
- `google/gemma-2-9b-it`
- 42 transformer layers
- target residual layer: 20

SAE:
- Repository: `google/gemma-scope-9b-it-res`
- Checkpoint: `layer_20/width_16k/average_l0_91/params.npz`
- Site: residual stream
- Width: 16,384 features
- Architecture: JumpReLU
- Trained on the instruction-tuned Gemma2-9B model.

In [ ]:
def hf_hub_download_robust(repo_id: str, filename: str) -> str:
    kwargs = {
        "repo_id": repo_id,
        "filename": filename,
    }

    if HF_TOKEN is not None:
        kwargs["token"] = HF_TOKEN

    try:
        return hf_hub_download(**kwargs)

    except TypeError:
        kwargs.pop(
            "token",
            None,
        )

        if HF_TOKEN is not None:
            kwargs[
                "use_auth_token"
            ] = HF_TOKEN

        return hf_hub_download(
            **kwargs
        )


def load_tokenizer_robust(
    model_name: str,
):
    kwargs = {
        "pretrained_model_name_or_path":
            model_name,
        "trust_remote_code":
            True,
        "use_fast":
            True,
    }

    if HF_TOKEN is not None:
        kwargs["token"] = HF_TOKEN

    try:
        return AutoTokenizer.from_pretrained(
            **kwargs
        )

    except TypeError:
        kwargs.pop(
            "token",
            None,
        )

        if HF_TOKEN is not None:
            kwargs[
                "use_auth_token"
            ] = HF_TOKEN

        return AutoTokenizer.from_pretrained(
            **kwargs
        )


def load_model_robust(
    model_name: str,
):
    kwargs = {
        "pretrained_model_name_or_path":
            model_name,
        "torch_dtype":
            DTYPE,
        "trust_remote_code":
            True,
        "device_map":
            "auto",
    }

    if HF_TOKEN is not None:
        kwargs["token"] = HF_TOKEN

    try:
        return AutoModelForCausalLM.from_pretrained(
            **kwargs
        )

    except TypeError:
        kwargs.pop(
            "token",
            None,
        )

        if HF_TOKEN is not None:
            kwargs[
                "use_auth_token"
            ] = HF_TOKEN

        return AutoModelForCausalLM.from_pretrained(
            **kwargs
        )


class GemmaScopeJumpReLUSAE(
    torch.nn.Module
):
    """
    Minimal inference wrapper for the original Gemma Scope params.npz format.

    Preactivation:
        z_pre = x @ W_enc + b_enc

    JumpReLU:
        z = ReLU(z_pre) if z_pre > threshold else 0

    Decoder:
        x_hat = z @ W_dec + b_dec

    The main DPS experiment uses encoder PREACTIVATIONS by default.
    """

    def __init__(
        self,
        params,
        device,
    ):
        super().__init__()

        self.device = torch.device(
            device
        )

        # Official Gemma Scope NPZ parameters are float32.
        self.dtype = torch.float32

        W_enc = torch.from_numpy(
            np.array(
                params["W_enc"],
                copy=True,
            )
        )

        W_dec = torch.from_numpy(
            np.array(
                params["W_dec"],
                copy=True,
            )
        )

        b_enc = torch.from_numpy(
            np.array(
                params["b_enc"],
                copy=True,
            )
        )

        b_dec = torch.from_numpy(
            np.array(
                params["b_dec"],
                copy=True,
            )
        )

        threshold = torch.from_numpy(
            np.array(
                params["threshold"],
                copy=True,
            )
        )

        d_model = int(
            W_enc.shape[0]
        )

        n_features = int(
            W_enc.shape[1]
        )

        if W_dec.shape != (
            n_features,
            d_model,
        ):
            raise ValueError(
                f"Unexpected Gemma Scope W_dec shape "
                f"{tuple(W_dec.shape)}; "
                f"expected {(n_features, d_model)}."
            )

        if b_enc.numel() != n_features:
            raise ValueError(
                f"Unexpected b_enc size "
                f"{b_enc.numel()} != {n_features}."
            )

        if b_dec.numel() != d_model:
            raise ValueError(
                f"Unexpected b_dec size "
                f"{b_dec.numel()} != {d_model}."
            )

        if threshold.numel() not in (
            1,
            n_features,
        ):
            raise ValueError(
                f"Unexpected threshold size "
                f"{threshold.numel()}; "
                f"expected 1 or {n_features}."
            )

        self.register_buffer(
            "W_enc",
            W_enc.to(
                device=self.device,
                dtype=self.dtype,
            ),
        )

        self.register_buffer(
            "W_dec",
            W_dec.to(
                device=self.device,
                dtype=self.dtype,
            ),
        )

        self.register_buffer(
            "b_enc",
            b_enc.reshape(
                -1
            ).to(
                device=self.device,
                dtype=self.dtype,
            ),
        )

        self.register_buffer(
            "b_dec",
            b_dec.reshape(
                -1
            ).to(
                device=self.device,
                dtype=self.dtype,
            ),
        )

        self.register_buffer(
            "threshold",
            threshold.reshape(
                -1
            ).to(
                device=self.device,
                dtype=self.dtype,
            ),
        )

        self.d_model = d_model
        self.n_features = n_features

    def encode_pre(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        x = x.to(
            device=self.device,
            dtype=self.dtype,
        )

        return (
            x @ self.W_enc
            + self.b_enc
        )

    def activation_fn(
        self,
        pre: torch.Tensor,
    ) -> torch.Tensor:
        return torch.where(
            pre > self.threshold,
            torch.relu(
                pre
            ),
            torch.zeros_like(
                pre
            ),
        )

    def encode(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        return self.activation_fn(
            self.encode_pre(
                x
            )
        )

    def decode(
        self,
        acts: torch.Tensor,
    ) -> torch.Tensor:
        acts = acts.to(
            device=self.device,
            dtype=self.dtype,
        )

        return (
            acts @ self.W_dec
            + self.b_dec
        )


def get_model_input_device(
    model,
):
    return (
        model.get_input_embeddings()
        .weight.device
    )


def get_layer_device(
    model,
    layer_idx: int,
):
    block = model.get_submodule(
        f"model.layers.{layer_idx}"
    )

    try:
        return next(
            block.parameters()
        ).device

    except StopIteration:
        return get_model_input_device(
            model
        )



# IMPORTANT:
# This is the official Gemma Scope instruction-tuned residual SAE for
# Gemma2-9B-IT, not the PT SAE repository.


print(
    "Loading tokenizer..."
)

tokenizer = load_tokenizer_robust(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )

tokenizer.padding_side = "right"


print(
    "Loading model..."
)

model = load_model_robust(
    MODEL_NAME
)

model.eval()

actual_num_layers = int(
    getattr(
        model.config,
        "num_hidden_layers",
        -1,
    )
)

if (
    actual_num_layers
    != EXPECTED_NUM_HIDDEN_LAYERS
):
    raise ValueError(
        f"Unexpected Gemma2 layer count: "
        f"{actual_num_layers} != "
        f"{EXPECTED_NUM_HIDDEN_LAYERS}."
    )

actual_hidden_size = int(
    getattr(
        model.config,
        "hidden_size",
        -1,
    )
)

if (
    actual_hidden_size
    != EXPECTED_D_MODEL
):
    raise ValueError(
        f"Unexpected Gemma2 hidden size: "
        f"{actual_hidden_size} != "
        f"{EXPECTED_D_MODEL}."
    )


MODEL_INPUT_DEVICE = (
    get_model_input_device(
        model
    )
)

TARGET_LAYER_DEVICE = (
    get_layer_device(
        model,
        TARGET_LAYER,
    )
)

print(
    "MODEL_INPUT_DEVICE:",
    MODEL_INPUT_DEVICE,
)

print(
    "TARGET_LAYER_DEVICE:",
    TARGET_LAYER_DEVICE,
)


print(
    "Loading Gemma Scope SAE..."
)

print(
    "SAE source model:",
    SAE_SOURCE_MODEL,
)

print(
    "SAE repo:",
    SAE_REPO_ID,
)

print(
    "SAE file:",
    SAE_FILENAME,
)

sae_path = hf_hub_download_robust(
    SAE_REPO_ID,
    SAE_FILENAME,
)

with np.load(
    sae_path
) as params_npz:
    params = {
        k:
            np.array(
                params_npz[k],
                copy=True,
            )
        for k
        in params_npz.files
    }

required = {
    "W_enc",
    "W_dec",
    "b_enc",
    "b_dec",
    "threshold",
}

missing = (
    required
    - set(
        params
    )
)

if missing:
    raise KeyError(
        "Gemma Scope NPZ is missing expected keys: "
        f"{sorted(missing)}"
    )

sae = GemmaScopeJumpReLUSAE(
    params,
    device=
        TARGET_LAYER_DEVICE,
)

sae.eval()

if (
    int(
        sae.d_model
    )
    != EXPECTED_D_MODEL
):
    raise ValueError(
        f"Gemma Scope d_model={sae.d_model} "
        f"does not match model hidden size "
        f"{EXPECTED_D_MODEL}."
    )

if (
    int(
        sae.n_features
    )
    != 16384
):
    raise ValueError(
        f"Expected 16K Gemma Scope SAE, "
        f"got n_features={sae.n_features}."
    )

print(
    f"Gemma Scope SAE: "
    f"d_model={sae.d_model}, "
    f"n_features={sae.n_features}, "
    f"source={SAE_SOURCE_MODEL}"
)

## 5. Generic text, label, and prompt-format helpers


In [ ]:
QUERY_START_SENTINEL = "ZXQ_QUERY_START_20EQ_94c91"
QUERY_END_SENTINEL = "ZXQ_QUERY_END_20EQ_94c91"

def _has_key(example, key):
    try:
        return key in example.keys()
    except Exception:
        return key in example

def get_label_id(task_key: str, example) -> int:
    spec = TASK_SPECS[task_key]
    for key in spec["label_fields"]:
        if _has_key(example, key):
            raw = int(example[key])
            if raw in spec["label_to_word"]:
                return raw
            if (raw - 1) in spec["label_to_word"]:
                return raw - 1
            raise ValueError(
                f"{task_key}: label {raw} from {key!r} incompatible with "
                f"{spec['label_to_word']}"
            )
    raise KeyError(
        f"{task_key}: no label field in {spec['label_fields']}; keys={list(example.keys())}"
    )

def clean_task_text(text: str, max_chars: Optional[int]) -> str:
    text = str(text).replace("\n", " ").replace("\t", " ").strip()
    text = re.sub(r"\s+", " ", text)
    if max_chars is not None and len(text) > int(max_chars):
        text = text[: int(max_chars)].rstrip() + "..."
    return text

def get_example_text(task_key: str, example, max_chars: Optional[int]) -> str:
    spec = TASK_SPECS[task_key]

    if task_key == "yahoo":
        pieces = []
        field_labels = {
            "question_title": "Title",
            "question_content": "Question details",
            "best_answer": "Best answer",
            "text": "Text",
        }
        for key in spec["text_fields"]:
            if _has_key(example, key):
                val = str(example[key]).strip()
                if val and val.lower() != "none":
                    pieces.append(f"{field_labels.get(key, key)}: {val}")
        if not pieces:
            raise KeyError(f"Could not construct Yahoo text from keys={list(example.keys())}")
        return clean_task_text("\n".join(pieces), max_chars)

    for key in spec["text_fields"]:
        if _has_key(example, key):
            return clean_task_text(example[key], max_chars)
    raise KeyError(
        f"{task_key}: no text field among {spec['text_fields']}; keys={list(example.keys())}"
    )

def label_word(task_key: str, example) -> str:
    label_id = get_label_id(task_key, example)
    return TASK_SPECS[task_key]["label_to_word"][label_id]

def format_demo_text_block(task_key: str, example, idx: int) -> str:
    spec = TASK_SPECS[task_key]
    text = get_example_text(task_key, example, MAX_DEMO_CHARS)
    return (
        f"Example {idx}\n"
        f"{spec['input_field_name']}:\n{text}\n"
        f"{spec['output_field_name']}:\n{label_word(task_key, example)}"
    )

def format_user_request(task_key: str, example, idx: int, *, mark_text: bool) -> str:
    spec = TASK_SPECS[task_key]
    max_chars = MAX_QUERY_CHARS if mark_text else MAX_DEMO_CHARS
    text = get_example_text(task_key, example, max_chars)
    if mark_text:
        text = f"{QUERY_START_SENTINEL}{text}{QUERY_END_SENTINEL}"
    return (
        f"Example {idx}\n"
        f"{spec['input_field_name']}:\n{text}\n"
        f"{spec['output_field_name']}:"
    )

def manual_gemma_chat_template(
    messages: List[Dict[str, str]],
    *,
    add_generation_prompt: bool = False,
) -> str:
    """
    Manual fallback matching Gemma 2 IT's documented user/model format.
    This notebook intentionally avoids a system role for Gemma.
    """
    parts = ["<bos>"]

    for msg in messages:
        role = msg["role"]

        if role == "assistant":
            gemma_role = "model"
        elif role == "user":
            gemma_role = "user"
        else:
            raise ValueError(
                "Gemma2-IT prompt construction in this notebook supports "
                "only user/assistant roles. System text is folded into "
                "the first user turn."
            )

        parts.append(
            f"<start_of_turn>{gemma_role}\n"
            f"{msg['content']}<end_of_turn>\n"
        )

    if add_generation_prompt:
        parts.append("<start_of_turn>model\n")

    return "".join(parts).rstrip()


def apply_model_chat_template(
    messages: List[Dict[str, str]],
    *,
    add_generation_prompt: bool = False,
) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
    except Exception:
        return manual_gemma_chat_template(
            messages,
            add_generation_prompt=add_generation_prompt,
        )


def render_prompt_with_query_span(messages):
    rendered = apply_model_chat_template(
        messages,
        add_generation_prompt=False,
    )

    s = rendered.find(QUERY_START_SENTINEL)
    e = rendered.find(QUERY_END_SENTINEL)

    if s < 0 or e < 0 or e <= s:
        raise ValueError(
            "Could not locate query sentinels in rendered prompt."
        )

    q_start_marked = s + len(
        QUERY_START_SENTINEL
    )
    query_text = rendered[
        q_start_marked:e
    ]

    prompt = (
        rendered[:s]
        + query_text
        + rendered[
            e + len(QUERY_END_SENTINEL):
        ]
    )

    query_span = (
        s,
        s + len(query_text),
    )

    return prompt.rstrip(), query_span


def build_prompt_format1(
    task_key: str,
    demos,
    query_example,
):
    """
    Same semantic prompt as the Qwen format-1 experiment, but Gemma 2 IT
    has no native system role. Therefore instruction + demonstrations +
    final query are folded into one user turn, followed by the gold model
    answer turn.
    """
    spec = TASK_SPECS[task_key]

    demo_blocks = [
        format_demo_text_block(
            task_key,
            d,
            i + 1,
        )
        for i, d in enumerate(demos)
    ]

    sections = [
        spec["instruction"],
    ]

    if demo_blocks:
        sections.append(
            "Demonstrations:\n\n"
            + "\n\n".join(demo_blocks)
        )

    sections.append(
        format_user_request(
            task_key,
            query_example,
            len(demos) + 1,
            mark_text=True,
        )
    )

    messages = [
        {
            "role": "user",
            "content": "\n\n".join(sections),
        },
        {
            "role": "assistant",
            "content": label_word(
                task_key,
                query_example,
            ),
        },
    ]

    return render_prompt_with_query_span(
        messages
    )


def build_prompt_format2(
    task_key: str,
    demos,
    query_example,
):
    """
    Gemma-compatible wrapped-turn variant.

    The task instruction is prepended to the first user message so role
    alternation remains user/model/user/model/...
    """
    spec = TASK_SPECS[task_key]
    messages = []

    for i, demo in enumerate(
        demos,
        start=1,
    ):
        user_content = format_user_request(
            task_key,
            demo,
            i,
            mark_text=False,
        )

        if i == 1:
            user_content = (
                spec["instruction"]
                + "\n\n"
                + user_content
            )

        messages.append({
            "role": "user",
            "content": user_content,
        })
        messages.append({
            "role": "assistant",
            "content": label_word(
                task_key,
                demo,
            ),
        })

    query_content = format_user_request(
        task_key,
        query_example,
        len(demos) + 1,
        mark_text=True,
    )

    # In the zero-demo edge case, preserve the instruction.
    if not demos:
        query_content = (
            spec["instruction"]
            + "\n\n"
            + query_content
        )

    messages.append({
        "role": "user",
        "content": query_content,
    })
    messages.append({
        "role": "assistant",
        "content": label_word(
            task_key,
            query_example,
        ),
    })

    return render_prompt_with_query_span(
        messages
    )


def build_prompt(
    task_key: str,
    demos,
    query_example,
):
    if (
        PROMPT_FORMAT_NAME
        == "format1_system_demos_user_query_assistant_label"
    ):
        return build_prompt_format1(
            task_key,
            demos,
            query_example,
        )

    if (
        PROMPT_FORMAT_NAME
        == "format2_wrapped_user_assistant_demos"
    ):
        return build_prompt_format2(
            task_key,
            demos,
            query_example,
        )

    raise ValueError(
        f"Unknown PROMPT_FORMAT_NAME={PROMPT_FORMAT_NAME!r}"
    )

## 6. Sampling plans: fixed queries, balanced fixed contexts, and position-controlled variable demos

The experiment uses a fixed balanced set of final queries for each task and
splits those queries into **discovery** and **holdout** halves.

- **AGNews / Yahoo:** demonstrations come from the original training split and
  final queries come from the original test split.
- **TREC:** the official test split contains only **9 Abbreviation** examples,
  which would unnecessarily cap every class at 9 queries. Because our goal is
  a mechanistic representation/causal analysis rather than reporting the
  benchmark's held-out test accuracy, we instead sample TREC final queries from
  its original training split.
- To keep this design clean, the entire selected TREC query set is **globally
  reserved**: none of those examples may be used as a fixed or variable
  demonstration anywhere in the experiment.
- Discovery and holdout query sets remain disjoint.

Thus TREC can use the same requested number of final queries per label as the
other tasks while maintaining strict query--demonstration disjointness.

In [ ]:
def index_by_label(task_key: str, dataset) -> Dict[int, List[int]]:
    spec = TASK_SPECS[task_key]
    out = {int(l): [] for l in spec["label_ids"]}

    for i, row in enumerate(dataset):
        l = get_label_id(task_key, row)
        if l in out:
            out[l].append(int(i))

    return out


# ============================================================
# Build label-indexed demo/query pools
# ============================================================
for task_key in TASK_ORDER:
    TASK_DATA[task_key]["demo_by_label"] = index_by_label(
        task_key,
        TASK_DATA[task_key]["demo"],
    )

    TASK_DATA[task_key]["query_by_label"] = index_by_label(
        task_key,
        TASK_DATA[task_key]["query"],
    )

    spec = TASK_SPECS[task_key]

    print(f"\n{spec['display_name']} label counts")

    print(
        f"demo pool ({TASK_DATA[task_key]['demo_source']}):",
        {
            spec["label_to_word"][l]:
            len(TASK_DATA[task_key]["demo_by_label"][l])
            for l in spec["label_ids"]
        },
    )

    print(
        f"query pool ({TASK_DATA[task_key]['query_source']}):",
        {
            spec["label_to_word"][l]:
            len(TASK_DATA[task_key]["query_by_label"][l])
            for l in spec["label_ids"]
        },
    )


def sample_fixed_queries(task_key: str) -> pd.DataFrame:
    """
    Sample a fixed, balanced set of final queries.

    IMPORTANT FOR TREC
    ------------------
    TREC's final queries are sampled from the original train split because
    the official test split contains only 9 Abbreviation examples.

    Since TREC demos also come from the train split, ALL selected TREC query
    indices are globally reserved and excluded from every demo pool. This is
    stronger than merely preventing overlap within one prompt: no selected
    final query can appear as a demonstration anywhere in the experiment.
    """
    spec = TASK_SPECS[task_key]
    by_label = TASK_DATA[task_key]["query_by_label"]

    min_count = min(
        len(by_label[l])
        for l in spec["label_ids"]
    )

    per_label = int(N_QUERIES_PER_LABEL)

    if per_label > min_count:
        if AUTO_REDUCE_N_QUERIES_PER_LABEL:
            print(
                f"[{task_key}] reducing N_QUERIES_PER_LABEL "
                f"{per_label} -> {min_count}"
            )
            per_label = int(min_count)
        else:
            raise ValueError(
                f"{task_key}: requested {per_label}, "
                f"smallest query-pool class has {min_count}"
            )

    if per_label < 2:
        raise ValueError(
            f"{task_key}: need at least 2 final queries per label."
        )

    discovery_n = int(
        round(DISCOVERY_FRACTION * per_label)
    )
    discovery_n = max(
        1,
        min(per_label - 1, discovery_n),
    )

    rng = random.Random(
        SEED + TASK_SEED_OFFSET[task_key]
    )

    rows = []
    selected_query_indices = []

    for l in spec["label_ids"]:
        chosen = rng.sample(
            list(by_label[l]),
            per_label,
        )

        selected_query_indices.extend(
            map(int, chosen)
        )

        for qpos, query_idx in enumerate(chosen):
            rows.append({
                "query_global_id": len(rows),
                "query_label_id": int(l),
                "query_label": spec["label_to_word"][l],
                "query_index_within_label": int(qpos),
                "query_index": int(query_idx),
                "query_source": TASK_DATA[task_key]["query_source"],
                "split": (
                    "discovery"
                    if qpos < discovery_n
                    else "holdout"
                ),
            })

    # If queries and demos refer to the same underlying dataset
    # (currently TREC), reserve the ENTIRE selected query set.
    if TASK_DATA[task_key]["query_demo_same_dataset"]:
        reserved = set(
            map(int, selected_query_indices)
        )
    else:
        # Index values from distinct HF splits are unrelated, so no
        # numerical exclusion is necessary.
        reserved = set()

    TASK_DATA[task_key][
        "reserved_query_indices_from_demo_pool"
    ] = reserved

    TASK_DATA[task_key][
        "actual_n_queries_per_label"
    ] = int(per_label)

    TASK_DATA[task_key][
        "discovery_n_per_label"
    ] = int(discovery_n)

    return pd.DataFrame(rows)


def sample_fixed_context_pool(
    task_key: str,
    query_global_id: int,
) -> Tuple[int, ...]:
    spec = TASK_SPECS[task_key]
    demo_by_label = TASK_DATA[task_key]["demo_by_label"]
    fixed_per_label = int(
        FIXED_DEMOS_PER_LABEL[task_key]
    )

    # Globally exclude all selected final queries whenever the
    # query/demo pools share the same underlying dataset.
    reserved_queries = set(
        TASK_DATA[task_key].get(
            "reserved_query_indices_from_demo_pool",
            set(),
        )
    )

    rng = random.Random(
        SEED
        + TASK_SEED_OFFSET[task_key]
        + 31_000
        + 1_003 * int(query_global_id)
    )

    indices = []

    for l in spec["label_ids"]:
        pool = [
            int(i)
            for i in demo_by_label[l]
            if int(i) not in reserved_queries
        ]

        if len(pool) < fixed_per_label:
            raise ValueError(
                f"{task_key}: label {l} has only {len(pool)} "
                f"demo rows after reserving query examples, "
                f"but fixed_per_label={fixed_per_label}."
            )

        indices.extend(
            rng.sample(
                pool,
                fixed_per_label,
            )
        )

    rng.shuffle(indices)

    expected = (
        fixed_per_label
        * spec["num_labels"]
    )

    assert len(indices) == expected

    return tuple(
        map(int, indices)
    )


def sample_variable_demo_indices_by_label(
    task_key: str,
    query_global_id: int,
    fixed_context_indices: Sequence[int],
) -> Dict[int, int]:
    spec = TASK_SPECS[task_key]
    demo_by_label = TASK_DATA[task_key]["demo_by_label"]

    exclude = set(
        map(int, fixed_context_indices)
    )

    # Again, globally exclude every selected final query.
    exclude.update(
        TASK_DATA[task_key].get(
            "reserved_query_indices_from_demo_pool",
            set(),
        )
    )

    out = {}

    for l in spec["label_ids"]:
        pool = [
            int(i)
            for i in demo_by_label[l]
            if int(i) not in exclude
        ]

        if not pool:
            raise ValueError(
                f"{task_key}: no variable demo available for label {l} "
                "after excluding fixed demos and reserved final queries."
            )

        rng = random.Random(
            SEED
            + TASK_SEED_OFFSET[task_key]
            + 59_000
            + 10_007 * int(query_global_id)
            + int(l)
        )

        out[l] = int(
            rng.choice(pool)
        )

    return out


def insert_variable_at_position(
    fixed_context_indices: Sequence[int],
    variable_idx: int,
    varied_position: int,
) -> Tuple[int, ...]:
    fixed = list(
        map(int, fixed_context_indices)
    )

    p0 = int(varied_position) - 1

    if p0 < 0 or p0 > len(fixed):
        raise IndexError(
            f"varied_position={varied_position} invalid for "
            f"{len(fixed) + 1} total demos"
        )

    return tuple(
        fixed[:p0]
        + [int(variable_idx)]
        + fixed[p0:]
    )


def build_task_position_plan(
    task_key: str,
) -> pd.DataFrame:
    spec = TASK_SPECS[task_key]

    query_df = sample_fixed_queries(
        task_key
    )

    rows = []

    for r in query_df.itertuples(index=False):
        fixed_context = sample_fixed_context_pool(
            task_key,
            int(r.query_global_id),
        )

        variable_by_label = (
            sample_variable_demo_indices_by_label(
                task_key,
                int(r.query_global_id),
                fixed_context,
            )
        )

        for p in POSITIONS_TO_TEST:
            for c in spec["label_ids"]:
                variable_idx = (
                    variable_by_label[c]
                )

                demo_indices = (
                    insert_variable_at_position(
                        fixed_context,
                        variable_idx,
                        p,
                    )
                )

                rows.append({
                    "query_global_id":
                        int(r.query_global_id),

                    "query_label_id":
                        int(r.query_label_id),

                    "query_label":
                        str(r.query_label),

                    "query_index_within_label":
                        int(r.query_index_within_label),

                    "query_index":
                        int(r.query_index),

                    "query_source":
                        str(r.query_source),

                    "split":
                        str(r.split),

                    "varied_position":
                        int(p),

                    "varied_label_id":
                        int(c),

                    "varied_label":
                        spec["label_to_word"][c],

                    "variable_demo_index":
                        int(variable_idx),

                    "demo_indices":
                        tuple(demo_indices),
                })

    df = pd.DataFrame(rows)

    expected_queries = (
        spec["num_labels"]
        * TASK_DATA[task_key][
            "actual_n_queries_per_label"
        ]
    )

    expected_prompts = (
        expected_queries
        * len(POSITIONS_TO_TEST)
        * spec["num_labels"]
    )

    assert len(df) == expected_prompts
    assert (
        df["query_global_id"].nunique()
        == expected_queries
    )

    assert (
        df.groupby(
            [
                "query_global_id",
                "varied_position",
            ]
        )
        .size()
        .eq(spec["num_labels"])
        .all()
    )

    # --------------------------------------------------------
    # Strong global no-overlap assertion for same-pool tasks.
    # --------------------------------------------------------
    if TASK_DATA[task_key][
        "query_demo_same_dataset"
    ]:
        query_indices = set(
            df["query_index"]
            .astype(int)
            .unique()
            .tolist()
        )

        used_demo_indices = {
            int(i)
            for demo_tuple in df["demo_indices"]
            for i in tuple(demo_tuple)
        }

        overlap = (
            query_indices
            & used_demo_indices
        )

        assert len(overlap) == 0, (
            f"{task_key}: found query/demo overlap: "
            f"{sorted(overlap)[:20]}"
        )

        print(
            f"[{task_key}] global query/demo overlap check: "
            f"0 overlaps across "
            f"{len(query_indices)} selected final queries."
        )

    return df


POSITION_PLANS = {}

for task_key in TASK_ORDER:
    POSITION_PLANS[task_key] = (
        build_task_position_plan(
            task_key
        )
    )

    spec = TASK_SPECS[task_key]

    print(
        f"{spec['display_name']}: "
        f"{len(POSITION_PLANS[task_key])} prompts, "
        f"{POSITION_PLANS[task_key]['query_global_id'].nunique()} queries "
        f"({TASK_DATA[task_key]['actual_n_queries_per_label']}/label; "
        f"query source={TASK_DATA[task_key]['query_source']})"
    )


print(
    "TOTAL MODEL FORWARDS (before batching):",
    sum(
        len(POSITION_PLANS[k])
        for k in TASK_ORDER
    ),
)

## 7. Prompt and measurement-token sanity check

`LAST_K` is defined relative to the **first token of the final answer label**,
rather than relative to the terminal chat token. Consequently, `LAST_K=[-1]`
always measures the first label token itself, while `LAST_K=[-2]` measures the
immediately preceding residual position—the position from which the model
predicts that first label token. This definition is invariant to whether a
label consists of one token or multiple tokens.

In [ ]:
def normalize_last_k_spec(
    last_k=LAST_K,
):
    """
    Normalize LAST_K.

    Negative offsets are interpreted RELATIVE TO THE FIRST TOKEN
    OF THE FINAL GOLD LABEL:

        -1 -> first label token
        -2 -> immediately preceding token
        -3 -> two tokens before first label token
        ...

    Positive values, if supplied, are retained as absolute token indices.
    """
    if last_k is None:
        return None

    if isinstance(
        last_k,
        (
            int,
            np.integer,
        ),
    ):
        k = int(
            last_k
        )

        if k <= 0:
            raise ValueError(
                "Integer LAST_K must be positive. "
                "Use a list such as [-1] or [-2] for "
                "label-boundary-relative offsets."
            )

        # Preserve the notebook's historical integer shorthand:
        # LAST_K = 2 -> [-2, -1]
        return list(
            range(
                -k,
                0,
            )
        )

    vals = [
        int(x)
        for x
        in list(
            last_k
        )
    ]

    if not vals:
        raise ValueError(
            "LAST_K list cannot be empty."
        )

    return vals


def _find_final_gold_label_char_span(
    prompt_text: str,
    gold_label: str,
) -> Tuple[int, int]:
    """
    Locate the final assistant gold label in the rendered prompt.

    `rfind` is intentional: the same label word may occur in the
    demonstrations or query, but the final assistant answer is the
    final occurrence in this prompt format.
    """
    gold_label = str(
        gold_label
    )

    start = prompt_text.rfind(
        gold_label
    )

    if start < 0:
        raise RuntimeError(
            f"Could not locate final gold label "
            f"{gold_label!r} in rendered prompt."
        )

    end = (
        start
        + len(
            gold_label
        )
    )

    return (
        int(
            start
        ),
        int(
            end
        ),
    )


def _find_first_final_label_token_index(
    input_ids_1d,
    attention_mask_1d,
    offset_mapping_2d,
    prompt_text: str,
    gold_label: str,
) -> Tuple[int, List[int]]:
    """
    Return the token index of the FIRST token of the FINAL gold label.

    This uses the label's exact character span inside the complete rendered
    prompt, so multi-token labels such as "Abbreviation" or
    "Society & Culture" are handled correctly.

    Returns
    -------
    first_label_token_idx:
        Token index of the first token overlapping the final gold-label span.

    all_label_token_indices:
        All tokens overlapping that final gold-label span.
    """
    length = int(
        np.asarray(
            attention_mask_1d
        ).sum()
    )

    (
        label_char_start,
        label_char_end,
    ) = _find_final_gold_label_char_span(
        prompt_text,
        gold_label,
    )

    label_token_indices = []

    for i in range(
        length
    ):
        s, e = map(
            int,
            offset_mapping_2d[
                i
            ],
        )

        # Empty offsets are normally chat-special tokens.
        if e <= s:
            continue

        if (
            e > label_char_start
            and s < label_char_end
        ):
            label_token_indices.append(
                int(
                    i
                )
            )

    if not label_token_indices:
        raise RuntimeError(
            "No token overlaps the final gold-label character span.\n"
            f"gold_label={gold_label!r}\n"
            f"label_char_span="
            f"({label_char_start}, {label_char_end})"
        )

    first_label_token_idx = int(
        min(
            label_token_indices
        )
    )

    # --------------------------------------------------------
    # Strong contextual-boundary sanity check:
    # the first selected token must actually overlap the first
    # character(s) of the final label.
    # --------------------------------------------------------
    first_s, first_e = map(
        int,
        offset_mapping_2d[
            first_label_token_idx
        ],
    )

    if not (
        first_e > label_char_start
        and first_s < label_char_end
    ):
        raise AssertionError(
            "Resolved first label token does not overlap "
            "the final gold-label span."
        )

    return (
        first_label_token_idx,
        label_token_indices,
    )


def select_measurement_indices(
    input_ids_1d,
    attention_mask_1d,
    offset_mapping_2d,
    query_char_span,
    *,
    prompt_text: str,
    gold_label: str,
):
    """
    Select residual-stream positions for the activation experiment.

    If LAST_K is not None, negative offsets are anchored to the
    FIRST TOKEN OF THE FINAL GOLD LABEL:

        offset = -1:
            first gold-label token

        offset = -2:
            token immediately before the first gold-label token
            (the position that predicts the label's first token)

        offset = -3:
            one position further back

    If LAST_K is None, fall back to the original query-span measurement.
    """
    length = int(
        np.asarray(
            attention_mask_1d
        ).sum()
    )

    last_spec = (
        normalize_last_k_spec(
            LAST_K
        )
    )

    if last_spec is not None:
        (
            first_label_idx,
            _,
        ) = _find_first_final_label_token_index(
            input_ids_1d,
            attention_mask_1d,
            offset_mapping_2d,
            prompt_text=
                prompt_text,
            gold_label=
                gold_label,
        )

        out = []

        for offset in last_spec:
            offset = int(
                offset
            )

            if offset < 0:
                # ------------------------------------------------
                # New label-boundary-relative convention:
                #
                # -1 -> first_label_idx
                # -2 -> first_label_idx - 1
                # -3 -> first_label_idx - 2
                # ------------------------------------------------
                idx = (
                    first_label_idx
                    + offset
                    + 1
                )

            else:
                # Preserve support for explicit absolute indices.
                idx = offset

            if (
                idx < 0
                or idx >= length
            ):
                raise IndexError(
                    f"LAST_K offset {offset} -> idx {idx}, "
                    f"first_label_idx={first_label_idx}, "
                    f"sequence length={length}"
                )

            out.append(
                int(
                    idx
                )
            )

        return out

    # ========================================================
    # Original query-span mode when LAST_K is None.
    # ========================================================
    q0, q1 = (
        query_char_span
    )

    out = []

    for i in range(
        length
    ):
        s, e = map(
            int,
            offset_mapping_2d[
                i
            ],
        )

        if (
            e > q0
            and s < q1
            and e > s
        ):
            out.append(
                int(
                    i
                )
            )

    if not out:
        raise RuntimeError(
            "No tokens overlap the final-query text span."
        )

    return out


# ============================================================
# Sanity check on one real prompt from each task
# ============================================================

for task_key in TASK_ORDER:
    row = (
        POSITION_PLANS[
            task_key
        ].iloc[
            0
        ]
    )

    demos = [
        TASK_DATA[
            task_key
        ][
            "demo"
        ][
            int(
                i
            )
        ]
        for i
        in row[
            "demo_indices"
        ]
    ]

    query = (
        TASK_DATA[
            task_key
        ][
            "query"
        ][
            int(
                row[
                    "query_index"
                ]
            )
        ]
    )

    gold_label = (
        label_word(
            task_key,
            query,
        )
    )

    (
        prompt,
        query_span,
    ) = build_prompt(
        task_key,
        demos,
        query,
    )

    enc = tokenizer(
        prompt,
        return_tensors=
            "pt",
        add_special_tokens=
            False,
        return_offsets_mapping=
            True,
        truncation=
            False,
    )

    offsets = (
        enc[
            "offset_mapping"
        ][
            0
        ].tolist()
    )

    mask = (
        enc[
            "attention_mask"
        ][
            0
        ].numpy()
    )

    ids = (
        enc[
            "input_ids"
        ][
            0
        ].tolist()
    )

    (
        first_label_idx,
        all_label_idxs,
    ) = _find_first_final_label_token_index(
        ids,
        mask,
        offsets,
        prompt_text=
            prompt,
        gold_label=
            gold_label,
    )

    idxs = (
        select_measurement_indices(
            ids,
            mask,
            offsets,
            query_span,
            prompt_text=
                prompt,
            gold_label=
                gold_label,
        )
    )

    print(
        f"\n[{TASK_SPECS[task_key]['display_name']}]"
    )

    print(
        "gold label:",
        repr(
            gold_label
        ),
    )

    print(
        "prompt chars:",
        len(
            prompt
        ),
        "tokens:",
        len(
            ids
        ),
    )

    print(
        "all label token indices:",
        all_label_idxs,
    )

    print(
        "all label tokens:",
        [
            repr(
                tokenizer.decode(
                    [
                        ids[
                            i
                        ]
                    ],
                    skip_special_tokens=
                        False,
                    clean_up_tokenization_spaces=
                        False,
                )
            )
            for i
            in all_label_idxs
        ],
    )

    print(
        "FIRST label token index:",
        first_label_idx,
    )

    print(
        "FIRST label token:",
        repr(
            tokenizer.decode(
                [
                    ids[
                        first_label_idx
                    ]
                ],
                skip_special_tokens=
                    False,
                clean_up_tokenization_spaces=
                    False,
            )
        ),
    )

    print(
        "token immediately before first label:",
        repr(
            tokenizer.decode(
                [
                    ids[
                        first_label_idx
                        - 1
                    ]
                ],
                skip_special_tokens=
                    False,
                clean_up_tokenization_spaces=
                    False,
            )
        ),
    )

    print(
        "LAST_K:",
        LAST_K,
    )

    print(
        "measurement indices:",
        idxs,
    )

    print(
        "measurement tokens:",
        [
            repr(
                tokenizer.decode(
                    [
                        ids[
                            i
                        ]
                    ],
                    skip_special_tokens=
                        False,
                    clean_up_tokenization_spaces=
                        False,
                )
            )
            for i
            in idxs
        ],
    )

    # --------------------------------------------------------
    # Explicit semantic assertions requested by the experiment:
    # --------------------------------------------------------
    # If LAST_K == [-1], selected position must be first label token.
    if (
        normalize_last_k_spec(
            LAST_K
        )
        == [
            -1
        ]
    ):
        assert idxs == [
            first_label_idx
        ]

    # If LAST_K == [-2], selected position must be exactly one token
    # before the first label token.
    if (
        normalize_last_k_spec(
            LAST_K
        )
        == [
            -2
        ]
    ):
        assert idxs == [
            first_label_idx
            - 1
        ]

    print(
        "prompt tail:\n",
        prompt[
            -800:
        ],
    )

## 8. Batched layer-20 Gemma Scope SAE extraction and cached streaming aggregation

In [ ]:
@torch.no_grad()
def extract_batch_sae_vectors(
    prompts: List[str],
    query_spans: List[Tuple[int, int]],
    gold_labels: List[str],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return:
        values: [batch, n_features]
            Feature statistic used for DPS-C (preactivation by default).

        fired: [batch, n_features] bool
            True when the feature's actual sparse SAE post-activation is
            nonzero on at least one selected measurement token for the prompt.

    The layer-20 forward hook captures residual-post representations.

    LAST_K is anchored to the first token of each example's final gold label,
    so `gold_labels` is required even when labels contain multiple tokens.
    """
    if not (
        len(prompts)
        == len(query_spans)
        == len(gold_labels)
    ):
        raise ValueError(
            "prompts, query_spans, and gold_labels must have equal length."
        )
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )
    offset_mapping = enc.pop("offset_mapping").cpu().numpy()
    attention_mask_cpu = enc["attention_mask"].cpu().numpy()
    input_ids_cpu = enc["input_ids"].cpu().numpy()

    selected_indices = []
    for i in range(len(prompts)):
        selected_indices.append(
            select_measurement_indices(
                input_ids_cpu[i],
                attention_mask_cpu[i],
                offset_mapping[i],
                query_spans[i],
                prompt_text=prompts[i],
                gold_label=gold_labels[i],
            )
        )

    input_ids = enc["input_ids"].to(MODEL_INPUT_DEVICE)
    attention_mask = enc["attention_mask"].to(MODEL_INPUT_DEVICE)

    captured = {}
    block = model.get_submodule(f"model.layers.{TARGET_LAYER}")

    def hook_fn(module, inputs, output):
        x = output[0] if isinstance(output, tuple) else output
        captured["x"] = x.detach()
        return output

    handle = block.register_forward_hook(hook_fn)
    try:
        # Skip the LM head when possible; we only need hidden states.
        backbone = model.model if hasattr(model, "model") else model
        _ = backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
    finally:
        handle.remove()

    if "x" not in captured:
        raise RuntimeError(f"Failed to capture model.layers.{TARGET_LAYER}")

    layer_out = captured["x"]  # [B, seq, d_model]

    vectors = []
    fired_vectors = []

    for i, idxs in enumerate(
        selected_indices
    ):
        idx = torch.tensor(
            idxs,
            device=
                layer_out.device,
            dtype=
                torch.long,
        )

        hidden = (
            layer_out[
                i
            ]
            .index_select(
                0,
                idx,
            )
        )

        pre = sae.encode_pre(
            hidden.to(
                sae.device
            )
        )

        # Actual sparse SAE activation.
        post = sae.activation_fn(
            pre
        )

        vals = (
            pre
            if USE_SAE_PREACTIVATIONS
            else post
        )

        vectors.append(
            vals.mean(
                dim=0
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )

        # Prompt-level firing indicator.
        #
        # A feature counts as "fired on this prompt" when it has nonzero
        # post-activation at ANY selected measurement token.
        fired_vectors.append(
            (
                post
                > 0
            )
            .any(
                dim=0
            )
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.bool_
            )
        )

    return (
        np.stack(
            vectors,
            axis=0,
        ),
        np.stack(
            fired_vectors,
            axis=0,
        ),
    )

def task_plan_fingerprint(task_key: str, plan: pd.DataFrame) -> str:
    spec = TASK_SPECS[task_key]
    core = plan[
        [
            "query_global_id",
            "query_index",
            "query_label_id",
            "split",
            "varied_position",
            "varied_label_id",
            "demo_indices",
        ]
    ].copy()
    core["demo_indices"] = core["demo_indices"].map(
        lambda x: ",".join(map(str, x))
    )

    payload_head = {
        "task_key": task_key,
        "dataset": TASK_DATA[task_key]["dataset_name"],
        "demo_source": TASK_DATA[task_key]["demo_source"],
        "query_source": TASK_DATA[task_key]["query_source"],
        "query_demo_same_dataset": TASK_DATA[task_key]["query_demo_same_dataset"],
        "model": MODEL_NAME,
        "layer": TARGET_LAYER,
                "sae": f"{SAE_REPO_ID}/{SAE_FILENAME}",
        "prompt_format": PROMPT_FORMAT_NAME,
        "last_k": normalize_last_k_spec(LAST_K),
        "measurement_anchor": MEASUREMENT_ANCHOR,
        "use_preactivations": USE_SAE_PREACTIVATIONS,
        "prompt_firing_mode": PROMPT_FIRING_MODE,
        "positions": POSITIONS_TO_TEST,
        "fixed_demos_per_label": FIXED_DEMOS_PER_LABEL[task_key],
        "labels": spec["label_to_word"],
    }
    blob = (
        json.dumps(payload_head, sort_keys=True)
        + "\n"
        + core.to_csv(index=False)
    )
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]

def cache_path_for_task(task_key: str, plan: pd.DataFrame) -> Path:
    fp = task_plan_fingerprint(task_key, plan)
    mode = "preact" if USE_SAE_PREACTIVATIONS else "sparse"
    last_slug = "query" if LAST_K is None else "_".join(
        f"m{abs(x)}" if x < 0 else f"p{x}"
        for x in normalize_last_k_spec(LAST_K)
    )
    anchor_slug = "firstLabel"
    return CACHE_ROOT / (
        f"{task_key}_{MODEL_SHORT_NAME}_L{TARGET_LAYER}_{mode}_"
        f"{PROMPT_FORMAT_NAME}_{anchor_slug}_{last_slug}_"
        f"P{'-'.join(map(str, POSITIONS_TO_TEST))}_{fp}.npz"
    )

def run_task_scan(task_key: str, plan: pd.DataFrame):
    spec = TASK_SPECS[task_key]
    P = len(POSITIONS_TO_TEST)
    K = spec["num_labels"]
    F = int(sae.n_features)
    pos_to_idx = {p: i for i, p in enumerate(POSITIONS_TO_TEST)}

    cache_path = cache_path_for_task(task_key, plan)
    if USE_CACHE and cache_path.exists() and not FORCE_RECOMPUTE:
        print(f"[{task_key}] loading cache: {cache_path}")
        z = np.load(cache_path, allow_pickle=False)
        required_cache_fields = {
            "mean_A_discovery",
            "mean_A_holdout",
            "count_discovery",
            "count_holdout",
            "fire_count_discovery",
            "fire_count_holdout",
            "n_prompts_discovery",
            "n_prompts_holdout",
        }

        if not required_cache_fields.issubset(
            set(
                z.files
            )
        ):
            print(
                f"[{task_key}] cache predates activation-frequency "
                "tracking; recomputing."
            )

        else:
            return {
                "mean_A_discovery":
                    z[
                        "mean_A_discovery"
                    ],

                "mean_A_holdout":
                    z[
                        "mean_A_holdout"
                    ],

                "count_discovery":
                    z[
                        "count_discovery"
                    ],

                "count_holdout":
                    z[
                        "count_holdout"
                    ],

                "fire_count_discovery":
                    z[
                        "fire_count_discovery"
                    ],

                "fire_count_holdout":
                    z[
                        "fire_count_holdout"
                    ],

                "n_prompts_discovery":
                    int(
                        z[
                            "n_prompts_discovery"
                        ]
                    ),

                "n_prompts_holdout":
                    int(
                        z[
                            "n_prompts_holdout"
                        ]
                    ),

                "cache_path":
                    str(
                        cache_path
                    ),
            }

    shape = (P, K, K, F)  # [position, varied_label, query_label, feature]
    count_shape = (P, K, K)

    sums = {
        "discovery": np.zeros(shape, dtype=np.float32),
        "holdout": np.zeros(shape, dtype=np.float32),
    }
    counts = {
        "discovery":
            np.zeros(
                count_shape,
                dtype=
                    np.int32,
            ),

        "holdout":
            np.zeros(
                count_shape,
                dtype=
                    np.int32,
            ),
    }

    # Prompt-level feature firing counts aggregated over all prompt rows.
    # Shape [F], not condition-specific, because the requested filter is the
    # average activation frequency across the complete prompt collection.
    fire_counts = {
        "discovery":
            np.zeros(
                F,
                dtype=
                    np.int64,
            ),

        "holdout":
            np.zeros(
                F,
                dtype=
                    np.int64,
            ),
    }

    n_prompts_by_split = {
        "discovery": 0,
        "holdout": 0,
    }

    n_rows = len(plan)
    n_batches = math.ceil(n_rows / BATCH_SIZE)

    for b, start in enumerate(range(0, n_rows, BATCH_SIZE), start=1):
        batch = plan.iloc[start:start + BATCH_SIZE]

        prompts = []
        spans = []
        gold_labels = []
        meta = []

        for row in batch.itertuples(index=False):
            demos = [
                TASK_DATA[task_key]["demo"][int(i)]
                for i in tuple(map(int, row.demo_indices))
            ]
            query = TASK_DATA[task_key]["query"][int(row.query_index)]
            prompt, span = build_prompt(task_key, demos, query)
            prompts.append(prompt)
            spans.append(span)
            gold_labels.append(
                label_word(
                    task_key,
                    query,
                )
            )
            meta.append(row)

        (
            vecs,
            fired_vecs,
        ) = extract_batch_sae_vectors(
            prompts,
            spans,
            gold_labels,
        )

        for (
            vec,
            fired,
            row,
        ) in zip(
            vecs,
            fired_vecs,
            meta,
        ):
            split = str(
                row.split
            )

            p_idx = pos_to_idx[
                int(
                    row.varied_position
                )
            ]

            c = int(
                row.varied_label_id
            )

            y = int(
                row.query_label_id
            )

            sums[
                split
            ][
                p_idx,
                c,
                y,
                :,
            ] += vec

            counts[
                split
            ][
                p_idx,
                c,
                y,
            ] += 1

            fire_counts[
                split
            ] += (
                fired.astype(
                    np.int64
                )
            )

            n_prompts_by_split[
                split
            ] += 1

        if b % PRINT_EVERY_BATCHES == 0 or b == n_batches:
            print(
                f"[{task_key}] batch {b}/{n_batches} "
                f"({min(start + BATCH_SIZE, n_rows)}/{n_rows} prompts)"
            )

    out = {}

    for split in [
        "discovery",
        "holdout",
    ]:
        denom = np.maximum(
            counts[
                split
            ][
                ...,
                None,
            ],
            1,
        )

        out[
            f"mean_A_{split}"
        ] = (
            sums[
                split
            ]
            / denom
        ).astype(
            np.float32
        )

        out[
            f"count_{split}"
        ] = counts[
            split
        ]

        out[
            f"fire_count_{split}"
        ] = fire_counts[
            split
        ].astype(
            np.int64
        )

        out[
            f"n_prompts_{split}"
        ] = int(
            n_prompts_by_split[
                split
            ]
        )

    out[
        "cache_path"
    ] = str(
        cache_path
    )

    if USE_CACHE:
        print(f"[{task_key}] saving cache: {cache_path}")
        np.savez_compressed(
            cache_path,

            mean_A_discovery=
                out[
                    "mean_A_discovery"
                ],

            mean_A_holdout=
                out[
                    "mean_A_holdout"
                ],

            count_discovery=
                out[
                    "count_discovery"
                ],

            count_holdout=
                out[
                    "count_holdout"
                ],

            fire_count_discovery=
                out[
                    "fire_count_discovery"
                ],

            fire_count_holdout=
                out[
                    "fire_count_holdout"
                ],

            n_prompts_discovery=
                np.asarray(
                    out[
                        "n_prompts_discovery"
                    ],
                    dtype=
                        np.int64,
                ),

            n_prompts_holdout=
                np.asarray(
                    out[
                        "n_prompts_holdout"
                    ],
                    dtype=
                        np.int64,
                ),
        )

    return out


## 9. Run all three task scans


In [ ]:
TASK_SCAN = {}

for task_key in TASK_ORDER:
    print("\n" + "=" * 80)
    print("RUNNING:", TASK_SPECS[task_key]["display_name"])
    print("=" * 80)
    TASK_SCAN[task_key] = run_task_scan(
        task_key,
        POSITION_PLANS[task_key],
    )

    for split in ["discovery", "holdout"]:
        print(
            task_key,
            split,
            TASK_SCAN[task_key][f"mean_A_{split}"].shape,
            "counts min/max:",
            TASK_SCAN[task_key][f"count_{split}"].min(),
            TASK_SCAN[task_key][f"count_{split}"].max(),
        )


# ============================================================
# Global prompt-level SAE activation frequency
# ============================================================
#
# Requested definition:
#   frequency(f)
#     = number of prompts on which feature f fired
#       / total number of prompts.
#
# "Fired" uses the TRUE sparse JumpReLU post-activation, whereas the
# DPS-C equations continue to use preactivation.
#
# By default we aggregate ALL AGNews + TREC + Yahoo prompts and both
# discovery + holdout partitions.
# ============================================================

GLOBAL_FEATURE_FIRE_COUNT = np.zeros(
    int(
        sae.n_features
    ),
    dtype=
        np.int64,
)

GLOBAL_ACTIVATION_FREQUENCY_N_PROMPTS = 0

TASK_ACTIVATION_FREQUENCY = {}

for task_key in TASK_ORDER:
    task_fire_count = np.zeros(
        int(
            sae.n_features
        ),
        dtype=
            np.int64,
    )

    task_n_prompts = 0

    for split in ACTIVATION_FREQUENCY_SPLITS:
        if split not in (
            "discovery",
            "holdout",
        ):
            raise ValueError(
                f"Unknown ACTIVATION_FREQUENCY_SPLITS entry: "
                f"{split!r}"
            )

        task_fire_count += (
            TASK_SCAN[
                task_key
            ][
                f"fire_count_{split}"
            ].astype(
                np.int64
            )
        )

        task_n_prompts += int(
            TASK_SCAN[
                task_key
            ][
                f"n_prompts_{split}"
            ]
        )

    GLOBAL_FEATURE_FIRE_COUNT += (
        task_fire_count
    )

    GLOBAL_ACTIVATION_FREQUENCY_N_PROMPTS += (
        task_n_prompts
    )

    TASK_ACTIVATION_FREQUENCY[
        task_key
    ] = (
        task_fire_count
        / max(
            task_n_prompts,
            1,
        )
    ).astype(
        np.float32
    )


FEATURE_ACTIVATION_FREQUENCY = (
    GLOBAL_FEATURE_FIRE_COUNT
    / max(
        GLOBAL_ACTIVATION_FREQUENCY_N_PROMPTS,
        1,
    )
).astype(
    np.float32
)


print(
    "\n"
    + "=" * 80
)

print(
    "GLOBAL SAE ACTIVATION FREQUENCY"
)

print(
    "=" * 80
)

print(
    "splits included:",
    ACTIVATION_FREQUENCY_SPLITS,
)

print(
    "total prompts:",
    GLOBAL_ACTIVATION_FREQUENCY_N_PROMPTS,
)

print(
    "firing definition:",
    PROMPT_FIRING_MODE,
)

print(
    "minimum frequency threshold:",
    f"{100 * ACTIVATION_FREQUENCY_THRESHOLD:.2f}%",
)

print(
    "features meeting frequency threshold:",
    int(
        (
            FEATURE_ACTIVATION_FREQUENCY
            >= float(
                ACTIVATION_FREQUENCY_THRESHOLD
            )
        ).sum()
    ),
    "/",
    len(
        FEATURE_ACTIVATION_FREQUENCY
    ),
)

frequency_quantiles = np.quantile(
    FEATURE_ACTIVATION_FREQUENCY,
    [
        0.00,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        1.00,
    ],
)

display(
    pd.DataFrame({
        "quantile":
            [
                "min",
                "25%",
                "50%",
                "75%",
                "90%",
                "95%",
                "99%",
                "max",
            ],

        "activation_frequency":
            frequency_quantiles,

        "activation_frequency_pct":
            100
            * frequency_quantiles,
    })
)


## 10. Compute the 20 strict DPS-C equations and aggregate them across tasks


In [ ]:
def compute_task_dps_stats(task_key: str, mean_A: np.ndarray):
    """
    mean_A: [position, varied_label, query_label, feature]
    DPS-C criterion:
      matched preactivation < every nonmatched preactivation

    Returns positive DPS-C gaps/conditions:
      gap = min(nonmatched) - matched
      gap[position, query_label, feature]
    """
    spec = TASK_SPECS[task_key]
    P, K, K2, F = mean_A.shape
    assert K == K2 == spec["num_labels"]

    gap = np.empty((P, K, F), dtype=np.float32)
    condition = np.empty((P, K, F), dtype=bool)
    matched_mean = np.empty((P, K, F), dtype=np.float32)

    for y in spec["label_ids"]:
        vals = mean_A[:, :, int(y), :]       # [P, varied_label, F]
        matched = vals[:, int(y), :]
        other = [int(c) for c in spec["label_ids"] if int(c) != int(y)]
        # DPS-C: matched preactivation should be the MINIMUM.
        min_mismatch = vals[:, other, :].min(axis=1)

        # Positive gap means the matched condition is strictly lower
        # than every mismatched first-demo condition.
        g = min_mismatch - matched
        gap[:, int(y), :] = g
        matched_mean[:, int(y), :] = matched
        condition[:, int(y), :] = g > DPS_TIE_TOL

    return {
        "gap": gap,
        "condition": condition,
        "count": condition.sum(axis=1),       # [P, F]
        "mean_gap": gap.mean(axis=1),         # [P, F]
        "min_gap": gap.min(axis=1),           # [P, F]
        "matched_mean": matched_mean,
    }

TASK_STATS = {split: {} for split in ["discovery", "holdout"]}

for split in ["discovery", "holdout"]:
    for task_key in TASK_ORDER:
        TASK_STATS[split][task_key] = compute_task_dps_stats(
            task_key,
            TASK_SCAN[task_key][f"mean_A_{split}"],
        )

# Equation metadata and slices.
equation_rows = []
TASK_EQ_SLICES = {}
cursor = 0

for task_key in TASK_ORDER:
    spec = TASK_SPECS[task_key]
    start = cursor
    for l in spec["label_ids"]:
        equation_rows.append({
            "equation_index": cursor,
            "task_key": task_key,
            "task": spec["display_name"],
            "label_id": int(l),
            "label": spec["label_to_word"][l],
            "equation_name": f"{spec['display_name']}::{spec['label_to_word'][l]}",
        })
        cursor += 1
    TASK_EQ_SLICES[task_key] = slice(start, cursor)

EQUATION_DF = pd.DataFrame(equation_rows)
NUM_EQUATIONS = len(EQUATION_DF)
assert NUM_EQUATIONS == 20

display(EQUATION_DF)

GLOBAL_STATS = {}

for split in ["discovery", "holdout"]:
    global_gap = np.concatenate(
        [TASK_STATS[split][k]["gap"] for k in TASK_ORDER],
        axis=1,
    )
    global_condition = np.concatenate(
        [TASK_STATS[split][k]["condition"] for k in TASK_ORDER],
        axis=1,
    )
    global_matched = np.concatenate(
        [TASK_STATS[split][k]["matched_mean"] for k in TASK_ORDER],
        axis=1,
    )

    # Equal-equation weighting: each of the 20 equations contributes equally.
    mean_gap_all20 = global_gap.mean(axis=1)

    # Equal-task weighting: AGNews, TREC, Yahoo each contribute 1/3.
    mean_gap_task_balanced = np.stack(
        [
            TASK_STATS[split][k]["gap"].mean(axis=1)
            for k in TASK_ORDER
        ],
        axis=0,
    ).mean(axis=0)

    GLOBAL_STATS[split] = {
        "gap": global_gap,                         # [P, 20, F]
        "condition": global_condition,             # [P, 20, F]
        "matched_mean": global_matched,            # [P, 20, F]
        "count": global_condition.sum(axis=1),     # [P, F]
        "mean_gap_all20": mean_gap_all20,          # [P, F]
        "mean_gap_task_balanced": mean_gap_task_balanced,  # [P, F]
        "min_gap_all20": global_gap.min(axis=1),   # [P, F]
    }

print("GLOBAL tensor shapes:")
for split in ["discovery", "holdout"]:
    for key, arr in GLOBAL_STATS[split].items():
        print(split, key, arr.shape)


## 11. Core result 1 — choose `(threshold, demo_position)` and report the qualifying features

`report_dps_features(threshold, demo_position, split)`:

- counts every feature whose number of satisfied equations is at least `threshold`;
- reports **mean signed activation gap over all 20 equations**;
- reports **mean gap over equations that are actually satisfied**;
- reports an **equal-task-weighted mean gap**, so Yahoo's 10 labels do not dominate the gap summary;
- reports per-task equation counts and per-task gaps for every selected feature;
- if selecting on discovery, also prints same-position holdout replication.

Optional `min_per_task` can prevent a lower global threshold from being met almost entirely by one dataset, e.g.

```python
min_per_task={"agnews": 3, "trec": 4, "yahoo": 7}
```

For the strongest criterion, simply use `threshold=20`.


In [ ]:
POSITION_TO_IDX = {p: i for i, p in enumerate(POSITIONS_TO_TEST)}
N_FEATURES = int(sae.n_features)

def _selection_mask(
    threshold: int,
    demo_position: int,
    split: str,
    min_per_task: Optional[Dict[str, int]] = None,
):
    if split not in GLOBAL_STATS:
        raise ValueError(f"split must be one of {list(GLOBAL_STATS)}")
    if demo_position not in POSITION_TO_IDX:
        raise ValueError(f"demo_position must be one of {POSITIONS_TO_TEST}")
    if not (0 <= int(threshold) <= NUM_EQUATIONS):
        raise ValueError(f"threshold must be in 0..{NUM_EQUATIONS}")

    p_idx = POSITION_TO_IDX[int(demo_position)]
    mask = GLOBAL_STATS[split]["count"][p_idx] >= int(threshold)

    if min_per_task is not None:
        for task_key, min_count in min_per_task.items():
            if task_key not in TASK_ORDER:
                raise ValueError(f"Unknown task_key={task_key!r}")
            task_count = TASK_STATS[split][task_key]["count"][p_idx]
            mask &= task_count >= int(min_count)

    return mask

def build_feature_summary_df(
    demo_position: int,
    split: str,
) -> pd.DataFrame:
    p_idx = POSITION_TO_IDX[int(demo_position)]
    gs = GLOBAL_STATS[split]

    gap = gs["gap"][p_idx]              # [20, F]
    cond = gs["condition"][p_idx]       # [20, F]
    count = gs["count"][p_idx]          # [F]

    sat_gap_sum = np.where(cond, gap, 0.0).sum(axis=0)
    mean_gap_satisfied = np.divide(
        sat_gap_sum,
        np.maximum(count, 1),
    )

    data = {
        "feature_idx":
            np.arange(
                N_FEATURES,
                dtype=int,
            ),

        "activation_frequency":
            FEATURE_ACTIVATION_FREQUENCY.astype(
                np.float32
            ),

        "activation_frequency_pct":
            (
                100.0
                * FEATURE_ACTIVATION_FREQUENCY
            ).astype(
                np.float32
            ),

        "n_equations_satisfied":
            count.astype(
                int
            ),
        "mean_signed_gap_all20": gs["mean_gap_all20"][p_idx],
        "mean_signed_gap_task_balanced": gs["mean_gap_task_balanced"][p_idx],
        "mean_gap_satisfied_equations": mean_gap_satisfied,
        "min_gap_all20": gs["min_gap_all20"][p_idx],
    }

    for task_key in TASK_ORDER:
        ts = TASK_STATS[split][task_key]
        data[f"{task_key}_n_satisfied"] = ts["count"][p_idx].astype(int)
        data[f"{task_key}_mean_gap"] = ts["mean_gap"][p_idx]
        data[f"{task_key}_min_gap"] = ts["min_gap"][p_idx]

    return pd.DataFrame(data)

def report_dps_features(
    threshold: int = 20,
    demo_position: int = 1,
    split: str = "discovery",
    min_per_task: Optional[Dict[str, int]] = None,
    min_activation_frequency: Optional[
        float
    ] = ACTIVATION_FREQUENCY_THRESHOLD,
    top_n: int = 40,
    save_csv: bool = True,
):
    p_idx = POSITION_TO_IDX[int(demo_position)]
    # --------------------------------------------------------
    # Structural DPS-C criterion (preactivation-based): matched is minimum
    # --------------------------------------------------------
    dps_mask = _selection_mask(
        threshold=
            threshold,
        demo_position=
            demo_position,
        split=
            split,
        min_per_task=
            min_per_task,
    )

    n_before_frequency_filter = int(
        dps_mask.sum()
    )

    # --------------------------------------------------------
    # Additional sparse-SAE firing-frequency criterion
    # --------------------------------------------------------
    mask = (
        dps_mask.copy()
    )

    if (
        min_activation_frequency
        is not None
    ):
        min_activation_frequency = float(
            min_activation_frequency
        )

        if not (
            0.0
            <= min_activation_frequency
            <= 1.0
        ):
            raise ValueError(
                "min_activation_frequency must lie in [0,1] "
                "or be None."
            )

        mask &= (
            FEATURE_ACTIVATION_FREQUENCY
            >= min_activation_frequency
        )

    selected = (
        np.flatnonzero(
            mask
        )
    )

    summary = (
        build_feature_summary_df(
            demo_position,
            split,
        )
    )

    selected_df = (
        summary.loc[
            mask
        ]
        .copy()
        .sort_values(
        [
            "n_equations_satisfied",
            "mean_signed_gap_all20",
            "min_gap_all20",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    )

    gs = GLOBAL_STATS[
        split
    ]
    pos_gap = gs["gap"][p_idx][:, mask] if len(selected) else np.empty((NUM_EQUATIONS, 0))
    pos_cond = gs["condition"][p_idx][:, mask] if len(selected) else np.empty((NUM_EQUATIONS, 0), dtype=bool)

    mean_signed_gap = float(pos_gap.mean()) if pos_gap.size else np.nan
    mean_task_balanced_gap = (
        float(gs["mean_gap_task_balanced"][p_idx, mask].mean())
        if len(selected) else np.nan
    )
    mean_satisfied_gap = (
        float(pos_gap[pos_cond].mean())
        if pos_gap.size and pos_cond.any() else np.nan
    )

    print("=" * 80)
    print(
        f"{split.upper()} | position={demo_position} | "
        f"threshold >= {threshold}/{NUM_EQUATIONS}"
    )
    if min_per_task:
        print(
            "Additional per-task minima:",
            min_per_task,
        )

    print(
        "DPS-C features before activation-frequency filter:",
        n_before_frequency_filter,
    )

    if (
        min_activation_frequency
        is None
    ):
        print(
            "Activation-frequency filter: DISABLED"
        )

    else:
        print(
            "Activation-frequency filter:",
            f">= {100 * min_activation_frequency:.2f}% "
            f"over {GLOBAL_ACTIVATION_FREQUENCY_N_PROMPTS} prompts",
        )

    print(
        "DPS-C features retained after frequency filter:",
        len(
            selected
        ),
    )

    if n_before_frequency_filter > 0:
        print(
            "Retention fraction:",
            f"{len(selected) / n_before_frequency_filter:.3f}",
        )

    print(
        "Number of qualifying SAE features:",
        len(
            selected
        ),
    )
    print("Mean DPS-C gap across all 20 equations:", mean_signed_gap)
    print("Equal-task-weighted mean DPS-C gap:", mean_task_balanced_gap)
    print("Mean DPS-C gap over satisfied equations only:", mean_satisfied_gap)

    # Same-position independent holdout replication when discovery is used for selection.
    if split == "discovery" and len(selected):
        hold_count = GLOBAL_STATS["holdout"]["count"][p_idx, selected]
        hold_retain = hold_count >= int(threshold)
        if min_per_task is not None:
            for task_key, min_count in min_per_task.items():
                hold_retain &= (
                    TASK_STATS["holdout"][task_key]["count"][p_idx, selected]
                    >= int(min_count)
                )
        print(
            "Same-position holdout replication among discovery-selected features:",
            int(hold_retain.sum()),
            "/",
            len(selected),
            f"({hold_retain.mean():.3f})",
        )

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            f"Features satisfying ≥{threshold}/{NUM_EQUATIONS}",
            "Average activation gap",
        ),
        horizontal_spacing=0.18,
    )
    fig.add_trace(
        go.Bar(
            x=[f"Position {demo_position}"],
            y=[len(selected)],
            name="Number of features",
            marker_color=COUNT_COLOR,
            text=[len(selected)],
            textposition="outside",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=["All 20 equations", "Satisfied equations only", "Task-balanced"],
            y=[mean_signed_gap, mean_satisfied_gap, mean_task_balanced_gap],
            name="Mean gap",
            marker_color=COUNT_COLOR,
            text=[
                f"{mean_signed_gap:.4f}" if np.isfinite(mean_signed_gap) else "NA",
                f"{mean_satisfied_gap:.4f}" if np.isfinite(mean_satisfied_gap) else "NA",
                f"{mean_task_balanced_gap:.4f}" if np.isfinite(mean_task_balanced_gap) else "NA",
            ],
            textposition="outside",
        ),
        row=1,
        col=2,
    )
    fig.update_yaxes(title_text="Number of features", row=1, col=1)
    fig.update_yaxes(title_text="Mean DPS-C gap (min mismatch − matched)", row=1, col=2)
    fig.update_layout(
        title=dict(
            text=(
                f"DPS-C cross-task feature search — {split}, "
                f"position {demo_position}, threshold ≥{threshold}/{NUM_EQUATIONS}"
                + (
                    ""
                    if min_activation_frequency is None
                    else (
                        f", firing ≥"
                        f"{100 * min_activation_frequency:.2f}%"
                    )
                )
            ),
            x=0.0,
            xanchor="left",
        ),
        font=dict(family=PLOT_FONT, size=15),
        width=1050,
        height=500,
        showlegend=False,
    )
    fig.show()

    print("\nTop qualifying features:")
    display(selected_df.head(int(top_n)))

    if save_csv:
        freq_slug = (
            "freqNone"
            if min_activation_frequency is None
            else (
                f"freq{100 * min_activation_frequency:.3g}pct"
                .replace(
                    ".",
                    "p",
                )
            )
        )

        suffix = (
            f"{split}_pos{demo_position}_thr{threshold}_"
            f"{freq_slug}"
        )
        csv_path = OUTPUT_ROOT / f"selected_features_{suffix}.csv"
        selected_df.to_csv(csv_path, index=False)
        print("Saved:", csv_path)

    return selected.tolist(), selected_df


## Activation-frequency filtering of DPS-C candidates

The DPS-C equations above are intentionally defined using **SAE preactivation**,
which allows us to compare the strength of a fixed representational direction
even when it lies below the SAE's sparse firing threshold. For downstream
analysis, however, we additionally remove features that are essentially never
active under the SAE itself.

For each feature \(f\), we compute its **prompt-level activation frequency**

\[
\mathrm{Freq}(f)
=
\frac{
\#\{\text{prompts on which the JumpReLU post-activation of } f \text{ is nonzero}\}
}{
\#\{\text{prompts}\}
}.
\]

By default, the denominator contains all prompt conditions used across AGNews,
TREC, and Yahoo, including both selection and held-out partitions. A feature is
retained only when \(\mathrm{Freq}(f)\ge 0.01\). Thus, the structural DPS-C
criterion remains preactivation-based, while the additional \(1\%\) threshold
ensures that retained features correspond to directions that genuinely fire
with non-negligible frequency.

In [ ]:
selected_features, selected_feature_df = report_dps_features(
    threshold=
        20,

    demo_position=
        1,

    split=
        "discovery",

    # Additional post-activation firing-frequency filter for DPS-C candidates.
    # Default = 1% across all AGNews/TREC/Yahoo prompts.
    min_activation_frequency=
        ACTIVATION_FREQUENCY_THRESHOLD,
)


## Chance-level non-DPS-C feature control

As a causal control, we sample features that do **not** exhibit the strong
activation-level DPS-C signature. Rather than drawing arbitrary features, which
could accidentally include moderately DPS-like directions, we sample from a
near-chance population.

For a feature with no systematic DPS-C structure, the matched first-demo
condition is one of \(K\) possible maxima for each \(K\)-class task. Hence the
expected number of satisfied conditions is approximately one per task:
\(1+1+1=3\) across AGNews, TREC, and Yahoo. We therefore use features satisfying
**3 or 4 of the 20 equations** as the default control pool and sample 10
features reproducibly. The feature pool is defined on the same activation-level
selection split as the main DPS-C feature search; causal effects are still
evaluated on the held-out split.

In [ ]:
# ============================================================
# Random chance-level non-DPS-C control configuration
# ============================================================
#
# IMPORTANT:
# This configuration must appear BEFORE
# sample_random_chance_level_control_features(...), because these
# values are used as default function arguments.
#
# Chance intuition:
#   AGNews -> ~1 matched maximum in expectation
#   TREC   -> ~1 matched maximum in expectation
#   Yahoo  -> ~1 matched maximum in expectation
#   Total  -> roughly 3/20 equations by chance.
#
# Therefore 3/20 or 4/20 is a natural near-chance control pool.
#
# To use exactly 4/20 only:
#     RANDOM_CONTROL_DPS_COUNTS = (4,)
#
RANDOM_CONTROL_N_FEATURES = 10
RANDOM_CONTROL_DPS_COUNTS = (3, 4)
RANDOM_CONTROL_SEED = 20260809
RANDOM_CONTROL_SELECTION_POSITION = 1
RANDOM_CONTROL_SELECTION_SPLIT = "discovery"

print(
    "RANDOM_CONTROL_N_FEATURES:",
    RANDOM_CONTROL_N_FEATURES,
)

print(
    "RANDOM_CONTROL_DPS_COUNTS:",
    RANDOM_CONTROL_DPS_COUNTS,
)

print(
    "RANDOM_CONTROL_SELECTION_POSITION:",
    RANDOM_CONTROL_SELECTION_POSITION,
)

print(
    "RANDOM_CONTROL_SELECTION_SPLIT:",
    RANDOM_CONTROL_SELECTION_SPLIT,
)

In [ ]:
# ============================================================
# Reproducible chance-level non-DPS-C control feature sampling
# ============================================================

def sample_random_chance_level_control_features(
    n_features: int = RANDOM_CONTROL_N_FEATURES,
    allowed_dps_counts: Sequence[int] = RANDOM_CONTROL_DPS_COUNTS,
    *,
    demo_position: int = RANDOM_CONTROL_SELECTION_POSITION,
    split: str = RANDOM_CONTROL_SELECTION_SPLIT,
    seed: int = RANDOM_CONTROL_SEED,
    exclude_features: Optional[Sequence[int]] = None,
    min_activation_frequency: Optional[
        float
    ] = ACTIVATION_FREQUENCY_THRESHOLD,
):
    """
    Randomly sample near-chance non-DPS-C features.

    Default control pool:
        features satisfying exactly 3 or 4 of the 20 activation-level
        DPS-C equations at position 1 on the selection split.

    Returns
    -------
    control_features:
        List[int] of sampled SAE feature indices.

    control_df:
        Per-feature activation-level statistics on both selection and
        holdout splits.
    """
    n_features = int(
        n_features
    )

    if n_features <= 0:
        raise ValueError(
            "n_features must be positive."
        )

    allowed_dps_counts = tuple(
        sorted(
            set(
                int(x)
                for x
                in allowed_dps_counts
            )
        )
    )

    if not allowed_dps_counts:
        raise ValueError(
            "allowed_dps_counts cannot be empty."
        )

    if any(
        x < 0
        or x > NUM_EQUATIONS
        for x
        in allowed_dps_counts
    ):
        raise ValueError(
            f"allowed_dps_counts must lie in 0..{NUM_EQUATIONS}."
        )

    summary = build_feature_summary_df(
        demo_position=
            int(
                demo_position
            ),
        split=
            str(
                split
            ),
    )

    mask = summary[
        "n_equations_satisfied"
    ].isin(
        allowed_dps_counts
    )

    # Match the DPS-C activity filter so the causal control is not
    # trivially weakened by dead / ultra-rare SAE features.
    if (
        min_activation_frequency
        is not None
    ):
        min_activation_frequency = float(
            min_activation_frequency
        )

        if not (
            0.0
            <= min_activation_frequency
            <= 1.0
        ):
            raise ValueError(
                "min_activation_frequency must lie in [0,1] "
                "or be None."
            )

        mask &= (
            summary[
                "activation_frequency"
            ]
            >= min_activation_frequency
        )

    exclude = set(
        int(x)
        for x
        in (
            exclude_features
            if exclude_features is not None
            else []
        )
    )

    # Always exclude the strict DPS-C set explicitly as a safety check.
    exclude.update(
        int(x)
        for x
        in selected_feature_df[
            "feature_idx"
        ].tolist()
    )

    if exclude:
        mask &= ~summary[
            "feature_idx"
        ].isin(
            exclude
        )

    candidates = (
        summary.loc[
            mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    if len(
        candidates
    ) < n_features:
        raise RuntimeError(
            f"Only {len(candidates)} candidate control features satisfy "
            f"counts={allowed_dps_counts}; requested {n_features}. "
            "Broaden RANDOM_CONTROL_DPS_COUNTS if needed."
        )

    rng = np.random.default_rng(
        int(
            seed
        )
    )

    sampled_rows = rng.choice(
        len(
            candidates
        ),
        size=
            n_features,
        replace=
            False,
    )

    control_df = (
        candidates.iloc[
            sampled_rows
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    # Add same-position holdout statistics for transparent reporting.
    holdout_summary = (
        build_feature_summary_df(
            demo_position=
                int(
                    demo_position
                ),
            split=
                "holdout",
        )
        .set_index(
            "feature_idx"
        )
    )

    control_df[
        "holdout_n_equations_satisfied"
    ] = [
        int(
            holdout_summary.loc[
                int(f),
                "n_equations_satisfied",
            ]
        )
        for f
        in control_df[
            "feature_idx"
        ]
    ]

    control_df[
        "holdout_mean_signed_gap_all20"
    ] = [
        float(
            holdout_summary.loc[
                int(f),
                "mean_signed_gap_all20",
            ]
        )
        for f
        in control_df[
            "feature_idx"
        ]
    ]

    control_df.insert(
        0,
        "control_rank",
        np.arange(
            1,
            len(
                control_df
            )
            + 1,
            dtype=int,
        ),
    )

    control_features = (
        control_df[
            "feature_idx"
        ]
        .astype(
            int
        )
        .tolist()
    )

    print(
        "=" * 80
    )

    print(
        "RANDOM CHANCE-LEVEL NON-DPS-C CONTROL"
    )

    print(
        "selection split:",
        split,
    )

    print(
        "selection position:",
        demo_position,
    )

    print(
        "allowed DPS-C equation counts:",
        allowed_dps_counts,
    )

    print(
        "minimum activation frequency:",
        (
            "disabled"
            if min_activation_frequency is None
            else (
                f"{100 * min_activation_frequency:.2f}%"
            )
        ),
    )

    print(
        "candidate pool size:",
        len(
            candidates
        ),
    )

    print(
        "random seed:",
        seed,
    )

    print(
        "sampled control features:",
        control_features,
    )

    display(
        control_df[
            [
                "control_rank",
                "feature_idx",
                "activation_frequency",
                "activation_frequency_pct",
                "n_equations_satisfied",
                "mean_signed_gap_all20",
                "holdout_n_equations_satisfied",
                "holdout_mean_signed_gap_all20",
            ]
        ]
    )

    return (
        control_features,
        control_df,
    )


RANDOM_CONTROL_FEATURES, random_control_feature_df = (
    sample_random_chance_level_control_features(
        n_features=
            RANDOM_CONTROL_N_FEATURES,
        allowed_dps_counts=
            RANDOM_CONTROL_DPS_COUNTS,
        demo_position=
            RANDOM_CONTROL_SELECTION_POSITION,
        split=
            RANDOM_CONTROL_SELECTION_SPLIT,
        seed=
            RANDOM_CONTROL_SEED,
        exclude_features=
            selected_features,

        min_activation_frequency=
            ACTIVATION_FREQUENCY_THRESHOLD,
    )
)

## 12. Optional: independent all-feature position profile ("blue-bar" analogue)

This plot does **not** track a fixed feature family. At each position it independently asks:

> How many features in the entire SAE satisfy at least `threshold` of the 20 equations here?

Use this to distinguish **global availability of DPS-C-like features** from the primacy-focused fixed-set analysis below.


In [ ]:
def plot_independent_position_profile(
    threshold: int = 20,
    split: str = "holdout",
    min_per_task: Optional[Dict[str, int]] = None,
):
    rows = []
    for p in POSITIONS_TO_TEST:
        mask = _selection_mask(threshold, p, split, min_per_task)
        p_idx = POSITION_TO_IDX[p]
        rows.append({
            "position": p,
            "n_features": int(mask.sum()),
            "mean_gap_selected": (
                float(GLOBAL_STATS[split]["mean_gap_all20"][p_idx, mask].mean())
                if mask.any() else np.nan
            ),
        })
    df = pd.DataFrame(rows)

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12,
        subplot_titles=(
            f"Independent features satisfying ≥{threshold}/{NUM_EQUATIONS}",
            "Mean signed gap among independently qualifying features",
        ),
    )
    fig.add_trace(
        go.Bar(
            x=df["position"],
            y=df["n_features"],
            marker_color=COUNT_COLOR,
            name="Independent qualifying features",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            x=df["position"],
            y=df["mean_gap_selected"],
            marker_color=COUNT_COLOR,
            name="Mean gap",
        ),
        row=2,
        col=1,
    )
    fig.update_yaxes(title_text="Number of features", row=1, col=1)
    fig.update_yaxes(title_text="Mean signed gap", row=2, col=1)
    fig.update_xaxes(title_text="Varied demonstration position", row=2, col=1)
    fig.update_layout(
        title=dict(
            text=f"Position-wise independent DPS-C profile ({split})",
            x=0.0,
            xanchor="left",
        ),
        font=dict(family=PLOT_FONT, size=15),
        width=950,
        height=720,
        showlegend=False,
    )
    fig.show()
    display(df)
    return df


## 13. Core result 2 — primacy-focused fixed position-1 feature set ("orange bars")

This is the camera-ready primacy analysis.

1. Select **all** features satisfying the requested threshold at **position 1 on discovery**.
2. Freeze that feature set.
3. On **holdout only**, move the variable demonstration through all tested positions.
4. At every position report:
   - how many of the fixed position-1 features still satisfy the threshold;
   - their retention fraction;
   - their average number of satisfied equations;
   - the **mean signed activation gap across the entire fixed feature set**, whether or not each feature still passes the threshold;
   - per-task mean gaps.

The fixed-set gap is the primary continuous companion to the orange count bars. It avoids the survivorship bias that would result from averaging only features that remain above threshold at each later position.


In [ ]:
def report_primacy_retention(
    threshold: int = 20,
    selection_position: int = 1,
    selection_split: str = "discovery",
    eval_split: str = "holdout",
    min_per_task: Optional[Dict[str, int]] = None,
    save_csv: bool = True,
):
    selected_mask = _selection_mask(
        threshold=threshold,
        demo_position=selection_position,
        split=selection_split,
        min_per_task=min_per_task,
    )
    selected = np.flatnonzero(selected_mask)

    print("=" * 80)
    print(
        f"Primacy feature set: {selection_split}, "
        f"position={selection_position}, threshold ≥{threshold}/{NUM_EQUATIONS}"
    )
    print("Number of fixed position-1-derived features:", len(selected))
    if min_per_task:
        print("Per-task minima:", min_per_task)

    if len(selected) == 0:
        print("No features selected; lower threshold or inspect discovery results.")
        return selected.tolist(), pd.DataFrame()

    rows = []

    for p in POSITIONS_TO_TEST:
        p_idx = POSITION_TO_IDX[p]
        gs = GLOBAL_STATS[eval_split]

        counts = gs["count"][p_idx, selected]
        survives = counts >= int(threshold)

        if min_per_task is not None:
            for task_key, min_count in min_per_task.items():
                survives &= (
                    TASK_STATS[eval_split][task_key]["count"][p_idx, selected]
                    >= int(min_count)
                )

        fixed_gap = gs["gap"][p_idx][:, selected]       # [20, selected]
        fixed_cond = gs["condition"][p_idx][:, selected]

        mean_gap_fixed = float(fixed_gap.mean())
        mean_gap_task_balanced_fixed = float(
            gs["mean_gap_task_balanced"][p_idx, selected].mean()
        )
        mean_gap_satisfied_fixed = (
            float(fixed_gap[fixed_cond].mean())
            if fixed_cond.any() else np.nan
        )

        if survives.any():
            surviving_gap = gs["gap"][p_idx][:, selected[survives]]
            mean_gap_survivors = float(surviving_gap.mean())
        else:
            mean_gap_survivors = np.nan

        row = {
            "position": int(p),
            "n_position1_features": int(len(selected)),
            "n_still_ge_threshold": int(survives.sum()),
            "retention_fraction": float(survives.mean()),
            "mean_equations_satisfied_fixed_set": float(counts.mean()),
            "mean_signed_gap_fixed_set_all20": mean_gap_fixed,
            "mean_signed_gap_fixed_set_task_balanced": mean_gap_task_balanced_fixed,
            "mean_gap_satisfied_equations_fixed_set": mean_gap_satisfied_fixed,
            "mean_signed_gap_survivors_all20": mean_gap_survivors,
        }

        # Per-task continuous and discrete summaries for the SAME fixed p1 set.
        for task_key in TASK_ORDER:
            ts = TASK_STATS[eval_split][task_key]
            row[f"{task_key}_mean_equations_satisfied_fixed_set"] = float(
                ts["count"][p_idx, selected].mean()
            )
            row[f"{task_key}_mean_signed_gap_fixed_set"] = float(
                ts["gap"][p_idx][:, selected].mean()
            )

        rows.append(row)

    df = pd.DataFrame(rows)

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12,
        subplot_titles=(
            (
                f"Position-1-derived features retaining ≥"
                f"{threshold}/{NUM_EQUATIONS} DPS-C equations"
            ),
            "Mean signed activation gap of the fixed position-1 feature set",
        ),
    )

    fig.add_trace(
        go.Bar(
            x=df["position"],
            y=df["n_still_ge_threshold"],
            marker_color=PRIMACY_COLOR,
            name="Position-1-derived features",
            text=df["n_still_ge_threshold"],
            textposition="outside",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Bar(
            x=df["position"],
            y=df["mean_signed_gap_fixed_set_all20"],
            marker_color=PRIMACY_COLOR,
            name="Fixed-set mean activation gap",
        ),
        row=2,
        col=1,
    )
    fig.add_hline(y=0.0, line_width=1, line_dash="dash", row=2, col=1)

    fig.update_yaxes(title_text="Number of features", row=1, col=1)
    fig.update_yaxes(title_text="Mean signed activation gap", row=2, col=1)
    fig.update_xaxes(title_text="Varied demonstration position", row=2, col=1)

    fig.update_layout(
        title=dict(
            text=(
                f"Primacy specificity of discovery position-{selection_position} "
                f"DPS-C features ({eval_split})"
            ),
            x=0.0,
            xanchor="left",
        ),
        font=dict(family=PLOT_FONT, size=15),
        width=1000,
        height=760,
        showlegend=False,
    )
    fig.show()

    print("\nPrimacy retention table:")
    display(df)

    if save_csv:
        csv_path = OUTPUT_ROOT / (
            f"primacy_retention_selpos{selection_position}_"
            f"thr{threshold}_{selection_split}_to_{eval_split}.csv"
        )
        df.to_csv(csv_path, index=False)
        print("Saved:", csv_path)

    return selected.tolist(), df


In [ ]:
# Main camera-ready call:
primacy_features, primacy_df = report_primacy_retention(
    threshold=20,
    selection_position=1,
    selection_split="discovery",
    eval_split="holdout",
)


## 14. Threshold sensitivity and per-task diagnostics


In [ ]:
def threshold_sensitivity_table(
    thresholds: Sequence[int] = (20, 19, 18, 17, 16, 15),
    split: str = "discovery",
):
    rows = []
    for threshold in thresholds:
        for p in POSITIONS_TO_TEST:
            p_idx = POSITION_TO_IDX[p]
            mask = GLOBAL_STATS[split]["count"][p_idx] >= int(threshold)

            row = {
                "threshold": int(threshold),
                "position": int(p),
                "n_features": int(mask.sum()),
                "mean_gap_selected_all20": (
                    float(GLOBAL_STATS[split]["mean_gap_all20"][p_idx, mask].mean())
                    if mask.any() else np.nan
                ),
            }

            for task_key in TASK_ORDER:
                row[f"{task_key}_mean_count_selected"] = (
                    float(TASK_STATS[split][task_key]["count"][p_idx, mask].mean())
                    if mask.any() else np.nan
                )
                row[f"{task_key}_mean_gap_selected"] = (
                    float(TASK_STATS[split][task_key]["mean_gap"][p_idx, mask].mean())
                    if mask.any() else np.nan
                )

            rows.append(row)
    return pd.DataFrame(rows)

# Example:
# sensitivity_df = threshold_sensitivity_table()
# display(sensitivity_df)


In [ ]:
sensitivity_df = threshold_sensitivity_table()
display(sensitivity_df)

## 15. Recommended camera-ready reporting

For a threshold \(T\), report both:

### Cross-task existence
At each position, the number of **all SAE features** satisfying at least \(T\) of the 20 equations.

### Primacy specificity
Select the full feature set satisfying at least \(T\) equations at position 1 on discovery. On holdout, report across positions:

1. number / fraction retaining at least \(T\) equations;
2. mean number of equations satisfied by the fixed set;
3. mean signed activation gap across all 20 equations for the fixed set;
4. per-task mean gaps.

The continuous gap profile is particularly important: binary equation retention can fall sharply when small positive gaps cross zero, whereas a simultaneous decay in the signed gap provides graded evidence of reduced first-position sensitivity.

For thresholds below 20, also inspect the per-task columns (or use `min_per_task`) because a global 20-equation count can otherwise be dominated by the task with the largest number of labels.


## Single-feature DPS-C bar visualization

This helper visualizes the activation-level DPS-C pattern for an individual SAE
feature. For a chosen task and demonstration position:

- the **x-axis** is the varied / first-demonstration label \(k\);
- grouped bars correspond to the **test-query label** \(\ell\);
- each bar reports \(A_f^{(p)}(k\!\to\!\ell)\);
- a red triangle marks the **matched condition** \(k=\ell\).

Thus, a clean DPS-C feature should show the marked bar as the tallest bar
within each first-demonstration-label group.

In [ ]:
def _resolve_dps_task_key(
    task_name: str,
) -> str:
    """
    Accept internal task keys or display names, case-insensitively.
    """
    name = str(
        task_name
    ).strip().lower()

    aliases = {
        "agnews":
            "agnews",
        "ag news":
            "agnews",
        "trec":
            "trec",
        "yahoo":
            "yahoo",
        "yahoo answers":
            "yahoo",
        "yahoo answers topics":
            "yahoo",
    }

    for task_key in TASK_ORDER:
        aliases[
            TASK_SPECS[
                task_key
            ][
                "display_name"
            ].lower()
        ] = task_key

    if name not in aliases:
        raise ValueError(
            f"Unknown task {task_name!r}. "
            f"Use one of {TASK_ORDER} "
            f"or their display names."
        )

    return aliases[
        name
    ]


def plot_dps_c_feature_bars(
    feature_idx: int,
    task_name: str = "agnews",
    *,
    demo_position: int = 1,
    split: str = "holdout",
    show_matched_markers: bool = True,
    width: int = 1250,
    height: int = 700,
):
    """
    Plot the activation-level DPS-C profile of one SAE feature.

    Orientation
    -----------
    x-axis:
        TEST QUERY label

    grouped bars:
        FIRST-DEMONSTRATION label

    red triangle:
        matched first-demo/query condition; for DPS-C this bar
        should be the LOWEST within each test-query group

    TASK_SCAN stores:
        mean_A[position, varied_demo_label, query_label, feature]
    """
    feature_idx = int(
        feature_idx
    )

    task_key = (
        _resolve_dps_task_key(
            task_name
        )
    )

    if not (
        0
        <= feature_idx
        < int(
            sae.n_features
        )
    ):
        raise ValueError(
            f"feature_idx must be in "
            f"[0, {sae.n_features - 1}], "
            f"got {feature_idx}."
        )

    if split not in (
        "discovery",
        "holdout",
    ):
        raise ValueError(
            "split must be 'discovery' or 'holdout'."
        )

    if (
        demo_position
        not in POSITION_TO_IDX
    ):
        raise ValueError(
            f"demo_position must be one of "
            f"{POSITIONS_TO_TEST}."
        )

    spec = TASK_SPECS[
        task_key
    ]

    p_idx = POSITION_TO_IDX[
        int(
            demo_position
        )
    ]

    mean_A = TASK_SCAN[
        task_key
    ][
        f"mean_A_{split}"
    ]

    # [first_demo_label, query_label]
    M = mean_A[
        p_idx,
        :,
        :,
        feature_idx,
    ].astype(
        float
    )

    label_ids = [
        int(l)
        for l
        in spec[
            "label_ids"
        ]
    ]

    labels = [
        spec[
            "label_to_word"
        ][l]
        for l
        in label_ids
    ]

    # --------------------------------------------------------
    # Long-form DataFrame
    # --------------------------------------------------------
    rows = []

    for first_demo_label_id in label_ids:
        for query_label_id in label_ids:
            rows.append({
                "feature_idx":
                    feature_idx,

                "task_key":
                    task_key,

                "task":
                    spec[
                        "display_name"
                    ],

                "demo_position":
                    int(
                        demo_position
                    ),

                "split":
                    split,

                "first_demo_label_id":
                    int(
                        first_demo_label_id
                    ),

                "first_demo_label":
                    spec[
                        "label_to_word"
                    ][
                        first_demo_label_id
                    ],

                "query_label_id":
                    int(
                        query_label_id
                    ),

                "query_label":
                    spec[
                        "label_to_word"
                    ][
                        query_label_id
                    ],

                "is_matched":
                    bool(
                        first_demo_label_id
                        == query_label_id
                    ),

                "mean_feature_value":
                    float(
                        M[
                            first_demo_label_id,
                            query_label_id,
                        ]
                    ),
            })

    bar_df = pd.DataFrame(
        rows
    )

    # --------------------------------------------------------
    # DPS-C equation summary
    # --------------------------------------------------------
    task_stats = (
        TASK_STATS[
            split
        ][
            task_key
        ]
    )

    cond = task_stats[
        "condition"
    ][
        p_idx,
        :,
        feature_idx,
    ]

    gaps = task_stats[
        "gap"
    ][
        p_idx,
        :,
        feature_idx,
    ]

    n_satisfied = int(
        cond.sum()
    )

    K = int(
        spec[
            "num_labels"
        ]
    )

    # --------------------------------------------------------
    # X-axis groups = TEST QUERY labels
    # --------------------------------------------------------
    group_centers = np.arange(
        K,
        dtype=float,
    )

    bar_width = min(
        0.18,
        0.78
        / max(
            1,
            K,
        ),
    )

    offsets = (
        np.arange(
            K
        )
        - (
            K - 1
        )
        / 2.0
    ) * bar_width

    palette = [
        "#4F46E5",
        "#10B981",
        "#F43F5E",
        "#F59E0B",
        "#0EA5E9",
        "#8B5CF6",
        "#14B8A6",
        "#EC4899",
        "#84CC16",
        "#F97316",
    ]

    fig = go.Figure()

    matched_marker_x = []
    matched_marker_y = []

    # One trace per FIRST-DEMONSTRATION label.
    for (
        first_j,
        first_demo_label_id,
    ) in enumerate(
        label_ids
    ):
        xs = (
            group_centers
            + offsets[
                first_j
            ]
        )

        ys = np.array(
            [
                M[
                    first_demo_label_id,
                    query_label_id,
                ]
                for query_label_id
                in label_ids
            ],
            dtype=float,
        )

        fig.add_trace(
            go.Bar(
                x=xs,
                y=ys,

                width=
                    bar_width
                    * 0.92,

                name=(
                    "First Demo Label = "
                    f"{spec['label_to_word'][first_demo_label_id]}"
                ),

                marker_color=
                    palette[
                        first_j
                        % len(
                            palette
                        )
                    ],

                hovertemplate=(
                    "<b>Feature %{customdata[0]}</b><br>"
                    "Test query label=%{customdata[1]}<br>"
                    "First demo label=%{customdata[2]}<br>"
                    "Mean value=%{y:.4f}"
                    "<extra></extra>"
                ),

                customdata=np.array(
                    [
                        [
                            feature_idx,
                            spec[
                                "label_to_word"
                            ][
                                query_label_id
                            ],
                            spec[
                                "label_to_word"
                            ][
                                first_demo_label_id
                            ],
                        ]
                        for query_label_id
                        in label_ids
                    ],
                    dtype=object,
                ),
            )
        )

        query_match_pos = (
            label_ids.index(
                first_demo_label_id
            )
        )

        matched_marker_x.append(
            xs[
                query_match_pos
            ]
        )

        matched_marker_y.append(
            ys[
                query_match_pos
            ]
        )

    # --------------------------------------------------------
    # Mark matched bars
    #
    # DPS-C:
    # Put the triangle slightly ABOVE the matched bar and point
    # it DOWN toward that bar.
    # --------------------------------------------------------
    if show_matched_markers:
        values = np.asarray(
            bar_df[
                "mean_feature_value"
            ],
            dtype=float,
        )

        span = float(
            np.nanmax(
                values
            )
            - np.nanmin(
                values
            )
        )

        marker_offset = max(
            0.05
            * span,
            0.05,
        )

        fig.add_trace(
            go.Scatter(
                x=
                    matched_marker_x,

                y=[
                    y
                    + marker_offset
                    for y
                    in matched_marker_y
                ],

                mode=
                    "markers",

                marker=dict(
                    symbol=
                        "triangle-down",

                    size=
                        16,

                    color=
                        "#D32F2F",
                ),

                name=
                    "Matched condition",

                hovertemplate=(
                    "Matched first-demo/query label"
                    "<extra></extra>"
                ),
            )
        )

    # --------------------------------------------------------
    # Separators between TEST QUERY groups
    # --------------------------------------------------------
    for i in range(
        K - 1
    ):
        fig.add_vline(
            x=
                i + 0.5,

            line_width=
                8,

            line_color=
                "rgba(93, 144, 196, 0.14)",
        )

    measure_name = (
        "Pre-Activation"
        if USE_SAE_PREACTIVATIONS
        else "Sparse Activation"
    )

    # --------------------------------------------------------
    # FIXED layout:
    #   - no huge per-label gap string in the title
    #   - more top margin for wrapped Yahoo legend
    #   - legend remains below the title / above plotting area
    # --------------------------------------------------------
    if K <= 4:
        top_margin = 180

        figure_height = max(
            int(
                height
            ),
            700,
        )

        legend_font_size = 14

    elif K <= 6:
        top_margin = 220

        figure_height = max(
            int(
                height
            ),
            760,
        )

        legend_font_size = 13

    else:
        top_margin = 300

        figure_height = max(
            int(
                height
            ),
            860,
        )

        legend_font_size = 12

    tick_angle = (
        -30
        if K >= 8
        else 0
    )

    fig.update_layout(
        title=dict(
            text=(
                f"DPS-C Feature Example "
                f"({MODEL_SHORT_NAME}, Feature #{feature_idx})"
                f"<br><sup>"
                f"{spec['display_name']} | "
                f"position={demo_position} | "
                f"{n_satisfied}/{K} "
                f"DPS-C conditions satisfied"
                f"</sup>"
            ),

            x=
                0.5,

            xanchor=
                "center",
        ),

        barmode=
            "overlay",

        xaxis=dict(
            title=
                "Test Query Label",

            tickmode=
                "array",

            tickvals=
                group_centers,

            ticktext=
                labels,

            tickangle=
                tick_angle,

            range=[
                -0.62,
                K - 0.38,
            ],
        ),

        yaxis=dict(
            title=
                f"Average {measure_name}",

            zeroline=
                True,

            zerolinewidth=
                1,
        ),

        legend=dict(
            orientation=
                "h",

            yanchor=
                "bottom",

            y=
                1.02,

            xanchor=
                "center",

            x=
                0.5,

            font=dict(
                size=
                    legend_font_size,
            ),
        ),

        font=dict(
            family=
                PLOT_FONT,

            size=
                15,
        ),

        width=
            int(
                width
            ),

        height=
            figure_height,

        margin=dict(
            l=
                90,

            r=
                40,

            t=
                top_margin,

            b=(
                150
                if K >= 8
                else 100
            ),
        ),
    )

    fig.show()

    print(
        f"{spec['display_name']} | "
        f"Feature {feature_idx} | "
        f"position {demo_position} | "
        f"{split}"
    )

    print(
        f"DPS-C conditions satisfied: "
        f"{n_satisfied}/{K}"
    )

    # Detailed gaps remain available here without cluttering the title.
    display(
        pd.DataFrame({
            "query_label":
                labels,

            "DPS_C_gap":
                gaps,

            "satisfies_DPS_C":
                cond,
        })
    )

    return (
        bar_df,
        fig,
    )


# Backward-compatible alias for any old downstream call.
plot_dps_a_feature_bars = plot_dps_c_feature_bars

### Representative DPS-C feature bar plot

After `selected_feature_df` is computed, the cell below plots the highest-ranked selected feature for any chosen task. The plot uses **Test Query Label** on the x-axis and first-demonstration labels as the grouped bars.

In [ ]:
representative_feature = selected_feature_df.iloc[6]["feature_idx"]

bar_df, fig = (
    plot_dps_c_feature_bars(
        feature_idx=representative_feature,
        task_name= "AGNews",
        demo_position= 1,
        split="holdout",
    )
)
bar_df, fig = (
    plot_dps_c_feature_bars(
        feature_idx=representative_feature,
        task_name= "TREC",
        demo_position= 1,
        split="holdout",
    )
)
bar_df, fig = (
    plot_dps_c_feature_bars(
        feature_idx=representative_feature,
        task_name= "Yahoo",
        demo_position= 1,
        split="holdout",
    )
)

## 16. Causal DPS-C amplification: does the feature family causally reinforce the matching condition?

The descriptive DPS-C analysis asks whether a feature is **more active** when
the varied demonstration label matches the test-query label. Here we test the
corresponding causal hypothesis.

Let \(F\) be a user-selected feature set. Increasing every selected SAE latent
by the same amount \(\alpha\) corresponds, in residual space, to adding

\[
v_F = \sum_{f\in F} W_{\mathrm{dec},f},
\qquad
h'_{20,t^\star}
=
h_{20,t^\star}
+
\alpha v_F.
\]

The intervention is identical across all varied-demo conditions.
`combine_mode="sum"` therefore means **+alpha to every selected SAE latent**;
`combine_mode="mean"` remains available as a scale-normalized diagnostic.

### Intervention location

With the first-label-anchored convention used throughout this notebook,
`LAST_K=[-2]` is the token immediately before the **first token of the final
label**. It is therefore exactly the residual position from which the model
predicts that first label token.

For the causal experiment, we take the exact rendered classification prompt
and slice it immediately before the final gold label. We intervene on the
final prefix token and score the resulting next-token candidate-label logits.
This remains well-defined when the full label string contains multiple tokens.

For each query with true label \(L\), and each varied first-demo label \(L'\),
define the correct-label decision margin

\[
M_L(L';\alpha)
=
z_L^{(\alpha)}(L')
-
\max_{k\neq L}z_k^{(\alpha)}(L'),
\]

and causal effect

\[
\delta M_L(L')
=
M_L(L';\alpha)-M_L(L';0).
\]

The causal section retains the same feature-selection, holdout evaluation,
candidate-label scoring, Top-20 heatmaps, and user-controlled color bounds as
the attached close-to-final notebook.

> **Output-score note.** For efficiency, the causal readout scores the first
> token of each candidate class label and asserts that those first tokens are
> unique within each task. The prefix-boundary verifier separately confirms
> that the intervention is immediately before the first token of the complete
> gold-label tokenization.

### Four intervention spans

`CAUSAL_STEERING_MODE` controls where the same fixed feature direction is
applied:

1. **`"last_token"`**: only the final token immediately before the first label
   token (the original setup).
2. **`"test_query"`**: the complete final test-query block. Under Gemma2
   format-1, the instruction, demonstrations, and query share one user turn,
   so the shared opening user-turn token is left untouched; the query span
   includes the complete structured query block and its trailing user-turn
   boundary token.
3. **`"first_demo_only"`**: only the tokens belonging to the first
   demonstration.
4. **`"first_demo_and_test_query"`**: the union of the first-demonstration
   tokens and the final-query span above.

Across all four modes, the SAE direction and the per-token steering magnitude
\(\alpha\) are identical; only the intervention span changes.

### DPS-C causal-specificity convention

For a fixed query label \(\ell\), the ideal causal DPS-C profile is

\[
\delta M_{f,\ell}(\ell;\alpha)
=
\min_{k\in\mathcal{Y}}
\delta M_{f,\ell}(k;\alpha).
\]

Accordingly, the notebook defines

\[
\mathrm{Spec}^{C}_{f,\ell}(\alpha)
=
\min_{k\neq\ell}
\delta M_{f,\ell}(k;\alpha)
-
\delta M_{f,\ell}(\ell;\alpha).
\]

Thus, **positive causal-specificity values still indicate success** and are
shown in blue in the feature-level specificity heatmap. By contrast, the raw
single-feature \(\delta M\) matrix is not sign-flipped: there, the matched
diagonal should be the smallest values and, under positive amplification, will
often appear as the most negative/red entries.

For the main causal DPS-C experiment, use a **positive \(\alpha\)** to amplify
the contrastive direction. A negative \(\alpha\) remains a valid suppression
control, but it can reverse the expected ordering and should not be interpreted
with the same matched-as-minimum criterion without accounting for that sign
reversal.

In [ ]:
# ============================================================
# User-editable Causal DPS-C configuration
# ============================================================

CAUSAL_FEATURES = [444, 1800, 2750]

# ============================================================
# DPS-C CAUSAL SIGN CONVENTION
# ============================================================
#
# MAIN TEST: use POSITIVE alpha to AMPLIFY the DPS-C feature direction.
#
# For an ideal causal DPS-C feature:
#
#     delta M_L(L)
#         = MIN over first-demo labels L' of delta M_L(L')
#
# i.e. the matched condition should have the SMALLEST causal effect
# on the correct-label margin (often the most negative value).
#
# Negative alpha is fully supported as a complementary SUPPRESSION test,
# but it can reverse this ordering. Therefore negative alpha should not be
# interpreted with the same matched-as-minimum criterion unless you
# intentionally analyze the sign-reversed intervention.
#
# Any real alpha is technically valid:
#   +10 -> add 10 * decoder direction      [recommended DPS-C test]
#   -10 -> subtract 10 * decoder direction [suppression control]
CAUSAL_ALPHA = 50.0

# Primacy test: vary the demonstration at position 1.
CAUSAL_DEMO_POSITION = 1

# Recommended: select features on discovery, evaluate causality on holdout.
CAUSAL_SPLIT = "holdout"

# None -> use every query available in the selected split.
# Set e.g. 5 or 10 for a quick diagnostic run.
CAUSAL_N_QUERIES_PER_LABEL = None

# Number of prompt conditions processed together.
CAUSAL_BATCH_SIZE = 16

# "sum"  = add +alpha to EVERY selected latent:
#          residual shift = alpha * sum_f W_dec[f]
# "mean" = residual shift = alpha * mean_f W_dec[f]
CAUSAL_COMBINE_MODE = "sum"

# ============================================================
# USER-SELECTABLE CAUSAL STEERING SPAN
# ============================================================
#
# "last_token"
#     Original/current behavior: steer only the final token immediately
#     before the first answer-label token.
#
# "test_query"
#     Steer the complete final test-query block.
#     With Gemma2 format-1, instruction + demos + query share one user turn,
#     so the opening <start_of_turn>user token belongs to the whole prompt.
#     We therefore steer the complete structured query block plus its trailing
#     user-turn boundary special token, while leaving preceding context intact.
#
# "first_demo_only"
#     Steer only the tokens belonging to the first demonstration.
#
# "first_demo_and_test_query"
#     Steer all tokens in the first demonstration plus the test-query span.
#

# aaaa
CAUSAL_STEERING_MODE = "test_query"

CAUSAL_STEERING_MODES = (
    "last_token",
    "test_query",
    "first_demo_only",
    "first_demo_and_test_query",
)

CAUSAL_TASK_KEYS = ["agnews", "trec", "yahoo"]
CAUSAL_TIE_TOL = 1e-9

CAUSAL_OUTPUT_DIR = OUTPUT_ROOT / "causal_dps_c"
CAUSAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CAUSAL_FEATURES:", CAUSAL_FEATURES)
print("CAUSAL_ALPHA:", CAUSAL_ALPHA)
print("CAUSAL_DEMO_POSITION:", CAUSAL_DEMO_POSITION)
print("CAUSAL_SPLIT:", CAUSAL_SPLIT)
print("CAUSAL_N_QUERIES_PER_LABEL:", CAUSAL_N_QUERIES_PER_LABEL)
print("CAUSAL_COMBINE_MODE:", CAUSAL_COMBINE_MODE)
print("CAUSAL_STEERING_MODE:", CAUSAL_STEERING_MODE)

# Publication-friendly causal heatmap:
# negative = red, zero = white, positive = blue.
CAUSAL_HEATMAP_COLORSCALE = [
    [0.00, "#B2182B"],
    [0.50, "#F7F7F7"],
    [1.00, "#2166AC"],
]


In [ ]:
# ============================================================
# Causal DPS-C helpers
# ============================================================

def _format_final_query_no_sentinel(task_key: str, example, idx: int) -> str:
    """Same final user request as the main experiment, but without query-span sentinels."""
    spec = TASK_SPECS[task_key]
    text = get_example_text(task_key, example, MAX_QUERY_CHARS)
    return (
        f"Example {idx}\n"
        f"{spec['input_field_name']}:\n{text}\n"
        f"{spec['output_field_name']}:"
    )


def _manual_gemma_generation_prefix(
    messages: List[Dict[str, str]],
) -> str:
    return manual_gemma_chat_template(
        messages,
        add_generation_prompt=True,
    )


def _apply_generation_template(
    messages: List[Dict[str, str]],
) -> str:
    """
    Render chat history immediately before the model's answer.
    """
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return _manual_gemma_generation_prefix(
            messages
        )




def build_causal_generation_prefix(
    task_key: str,
    demos,
    query_example,
) -> str:
    """
    Exact classification prompt immediately before the final gold label.
    """
    gold_label = label_word(
        task_key,
        query_example,
    )

    full_prompt, _ = build_prompt(
        task_key,
        demos,
        query_example,
    )

    label_start = full_prompt.rfind(
        gold_label
    )

    if label_start < 0:
        raise RuntimeError(
            f"[{task_key}] could not locate final gold label "
            f"{gold_label!r}."
        )

    prefix = full_prompt[
        :label_start
    ]

    if not full_prompt.startswith(
        prefix + gold_label
    ):
        raise AssertionError(
            f"[{task_key}] exact causal-prefix construction failed."
        )

    return prefix


def _token_positions_overlapping_char_span(
    text: str,
    char_start: int,
    char_end: int,
) -> List[int]:
    enc = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    out = []

    for i, (s, e) in enumerate(
        enc["offset_mapping"]
    ):
        s = int(s)
        e = int(e)

        if e <= s:
            continue

        if (
            e > int(char_start)
            and s < int(char_end)
        ):
            out.append(
                int(i)
            )

    if not out:
        raise RuntimeError(
            "Could not map requested character span to tokens."
        )

    return out


def _gemma_special_token_id(
    literal: str,
) -> Optional[int]:
    try:
        tid = tokenizer.convert_tokens_to_ids(
            literal
        )

        unk = getattr(
            tokenizer,
            "unk_token_id",
            None,
        )

        if (
            tid is not None
            and int(tid) >= 0
            and (
                unk is None
                or int(tid) != int(unk)
            )
        ):
            return int(tid)

    except Exception:
        pass

    try:
        ids = tokenizer(
            literal,
            add_special_tokens=False,
        )["input_ids"]

        if len(ids) == 1:
            return int(
                ids[0]
            )

    except Exception:
        pass

    return None


def _complete_gemma_turn_from_content(
    prefix: str,
    content_positions: Sequence[int],
) -> List[int]:
    """
    Expand a content span to the nearest complete Gemma turn.
    Used when the content occupies its own user/model turn.
    """
    content_positions = sorted(
        set(
            map(
                int,
                content_positions,
            )
        )
    )

    prefix_ids = list(
        map(
            int,
            tokenizer(
                prefix,
                add_special_tokens=False,
            )["input_ids"],
        )
    )

    start_id = _gemma_special_token_id(
        "<start_of_turn>"
    )
    end_id = _gemma_special_token_id(
        "<end_of_turn>"
    )

    if (
        start_id is None
        or end_id is None
    ):
        return content_positions

    first_pos = min(
        content_positions
    )
    last_pos = max(
        content_positions
    )

    starts = [
        i
        for i, tok
        in enumerate(prefix_ids)
        if tok == start_id
        and i <= first_pos
    ]

    ends = [
        i
        for i, tok
        in enumerate(prefix_ids)
        if tok == end_id
        and i >= last_pos
    ]

    if not starts or not ends:
        return content_positions

    return list(
        range(
            max(starts),
            min(ends) + 1,
        )
    )


def _format1_query_span_with_trailing_boundary(
    prefix: str,
    query_positions: Sequence[int],
) -> List[int]:
    """
    Gemma2 format-1 puts instruction + demonstrations + test query in one
    user turn. The shared opening user-turn special token is therefore not
    query-specific.

    Start at the first token of the final structured query block and extend
    through the closing <end_of_turn> special token of the shared user turn.
    """
    query_positions = sorted(
        set(
            map(
                int,
                query_positions,
            )
        )
    )

    prefix_ids = list(
        map(
            int,
            tokenizer(
                prefix,
                add_special_tokens=False,
            )["input_ids"],
        )
    )

    end_id = _gemma_special_token_id(
        "<end_of_turn>"
    )

    if end_id is None:
        return query_positions

    last_q = max(
        query_positions
    )

    ends = [
        i
        for i, tok
        in enumerate(prefix_ids)
        if tok == end_id
        and i >= last_q
    ]

    if not ends:
        return query_positions

    return list(
        range(
            min(query_positions),
            min(ends) + 1,
        )
    )



def build_causal_prefix_and_steering_spans(
    task_key: str,
    demos,
    query_example,
):
    """
    Return the exact causal prefix plus token-index spans for all
    supported steering modes.

    IMPORTANT FIX
    -------------
    The final-query span is derived directly from the exact
    `query_text_span` returned by `build_prompt(...)`.

    We do NOT reconstruct the query with `format_user_request(...)`.
    This is important for long examples (especially Yahoo), because
    the real final query uses MAX_QUERY_CHARS while demonstration-style
    formatting uses MAX_DEMO_CHARS.

    Returned spans
    --------------
    last_token:
        Final causal-prefix token immediately before the first label token.

    test_query:
        Complete final structured query block:
            Example N
            <input field>:
            <exact rendered query text>
            <output field>:
        plus the trailing Gemma user-turn boundary token where applicable.

    first_demo:
        All tokens belonging to the first demonstration.

    first_demo_and_test_query:
        Union of first_demo and test_query.
    """
    gold_label = label_word(
        task_key,
        query_example,
    )

    # ========================================================
    # SINGLE SOURCE OF TRUTH:
    # use the exact same prompt as the activation experiment.
    # ========================================================
    (
        full_prompt,
        query_text_span,
    ) = build_prompt(
        task_key,
        demos,
        query_example,
    )

    # --------------------------------------------------------
    # Exact causal prefix: stop immediately before the final
    # assistant gold label.
    # --------------------------------------------------------
    label_start = full_prompt.rfind(
        gold_label
    )

    if label_start < 0:
        raise RuntimeError(
            f"[{task_key}] could not locate final gold label "
            f"{gold_label!r} in the exact rendered prompt."
        )

    prefix = full_prompt[
        :label_start
    ]

    if not full_prompt.startswith(
        prefix
        + gold_label
    ):
        raise AssertionError(
            f"[{task_key}] exact causal-prefix construction failed."
        )

    prefix_ids = list(
        map(
            int,
            tokenizer(
                prefix,
                add_special_tokens=False,
            )[
                "input_ids"
            ],
        )
    )

    if not prefix_ids:
        raise RuntimeError(
            "Causal prefix tokenized to zero tokens."
        )

    # ========================================================
    # FINAL TEST QUERY
    #
    # query_text_span is the exact raw-query-text span in the
    # FINAL rendered prompt AFTER sentinel removal.
    #
    # Recover only the small structural wrapper around that
    # already-rendered text. No query text is regenerated.
    # ========================================================
    q_text_start = int(
        query_text_span[
            0
        ]
    )

    q_text_end = int(
        query_text_span[
            1
        ]
    )

    if not (
        0
        <= q_text_start
        <= q_text_end
        <= len(
            prefix
        )
    ):
        raise RuntimeError(
            f"[{task_key}] invalid exact query span "
            f"{query_text_span} for causal prefix length "
            f"{len(prefix)}."
        )

    spec = TASK_SPECS[
        task_key
    ]

    query_header = (
        f"Example {len(demos) + 1}\n"
        f"{spec['input_field_name']}:\n"
    )

    # The query header must occur immediately before the exact
    # query-text span. Search backward only before q_text_start.
    q_block_start = prefix.rfind(
        query_header,
        0,
        q_text_start
        + 1,
    )

    if q_block_start < 0:
        raise RuntimeError(
            f"[{task_key}] could not locate final query header "
            f"{query_header!r} before exact query span "
            f"{query_text_span}."
        )

    output_marker = (
        f"\n"
        f"{spec['output_field_name']}:"
    )

    output_marker_start = prefix.find(
        output_marker,
        q_text_end,
    )

    if output_marker_start < 0:
        raise RuntimeError(
            f"[{task_key}] could not locate final query output marker "
            f"{output_marker!r} after exact query span "
            f"{query_text_span}."
        )

    q_block_end = (
        output_marker_start
        + len(
            output_marker
        )
    )

    # Strong string-level check: the structured query block must
    # contain the exact query-text span returned by build_prompt().
    if not (
        q_block_start
        <= q_text_start
        <= q_text_end
        <= q_block_end
    ):
        raise AssertionError(
            f"[{task_key}] malformed final-query block bounds: "
            f"block=({q_block_start}, {q_block_end}), "
            f"text={query_text_span}."
        )

    query_content_positions = (
        _token_positions_overlapping_char_span(
            prefix,
            q_block_start,
            q_block_end,
        )
    )

    if (
        PROMPT_FORMAT_NAME
        == "format1_system_demos_user_query_assistant_label"
    ):
        # Gemma2 format-1 puts instruction + demos + query inside
        # a single user turn. Do not steer the shared opening
        # user-turn token; include the full final query block and
        # its trailing <end_of_turn>.
        test_query_positions = (
            _format1_query_span_with_trailing_boundary(
                prefix,
                query_content_positions,
            )
        )

    elif (
        PROMPT_FORMAT_NAME
        == "format2_wrapped_user_assistant_demos"
    ):
        # Here the final query occupies its own user turn.
        test_query_positions = (
            _complete_gemma_turn_from_content(
                prefix,
                query_content_positions,
            )
        )

    else:
        raise ValueError(
            f"Unknown PROMPT_FORMAT_NAME="
            f"{PROMPT_FORMAT_NAME!r}"
        )

    # ========================================================
    # FIRST DEMONSTRATION
    # ========================================================
    first_demo_positions = []

    if len(
        demos
    ) > 0:
        if (
            PROMPT_FORMAT_NAME
            == "format1_system_demos_user_query_assistant_label"
        ):
            # In format-1 the demonstration block is embedded
            # verbatim inside the shared user turn.
            demo_block = format_demo_text_block(
                task_key,
                demos[
                    0
                ],
                1,
            )

            d_start = prefix.find(
                demo_block
            )

            if d_start < 0:
                raise RuntimeError(
                    f"[{task_key}] first demonstration block "
                    "not found inside exact causal prefix."
                )

            d_end = (
                d_start
                + len(
                    demo_block
                )
            )

            first_demo_positions = (
                _token_positions_overlapping_char_span(
                    prefix,
                    d_start,
                    d_end,
                )
            )

        elif (
            PROMPT_FORMAT_NAME
            == "format2_wrapped_user_assistant_demos"
        ):
            # Demo user request is safe to reconstruct with
            # mark_text=False because it is genuinely a DEMO and
            # therefore uses the same MAX_DEMO_CHARS convention
            # in the original prompt.
            first_user_block = format_user_request(
                task_key,
                demos[
                    0
                ],
                1,
                mark_text=False,
            )

            d_start = prefix.find(
                first_user_block
            )

            if d_start < 0:
                raise RuntimeError(
                    f"[{task_key}] first-demo user block not found."
                )

            first_label = label_word(
                task_key,
                demos[
                    0
                ],
            )

            first_label_start = prefix.find(
                first_label,
                (
                    d_start
                    + len(
                        first_user_block
                    )
                ),
            )

            if first_label_start < 0:
                raise RuntimeError(
                    f"[{task_key}] first-demo model label not found."
                )

            d_end = (
                first_label_start
                + len(
                    first_label
                )
            )

            content_positions = (
                _token_positions_overlapping_char_span(
                    prefix,
                    d_start,
                    d_end,
                )
            )

            # Expand across the complete first user + model
            # demonstration turns when Gemma special tokens are
            # available.
            prefix_token_ids = list(
                map(
                    int,
                    tokenizer(
                        prefix,
                        add_special_tokens=False,
                    )[
                        "input_ids"
                    ],
                )
            )

            start_id = _gemma_special_token_id(
                "<start_of_turn>"
            )

            end_id = _gemma_special_token_id(
                "<end_of_turn>"
            )

            if (
                start_id is None
                or end_id is None
            ):
                first_demo_positions = (
                    content_positions
                )

            else:
                first_pos = min(
                    content_positions
                )

                last_pos = max(
                    content_positions
                )

                starts = [
                    i
                    for i, tok
                    in enumerate(
                        prefix_token_ids
                    )
                    if (
                        tok == start_id
                        and i <= first_pos
                    )
                ]

                ends = [
                    i
                    for i, tok
                    in enumerate(
                        prefix_token_ids
                    )
                    if (
                        tok == end_id
                        and i >= last_pos
                    )
                ]

                if (
                    starts
                    and ends
                ):
                    first_demo_positions = list(
                        range(
                            max(
                                starts
                            ),
                            min(
                                ends
                            )
                            + 1,
                        )
                    )

                else:
                    first_demo_positions = (
                        content_positions
                    )

        else:
            raise ValueError(
                f"Unknown PROMPT_FORMAT_NAME="
                f"{PROMPT_FORMAT_NAME!r}"
            )

    # ========================================================
    # FINAL SPAN DICTIONARY
    # ========================================================
    last_token_positions = [
        len(
            prefix_ids
        )
        - 1
    ]

    spans = {
        "last_token":
            last_token_positions,

        "test_query":
            sorted(
                set(
                    test_query_positions
                )
            ),

        "first_demo":
            sorted(
                set(
                    first_demo_positions
                )
            ),

        "first_demo_and_test_query":
            sorted(
                set(
                    first_demo_positions
                )
                | set(
                    test_query_positions
                )
            ),
    }

    # --------------------------------------------------------
    # Bounds / emptiness checks
    # --------------------------------------------------------
    for (
        span_name,
        positions,
    ) in spans.items():
        if (
            span_name
            == "first_demo"
            and len(
                demos
            )
            == 0
        ):
            continue

        if not positions:
            raise RuntimeError(
                f"[{task_key}] causal steering span "
                f"{span_name!r} is empty."
            )

        if (
            min(
                positions
            )
            < 0
            or max(
                positions
            )
            >= len(
                prefix_ids
            )
        ):
            raise RuntimeError(
                f"[{task_key}] steering span "
                f"{span_name!r} is out of bounds for "
                f"prefix length {len(prefix_ids)}."
            )

    return (
        prefix,
        spans,
    )


def _steering_positions_for_mode(
    spans: Dict[str, Sequence[int]],
    steering_mode: str,
) -> List[int]:
    if (
        steering_mode
        not in CAUSAL_STEERING_MODES
    ):
        raise ValueError(
            f"steering_mode must be one of "
            f"{CAUSAL_STEERING_MODES}, "
            f"got {steering_mode!r}."
        )

    # The span dictionary uses the concise internal key "first_demo".
    # Expose the clearer public mode name "first_demo_only".
    span_key = (
        "first_demo"
        if steering_mode == "first_demo_only"
        else steering_mode
    )

    return list(
        map(
            int,
            spans[
                span_key
            ],
        )
    )



def inspect_causal_steering_spans(
    task_key: str,
    demos,
    query_example,
    *,
    max_tokens_per_span: int = 40,
):
    """
    Debug/inspection helper with NO model forward pass.

    Prints the token ranges selected by each causal steering mode.
    Useful for confirming long Yahoo prompts before launching Top-20.
    """
    (
        prefix,
        spans,
    ) = build_causal_prefix_and_steering_spans(
        task_key,
        demos,
        query_example,
    )

    ids = list(
        map(
            int,
            tokenizer(
                prefix,
                add_special_tokens=False,
            )[
                "input_ids"
            ],
        )
    )

    print(
        f"Task: {TASK_SPECS[task_key]['display_name']}"
    )

    print(
        "Prefix tokens:",
        len(
            ids
        ),
    )

    for mode in [
        "last_token",
        "test_query",
        "first_demo",
        "first_demo_and_test_query",
    ]:
        positions = spans[
            mode
        ]

        shown = positions[
            :int(
                max_tokens_per_span
            )
        ]

        print(
            f"\n[{mode}] "
            f"{len(positions)} token(s) | "
            f"range={positions[0]}..{positions[-1]}"
        )

        print(
            [
                repr(
                    tokenizer.decode(
                        [
                            ids[
                                p
                            ]
                        ],
                        skip_special_tokens=False,
                        clean_up_tokenization_spaces=False,
                    )
                )
                for p
                in shown
            ]
        )

        if len(
            positions
        ) > len(
            shown
        ):
            print(
                f"... +{len(positions) - len(shown)} more token(s)"
            )

    return (
        prefix,
        spans,
    )


def causal_candidate_label_tokens(task_key: str) -> pd.DataFrame:
    """
    Return the first generated token used to score every candidate label.
    A uniqueness assertion prevents ambiguous candidate-label scoring.
    """
    spec = TASK_SPECS[task_key]
    rows = []

    for label_id in spec["label_ids"]:
        label = spec["label_to_word"][int(label_id)]
        ids = tokenizer(
            label,
            add_special_tokens=False,
        )["input_ids"]

        if len(ids) == 0:
            raise RuntimeError(
                f"Label {label!r} produced zero tokens."
            )

        rows.append({
            "label_id": int(label_id),
            "label": label,
            "first_token_id": int(ids[0]),
            "first_token": repr(
                tokenizer.decode(
                    [int(ids[0])],
                    skip_special_tokens=False,
                    clean_up_tokenization_spaces=False,
                )
            ),
            "full_token_ids": tuple(map(int, ids)),
            "n_label_tokens": len(ids),
        })

    df = pd.DataFrame(rows)

    if df["first_token_id"].nunique() != len(df):
        raise RuntimeError(
            f"{task_key}: two candidate labels share the same first token. "
            "Use full-label sequence scoring instead.\n"
            + df.to_string(index=False)
        )

    return df


def causal_feature_direction(
    feature_indices: Sequence[int],
    combine_mode: str = "sum",
) -> torch.Tensor:
    """
    Residual-space direction corresponding to increasing selected SAE latents.

    For combine_mode='sum', +alpha * direction is exactly +alpha to every
    listed latent under the SAE decoder.
    """
    features = [int(f) for f in feature_indices]

    if len(features) == 0:
        raise ValueError("feature_indices cannot be empty.")

    bad = [
        f for f in features
        if not (0 <= f < int(sae.n_features))
    ]
    if bad:
        raise ValueError(
            f"Invalid feature indices: {bad}. "
            f"Valid range is 0..{sae.n_features - 1}."
        )

    dirs = sae.W_dec[features].detach()

    if combine_mode == "sum":
        direction = dirs.sum(dim=0)
    elif combine_mode == "mean":
        direction = dirs.mean(dim=0)
    else:
        raise ValueError(
            "combine_mode must be 'sum' or 'mean'."
        )

    return direction


def verify_causal_prefix_alignment(
    task_key: str,
    demos,
    query_example,
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Verify that the causal intervention point is strictly before the FIRST
    token of the output label, including for multi-token labels.
    """
    gold_label = label_word(task_key, query_example)

    prefix = build_causal_generation_prefix(task_key, demos, query_example)
    full_prompt, _ = build_prompt(task_key, demos, query_example)

    if not full_prompt.startswith(prefix):
        common = 0
        for a, b in zip(prefix, full_prompt):
            if a != b:
                break
            common += 1
        raise AssertionError(
            f"[{task_key}] causal generation prefix is not an exact prefix "
            f"of the full prompt; first mismatch at character {common}.\n"
            f"prefix tail={prefix[max(0, common-100):common+100]!r}\n"
            f"full tail={full_prompt[max(0, common-100):common+100]!r}"
        )

    remainder = full_prompt[len(prefix):]
    if not remainder.startswith(gold_label):
        raise AssertionError(
            f"[{task_key}] gold label does not start immediately after "
            f"the causal prefix.\n"
            f"gold_label={gold_label!r}\n"
            f"remainder head={remainder[:160]!r}"
        )

    prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_prompt, add_special_tokens=False)["input_ids"]
    label_ids = tokenizer(gold_label, add_special_tokens=False)["input_ids"]

    if full_ids[:len(prefix_ids)] != prefix_ids:
        raise AssertionError(
            f"[{task_key}] tokenized causal prefix is not an exact token "
            "prefix of the full prompt."
        )

    observed_label_ids = full_ids[
        len(prefix_ids):len(prefix_ids) + len(label_ids)
    ]
    if observed_label_ids != label_ids:
        raise AssertionError(
            f"[{task_key}] tokens immediately after the causal prefix do not "
            f"equal the complete gold-label tokenization.\n"
            f"label={gold_label!r}\n"
            f"expected={label_ids}\n"
            f"observed={observed_label_ids}\n"
            f"observed text={tokenizer.decode(observed_label_ids)!r}"
        )

    final_prefix_id = int(prefix_ids[-1])
    next_id = int(full_ids[len(prefix_ids)])

    result = {
        "task_key": task_key,
        "task": TASK_SPECS[task_key]["display_name"],
        "gold_label": gold_label,
        "n_label_tokens": int(len(label_ids)),
        "label_token_ids": tuple(map(int, label_ids)),
        "decoded_label_tokens": tuple(
            repr(
                tokenizer.decode(
                    [int(t)],
                    skip_special_tokens=False,
                    clean_up_tokenization_spaces=False,
                )
            )
            for t in label_ids
        ),
        "final_prefix_token_id": final_prefix_id,
        "final_prefix_token": repr(
            tokenizer.decode(
                [final_prefix_id],
                skip_special_tokens=False,
                clean_up_tokenization_spaces=False,
            )
        ),
        "next_token_id": next_id,
        "next_token": repr(
            tokenizer.decode(
                [next_id],
                skip_special_tokens=False,
                clean_up_tokenization_spaces=False,
            )
        ),
        "verified_prefix_immediately_before_first_label_token": True,
    }

    if verbose:
        print(
            f"[verified] {result['task']} | {gold_label!r} | "
            f"{len(label_ids)} label token(s) | "
            f"prefix final={result['final_prefix_token']} | "
            f"next={result['next_token']}"
        )

    return result


def verify_all_causal_label_boundaries(
    task_key: str,
    causal_plan: pd.DataFrame,
    *,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Verify one real prompt for EVERY class label in this task.
    """
    spec = TASK_SPECS[task_key]
    rows = []

    for label_id in spec["label_ids"]:
        candidates = causal_plan[
            causal_plan["query_label_id"] == int(label_id)
        ]
        if len(candidates) == 0:
            raise RuntimeError(
                f"{task_key}: no causal query available for label {label_id}."
            )

        matched = candidates[
            candidates["varied_label_id"] == int(label_id)
        ]
        row = matched.iloc[0] if len(matched) else candidates.iloc[0]

        demos = [
            TASK_DATA[task_key]["demo"][int(i)]
            for i in tuple(map(int, row["demo_indices"]))
        ]
        query_example = TASK_DATA[task_key]["query"][int(row["query_index"])]

        result = verify_causal_prefix_alignment(
            task_key,
            demos,
            query_example,
            verbose=False,
        )
        result["query_label_id"] = int(label_id)
        rows.append(result)

    df = pd.DataFrame(rows)
    if not df[
        "verified_prefix_immediately_before_first_label_token"
    ].all():
        raise AssertionError(
            f"{task_key}: at least one label failed causal-prefix validation."
        )

    if verbose:
        print(
            f"\nCausal boundary verification — {spec['display_name']}: "
            f"{len(df)}/{len(df)} labels passed"
        )
        display(df)

    return df


@torch.no_grad()
def _causal_last_position_logits(
    prefixes: Sequence[str],
    steering_positions: Optional[
        Sequence[Sequence[int]]
    ] = None,
    direction: Optional[torch.Tensor] = None,
    alpha: float = 0.0,
) -> np.ndarray:
    """
    Return next-token vocabulary logits from the final causal-prefix token.

    When steering is active, add alpha * direction independently at every
    selected token position. The direction and per-token alpha are unchanged
    across steering modes; only the intervention span differs.
    """
    enc = tokenizer(
        list(prefixes),
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
        truncation=False,
    )

    attention_mask_cpu = (
        enc["attention_mask"]
        .cpu()
    )

    final_pos_cpu = (
        attention_mask_cpu
        .sum(dim=1)
        .long()
        - 1
    )

    input_ids = (
        enc["input_ids"]
        .to(
            MODEL_INPUT_DEVICE
        )
    )

    attention_mask = (
        enc["attention_mask"]
        .to(
            MODEL_INPUT_DEVICE
        )
    )

    block = model.get_submodule(
        f"model.layers.{TARGET_LAYER}"
    )

    handle = None

    if (
        direction is not None
        and float(alpha) != 0.0
    ):
        if steering_positions is None:
            raise ValueError(
                "steering_positions must be supplied "
                "for a nonzero causal intervention."
            )

        if len(steering_positions) != len(prefixes):
            raise ValueError(
                "steering_positions must contain one "
                "position list per prefix."
            )

        base_direction = (
            direction.detach()
        )

        def hook_fn(
            module,
            inputs,
            output,
        ):
            x = (
                output[0]
                if isinstance(
                    output,
                    tuple,
                )
                else output
            )

            x_mod = (
                x.clone()
            )

            shift = (
                float(alpha)
                * base_direction.to(
                    device=
                        x_mod.device,
                    dtype=
                        x_mod.dtype,
                )
            )

            for (
                batch_i,
                raw_positions,
            ) in enumerate(
                steering_positions
            ):
                positions_list = [
                    int(p)
                    for p
                    in raw_positions
                ]

                if not positions_list:
                    continue

                valid_len = int(
                    attention_mask_cpu[
                        batch_i
                    ]
                    .sum()
                    .item()
                )

                if (
                    min(positions_list) < 0
                    or max(positions_list) >= valid_len
                ):
                    raise IndexError(
                        f"Invalid steering positions for batch "
                        f"{batch_i}: valid_len={valid_len}, "
                        f"range=[{min(positions_list)}, "
                        f"{max(positions_list)}]."
                    )

                positions = torch.as_tensor(
                    positions_list,
                    dtype=torch.long,
                    device=x_mod.device,
                )

                x_mod[
                    batch_i,
                    positions,
                    :
                ] = (
                    x_mod[
                        batch_i,
                        positions,
                        :
                    ]
                    + shift.unsqueeze(0)
                )

            if isinstance(
                output,
                tuple,
            ):
                return (
                    x_mod,
                ) + tuple(
                    output[1:]
                )

            return x_mod

        handle = (
            block.register_forward_hook(
                hook_fn
            )
        )

    try:
        backbone = (
            model.model
            if hasattr(
                model,
                "model",
            )
            else model
        )

        backbone_out = backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )

    finally:
        if handle is not None:
            handle.remove()

    hidden = (
        backbone_out.last_hidden_state
    )

    batch_idx = torch.arange(
        hidden.shape[0],
        device=hidden.device,
    )

    final_pos = (
        final_pos_cpu.to(
            hidden.device
        )
    )

    final_hidden = hidden[
        batch_idx,
        final_pos,
        :
    ]

    lm_head = (
        model.get_output_embeddings()
    )

    logits = lm_head(
        final_hidden.to(
            device=
                lm_head.weight.device,
            dtype=
                lm_head.weight.dtype,
        )
    )

    return (
        logits.detach()
        .float()
        .cpu()
        .numpy()
        .astype(
            np.float32
        )
    )


def _correct_label_margin(
    candidate_logits: np.ndarray,
    targets: np.ndarray,
) -> np.ndarray:
    """Correct-label logit minus strongest wrong-label logit."""
    candidate_logits = np.asarray(
        candidate_logits,
        dtype=np.float64,
    )
    targets = np.asarray(targets, dtype=int)

    correct = candidate_logits[
        np.arange(len(targets)),
        targets,
    ]

    wrong = candidate_logits.copy()
    wrong[
        np.arange(len(targets)),
        targets,
    ] = -np.inf

    return correct - wrong.max(axis=1)


def _candidate_correct_probability(
    candidate_logits: np.ndarray,
    targets: np.ndarray,
) -> np.ndarray:
    """Softmax probability conditional on the task's candidate label tokens."""
    x = np.asarray(candidate_logits, dtype=np.float64)
    x = x - x.max(axis=1, keepdims=True)
    p = np.exp(x)
    p = p / p.sum(axis=1, keepdims=True)

    targets = np.asarray(targets, dtype=int)
    return p[np.arange(len(targets)), targets]


def build_causal_dps_plan(
    task_key: str,
    split: str = "holdout",
    demo_position: int = 1,
    n_queries_per_label: Optional[int] = None,
) -> pd.DataFrame:
    """
    Reuse the exact existing POSITION_PLANS prompt conditions.
    Every chosen query retains every varied-demo label condition.
    """
    if task_key not in POSITION_PLANS:
        raise ValueError(
            f"Unknown task_key={task_key!r}."
        )
    if demo_position not in POSITIONS_TO_TEST:
        raise ValueError(
            f"demo_position must be one of {POSITIONS_TO_TEST}."
        )

    plan = POSITION_PLANS[task_key].copy()

    plan = plan[
        (plan["split"] == str(split))
        & (plan["varied_position"] == int(demo_position))
    ].copy()

    spec = TASK_SPECS[task_key]
    chosen_qids = []

    for label_id in spec["label_ids"]:
        qrows = (
            plan[
                plan["query_label_id"] == int(label_id)
            ][
                [
                    "query_global_id",
                    "query_index_within_label",
                ]
            ]
            .drop_duplicates()
            .sort_values(
                [
                    "query_index_within_label",
                    "query_global_id",
                ]
            )
        )

        if n_queries_per_label is not None:
            qrows = qrows.head(
                int(n_queries_per_label)
            )

        chosen_qids.extend(
            qrows["query_global_id"]
            .astype(int)
            .tolist()
        )

    out = plan[
        plan["query_global_id"].isin(chosen_qids)
    ].copy()

    out = out.sort_values(
        [
            "query_label_id",
            "query_global_id",
            "varied_label_id",
        ]
    ).reset_index(drop=True)

    K = int(spec["num_labels"])

    assert (
        out.groupby("query_global_id")
        .size()
        .eq(K)
        .all()
    ), "Each selected query must retain every varied-demo label condition."

    return out

In [25]:
def run_causal_dps_amplification(
    feature_indices: Sequence[int],
    alpha: float = 10.0,
    demo_position: int = 1,
    split: str = "holdout",
    n_queries_per_label: Optional[int] = None,
    task_keys: Sequence[str] = ("agnews", "trec", "yahoo"),
    batch_size: int = 4,
    combine_mode: str = "sum",
    causal_tie_tol: float = 1e-9,
    save_csv: bool = True,
    show_plots: bool = True,
    show_tables: bool = True,
    verify_prefix_alignment: bool = True,
    steering_mode: str = CAUSAL_STEERING_MODE,
):
    """
    Run the Causal DPS-C amplification test.

    `alpha` may be any real value. Positive alpha adds the decoder direction;
    negative alpha subtracts it.

    Returns
    -------
    prompt_result_df:
        One row for every (query, varied-demo-label) condition.

    causal_equation_df:
        One row for each task/true-label Causal DPS-C equation.
    """
    features = [int(f) for f in feature_indices]

    if (
        steering_mode
        not in CAUSAL_STEERING_MODES
    ):
        raise ValueError(
            f"steering_mode must be one of "
            f"{CAUSAL_STEERING_MODES}, "
            f"got {steering_mode!r}."
        )

    direction = causal_feature_direction(
        features,
        combine_mode=combine_mode,
    )

    print("=" * 96)
    print("CAUSAL DPS-C AMPLIFICATION")
    print("features:", features)
    print("alpha:", float(alpha))
    print("combine_mode:", combine_mode)
    print("direction L2 norm:", float(direction.float().norm().cpu()))
    print("demo_position:", demo_position)
    print("split:", split)
    print("steering_mode:", steering_mode)
    print("n_queries_per_label:", n_queries_per_label)
    print("=" * 96)

    all_rows = []

    for task_key in task_keys:
        if task_key not in TASK_ORDER:
            raise ValueError(
                f"Unknown task_key={task_key!r}. "
                f"Expected one of {TASK_ORDER}."
            )

        spec = TASK_SPECS[task_key]
        label_token_df = causal_candidate_label_tokens(
            task_key
        )

        print("\n" + "=" * 80)
        print("TASK:", spec["display_name"])
        print("Candidate-label first-token scoring:")
        if show_tables:
            display(label_token_df)

        candidate_token_ids = (
            label_token_df
            .sort_values("label_id")["first_token_id"]
            .astype(int)
            .to_numpy()
        )

        assert np.array_equal(
            np.sort(
                label_token_df["label_id"]
                .astype(int)
                .to_numpy()
            ),
            np.arange(spec["num_labels"]),
        )

        causal_plan = build_causal_dps_plan(
            task_key=task_key,
            split=split,
            demo_position=demo_position,
            n_queries_per_label=n_queries_per_label,
        )

        if verify_prefix_alignment:
            verify_all_causal_label_boundaries(
                task_key,
                causal_plan,
                verbose=show_tables,
            )

        print(
            f"Causal prompt conditions: {len(causal_plan)} | "
            f"queries={causal_plan['query_global_id'].nunique()}"
        )

        n_batches = math.ceil(
            len(causal_plan) / int(batch_size)
        )

        for b_idx, start in enumerate(
            range(
                0,
                len(causal_plan),
                int(batch_size),
            ),
            start=1,
        ):
            batch = causal_plan.iloc[
                start : start + int(batch_size)
            ]

            prefixes = []
            batch_steering_positions = []
            meta = []

            for row in batch.itertuples(index=False):
                demos = [
                    TASK_DATA[task_key]["demo"][int(i)]
                    for i in tuple(
                        map(int, row.demo_indices)
                    )
                ]

                query_example = (
                    TASK_DATA[task_key]["query"][
                        int(row.query_index)
                    ]
                )

                (
                    prefix,
                    steering_spans,
                ) = build_causal_prefix_and_steering_spans(
                    task_key,
                    demos,
                    query_example,
                )

                token_positions = (
                    _steering_positions_for_mode(
                        steering_spans,
                        steering_mode,
                    )
                )

                prefixes.append(
                    prefix
                )

                batch_steering_positions.append(
                    token_positions
                )

                meta.append(
                    row
                )

            baseline_vocab_logits = (
                _causal_last_position_logits(
                    prefixes,
                    steering_positions=None,
                    direction=None,
                    alpha=0.0,
                )
            )

            steered_vocab_logits = (
                _causal_last_position_logits(
                    prefixes,
                    steering_positions=(
                        batch_steering_positions
                    ),
                    direction=direction,
                    alpha=float(alpha),
                )
            )

            baseline_candidate_logits = (
                baseline_vocab_logits[
                    :,
                    candidate_token_ids,
                ]
            )

            steered_candidate_logits = (
                steered_vocab_logits[
                    :,
                    candidate_token_ids,
                ]
            )

            targets = np.array(
                [
                    int(row.query_label_id)
                    for row in meta
                ],
                dtype=int,
            )

            baseline_margin = _correct_label_margin(
                baseline_candidate_logits,
                targets,
            )

            steered_margin = _correct_label_margin(
                steered_candidate_logits,
                targets,
            )

            delta_margin = (
                steered_margin
                - baseline_margin
            )

            baseline_prob = (
                _candidate_correct_probability(
                    baseline_candidate_logits,
                    targets,
                )
            )

            steered_prob = (
                _candidate_correct_probability(
                    steered_candidate_logits,
                    targets,
                )
            )

            delta_prob = (
                steered_prob
                - baseline_prob
            )

            baseline_pred = (
                baseline_candidate_logits.argmax(axis=1)
            )

            steered_pred = (
                steered_candidate_logits.argmax(axis=1)
            )

            for local_i, row in enumerate(meta):
                y = int(row.query_label_id)
                c = int(row.varied_label_id)

                all_rows.append({
                    "task_key": task_key,
                    "task": spec["display_name"],
                    "query_global_id": int(row.query_global_id),
                    "query_label_id": y,
                    "query_label": spec["label_to_word"][y],
                    "varied_label_id": c,
                    "varied_label": spec["label_to_word"][c],
                    "is_matched_condition": bool(c == y),
                    "demo_position": int(row.varied_position),
                    "split": str(row.split),

                    "baseline_correct_margin": float(
                        baseline_margin[local_i]
                    ),
                    "steered_correct_margin": float(
                        steered_margin[local_i]
                    ),
                    "delta_correct_margin": float(
                        delta_margin[local_i]
                    ),

                    "baseline_correct_candidate_prob": float(
                        baseline_prob[local_i]
                    ),
                    "steered_correct_candidate_prob": float(
                        steered_prob[local_i]
                    ),
                    "delta_correct_candidate_prob": float(
                        delta_prob[local_i]
                    ),

                    "baseline_pred_label_id": int(
                        baseline_pred[local_i]
                    ),
                    "steered_pred_label_id": int(
                        steered_pred[local_i]
                    ),
                    "baseline_correct": bool(
                        baseline_pred[local_i] == y
                    ),
                    "steered_correct": bool(
                        steered_pred[local_i] == y
                    ),

                    "features": ",".join(
                        map(str, features)
                    ),
                    "alpha": float(alpha),
                    "combine_mode": combine_mode,
                    "steering_mode": steering_mode,
                })

            if (
                b_idx % 25 == 0
                or b_idx == n_batches
            ):
                print(
                    f"[{task_key}] batch "
                    f"{b_idx}/{n_batches}"
                )

    prompt_result_df = pd.DataFrame(
        all_rows
    )

    # ========================================================
    # Aggregate Causal DPS-C matrices and equations
    # ========================================================
    equation_rows = []

    for task_key in task_keys:
        spec = TASK_SPECS[task_key]

        task_df = prompt_result_df[
            prompt_result_df["task_key"]
            == task_key
        ]

        matrix = (
            task_df.groupby(
                [
                    "query_label_id",
                    "varied_label_id",
                ]
            )["delta_correct_margin"]
            .mean()
            .unstack("varied_label_id")
            .reindex(
                index=spec["label_ids"],
                columns=spec["label_ids"],
            )
        )

        labels = [
            spec["label_to_word"][l]
            for l in spec["label_ids"]
        ]

        print("\n" + "=" * 80)
        print(
            f"{spec['display_name']} mean "
            "Δ correct-label margin"
        )
        if show_tables:
            display(
                matrix.rename(
                    index=spec["label_to_word"],
                    columns=spec["label_to_word"],
                )
            )

        # Symmetric scale around zero:
        # negative = red, zero = white, positive = blue.
        matrix_values = matrix.to_numpy(dtype=float)
        heat_absmax = float(np.nanmax(np.abs(matrix_values)))
        if not np.isfinite(heat_absmax) or heat_absmax < 1e-12:
            heat_absmax = 1.0

        fig = go.Figure(
            data=go.Heatmap(
                z=matrix_values,
                x=labels,
                y=labels,
                zmin=-heat_absmax,
                zmax=heat_absmax,
                zmid=0.0,
                colorscale=CAUSAL_HEATMAP_COLORSCALE,
                colorbar=dict(
                    title="Δ margin"
                ),
                hovertemplate=(
                    "Varied demo=%{x}<br>"
                    "Query=%{y}<br>"
                    "Δ correct margin=%{z:.4f}"
                    "<extra></extra>"
                ),
            )
        )

        fig.update_layout(
            title=dict(
                text=(
                    f"Causal DPS-C raw Δmargin — "
                    f"{spec['display_name']}<br>"
                    f"<sup>features={features}, "
                    f"alpha={alpha}, "
                    f"position={demo_position}, "
                    f"steering={steering_mode}, "
                    f"split={split}</sup>"
                ),
                x=0.0,
                xanchor="left",
            ),
            xaxis_title="Varied demonstration label L'",
            yaxis_title="True query label L",
            font=dict(
                family=PLOT_FONT,
                size=14,
            ),
            width=820,
            height=720,
        )

        if show_plots:
            fig.show()

        for y in spec["label_ids"]:
            row_vals = matrix.loc[
                int(y)
            ].to_numpy(
                dtype=float
            )

            matched_effect = float(
                row_vals[int(y)]
            )

            mismatch_indices = [
                int(c)
                for c in spec["label_ids"]
                if int(c) != int(y)
            ]

            mismatch_vals = row_vals[
                mismatch_indices
            ]

            # DPS-C causal criterion:
            # the matched condition should have the SMALLEST delta margin.
            min_mismatch = float(
                np.min(
                    mismatch_vals
                )
            )

            # Positive specificity gap means:
            #     matched_effect < every mismatched effect.
            causal_gap = (
                min_mismatch
                - matched_effect
            )

            # Ordering criterion only. The formal DPS-C condition does not
            # require the matched effect itself to be negative.
            success = bool(
                causal_gap
                > float(
                    causal_tie_tol
                )
            )

            equation_rows.append({
                "task_key": task_key,
                "task": spec["display_name"],
                "query_label_id": int(y),
                "query_label": spec["label_to_word"][y],

                "matched_delta_margin":
                    matched_effect,

                # DPS-C comparison reference:
                # minimum causal effect among mismatched conditions.
                "min_mismatched_delta_margin":
                    min_mismatch,

                # Compatibility field name retained for older analysis code.
                "best_mismatched_delta_margin":
                    min_mismatch,

                # DPS-C specificity:
                # min(nonmatched) - matched.
                "causal_dps_gap":
                    causal_gap,

                # Useful diagnostic, but NOT part of equation satisfaction.
                "matched_effect_negative":
                    bool(
                        matched_effect
                        < -float(
                            causal_tie_tol
                        )
                    ),

                "causal_dps_satisfied":
                    success,

                "features": ",".join(
                    map(str, features)
                ),
                "alpha": float(alpha),
                "combine_mode": combine_mode,
                "steering_mode": steering_mode,
                "demo_position": int(
                    demo_position
                ),
                "split": split,
            })

    causal_equation_df = pd.DataFrame(
        equation_rows
    )

    n_success = int(
        causal_equation_df[
            "causal_dps_satisfied"
        ].sum()
    )

    print("\n" + "=" * 96)
    print("FINAL CAUSAL DPS-C SUMMARY")
    print(
        "Satisfied equations:",
        f"{n_success}/{len(causal_equation_df)}",
    )
    print(
        "Mean causal DPS-C specificity gap:",
        float(
            causal_equation_df[
                "causal_dps_gap"
            ].mean()
        ),
    )
    print(
        "Mean matched-condition Δ margin:",
        float(
            causal_equation_df[
                "matched_delta_margin"
            ].mean()
        ),
    )

    matched_rows = prompt_result_df[
        prompt_result_df[
            "is_matched_condition"
        ]
    ]
    mismatched_rows = prompt_result_df[
        ~prompt_result_df[
            "is_matched_condition"
        ]
    ]

    print(
        "Prompt-level mean Δ correct margin — matched:",
        float(
            matched_rows[
                "delta_correct_margin"
            ].mean()
        ),
    )
    print(
        "Prompt-level mean Δ correct margin — mismatched:",
        float(
            mismatched_rows[
                "delta_correct_margin"
            ].mean()
        ),
    )

    print(
        "Baseline candidate-label accuracy — matched:",
        float(
            matched_rows[
                "baseline_correct"
            ].mean()
        ),
    )
    print(
        "Steered candidate-label accuracy — matched:",
        float(
            matched_rows[
                "steered_correct"
            ].mean()
        ),
    )
    print(
        "Baseline candidate-label accuracy — mismatched:",
        float(
            mismatched_rows[
                "baseline_correct"
            ].mean()
        ),
    )
    print(
        "Steered candidate-label accuracy — mismatched:",
        float(
            mismatched_rows[
                "steered_correct"
            ].mean()
        ),
    )

    print("\nPer-equation results:")
    if show_tables:
        display(
            causal_equation_df.sort_values(
                [
                    "causal_dps_satisfied",
                    "causal_dps_gap",
                ],
                ascending=[
                    False,
                    False,
                ],
            )
        )

    plot_df = causal_equation_df.copy()
    plot_df["equation"] = (
        plot_df["task"]
        + " — "
        + plot_df["query_label"]
    )

    fig = go.Figure(
        go.Bar(
            x=plot_df["equation"],
            y=plot_df["causal_dps_gap"],
            text=[
                "✓" if x else "✗"
                for x in plot_df[
                    "causal_dps_satisfied"
                ]
            ],
            textposition="outside",
        )
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_width=1,
    )

    fig.update_layout(
        title=dict(
            text=(
                "Causal DPS-C gaps across the 20 "
                "task-label equations"
            ),
            x=0.0,
            xanchor="left",
        ),
        xaxis_title="Task / query label",
        yaxis_title=(
            "Min mismatched Δmargin − "
            "matched Δmargin"
        ),
        xaxis_tickangle=-55,
        font=dict(
            family=PLOT_FONT,
            size=14,
        ),
        width=1250,
        height=620,
        margin=dict(
            b=180,
        ),
        showlegend=False,
    )

    if show_plots:
        fig.show()

    if save_csv:
        feature_slug = "-".join(
            map(str, features)
        )
        alpha_slug = str(alpha).replace(
            ".",
            "p",
        ).replace(
            "-",
            "m",
        )

        prompt_path = (
            CAUSAL_OUTPUT_DIR
            / (
                f"causal_prompt_results_"
                f"F{feature_slug}_"
                f"a{alpha_slug}_"
                f"pos{demo_position}_"
                f"steer-{steering_mode}_"
                f"{split}.csv"
            )
        )

        equation_path = (
            CAUSAL_OUTPUT_DIR
            / (
                f"causal_equations_"
                f"F{feature_slug}_"
                f"a{alpha_slug}_"
                f"pos{demo_position}_"
                f"steer-{steering_mode}_"
                f"{split}.csv"
            )
        )

        prompt_result_df.to_csv(
            prompt_path,
            index=False,
        )
        causal_equation_df.to_csv(
            equation_path,
            index=False,
        )

        print("Saved:", prompt_path)
        print("Saved:", equation_path)

    return (
        prompt_result_df,
        causal_equation_df,
    )

In [26]:
# ============================================================
# Example causal run
# ============================================================
#
# For a quick smoke test, first set:
#     CAUSAL_N_QUERIES_PER_LABEL = 3
#
# For the final camera-ready evaluation, use:
#     CAUSAL_N_QUERIES_PER_LABEL = None
#
# Then uncomment:

# causal_prompt_df, causal_equation_df = run_causal_dps_amplification(
#     feature_indices=CAUSAL_FEATURES,
#     alpha=CAUSAL_ALPHA,
#     demo_position=CAUSAL_DEMO_POSITION,
#     split=CAUSAL_SPLIT,
#     n_queries_per_label=CAUSAL_N_QUERIES_PER_LABEL,
#     task_keys=CAUSAL_TASK_KEYS,
#     batch_size=CAUSAL_BATCH_SIZE,
#     combine_mode=CAUSAL_COMBINE_MODE,
#     causal_tie_tol=CAUSAL_TIE_TOL,
#     steering_mode=CAUSAL_STEERING_MODE,
# )

# Available steering spans:
#     "last_token"
#     "test_query"
#     "first_demo_only"
#     "first_demo_and_test_query"


## 16.5. Single-feature causal \(L \times L'\) heatmap

For qualitative inspection of an individual DPS feature, the helper below takes
a **feature index** and **task name** and runs the same causal intervention for
that single feature. It plots

\[
\delta M_L(L')
=
M_L(L';\alpha)-M_L(L';0),
\]

with:

- rows = true query label \(L\);
- columns = varied first-demonstration label \(L'\);
- **blue** = positive change in the correct-label margin;
- **white** = zero;
- **red** = negative change.

This reproduces the feature-wise causal matrix used in the qualitative figures.

In [27]:
def _resolve_task_key(task_name: str) -> str:
    """
    Accept either internal task keys ("agnews", "trec", "yahoo")
    or display names ("AGNews", "TREC", "Yahoo"), case-insensitively.
    """
    name = str(task_name).strip().lower()

    aliases = {
        "agnews": "agnews",
        "ag news": "agnews",
        "trec": "trec",
        "yahoo": "yahoo",
        "yahoo answers": "yahoo",
        "yahoo answers topics": "yahoo",
    }

    for task_key in TASK_ORDER:
        aliases[
            TASK_SPECS[task_key]["display_name"].lower()
        ] = task_key

    if name not in aliases:
        raise ValueError(
            f"Unknown task name {task_name!r}. "
            f"Use one of: AGNews, TREC, Yahoo."
        )

    return aliases[name]


def plot_single_feature_causal_heatmap(
    feature_idx: int,
    task_name: str,
    *,
    alpha: float = CAUSAL_ALPHA,
    demo_position: int = CAUSAL_DEMO_POSITION,
    split: str = CAUSAL_SPLIT,
    n_queries_per_label: Optional[int] = CAUSAL_N_QUERIES_PER_LABEL,
    batch_size: int = CAUSAL_BATCH_SIZE,
    combine_mode: str = CAUSAL_COMBINE_MODE,
    causal_tie_tol: float = CAUSAL_TIE_TOL,
    verify_prefix_alignment: bool = True,
    steering_mode: str = CAUSAL_STEERING_MODE,

    # Optional fixed color saturation.
    # If both are None, use an automatic symmetric range.
    color_min: Optional[float] = None,
    color_max: Optional[float] = None,

    width: int = 820,
    height: int = 720,
):
    """
    Run and plot the feature-wise causal matrix for one SAE feature / task.

    Cell (L, L') =
        δM_L(L')
        = M_L(L'; alpha) - M_L(L'; 0)

    Positive -> blue
    Zero     -> white
    Negative -> red

    DPS-C interpretation:
        For each fixed query label L, an ideal causal DPS-C feature has
        the SMALLEST matched value δM_L(L) across varied first-demo labels.
        Under positive amplification this often appears as a strongly red
        matched diagonal, but negativity itself is not mathematically required.

    Returns
    -------
    prompt_result_df
        Prompt-level causal results for this feature/task.

    causal_equation_df
        Per-query-label causal-DPS summary.

    matrix_df
        Query-label × varied-demo-label mean delta-margin matrix.

    fig
        Plotly heatmap figure.
    """
    feature_idx = int(feature_idx)
    task_key = _resolve_task_key(task_name)
    spec = TASK_SPECS[task_key]

    if not (0 <= feature_idx < int(sae.n_features)):
        raise ValueError(
            f"feature_idx must be in [0, {sae.n_features - 1}], "
            f"got {feature_idx}."
        )

    # Run ONLY the requested feature and task.
    prompt_result_df, causal_equation_df = (
        run_causal_dps_amplification(
            feature_indices=[feature_idx],
            alpha=float(alpha),
            demo_position=int(demo_position),
            split=str(split),
            n_queries_per_label=n_queries_per_label,
            task_keys=[task_key],
            batch_size=int(batch_size),
            combine_mode=str(combine_mode),
            causal_tie_tol=float(causal_tie_tol),
            save_csv=False,
            show_plots=False,
            show_tables=False,
            verify_prefix_alignment=bool(
                verify_prefix_alignment
            ),
            steering_mode=steering_mode,
        )
    )

    task_df = prompt_result_df[
        prompt_result_df["task_key"] == task_key
    ].copy()

    matrix = (
        task_df
        .groupby(
            [
                "query_label_id",
                "varied_label_id",
            ]
        )["delta_correct_margin"]
        .mean()
        .unstack("varied_label_id")
        .reindex(
            index=spec["label_ids"],
            columns=spec["label_ids"],
        )
    )

    labels = [
        spec["label_to_word"][int(l)]
        for l in spec["label_ids"]
    ]

    matrix_df = matrix.rename(
        index=spec["label_to_word"],
        columns=spec["label_to_word"],
    )

    z = matrix.to_numpy(dtype=float)

    # --------------------------------------------------------
    # Color range
    # --------------------------------------------------------
    if color_min is None and color_max is None:
        absmax = float(
            np.nanmax(np.abs(z))
        )

        if (
            not np.isfinite(absmax)
            or absmax < 1e-12
        ):
            absmax = 1.0

        zmin = -absmax
        zmax = absmax
        zero_position = 0.5

    elif color_min is not None and color_max is not None:
        zmin = float(color_min)
        zmax = float(color_max)

        if not zmin < 0 < zmax:
            raise ValueError(
                "Require color_min < 0 < color_max."
            )

        zero_position = (
            (0.0 - zmin)
            / (zmax - zmin)
        )

    else:
        raise ValueError(
            "Specify both color_min and color_max, "
            "or leave both as None."
        )

    # Publication-friendly:
    # negative = red, zero = white, positive = blue.
    colorscale = [
        [0.00, "#B2182B"],
        [zero_position, "#FFFFFF"],
        [1.00, "#2166AC"],
    ]

    fig = go.Figure(
        data=go.Heatmap(
            z=z,
            x=labels,
            y=labels,
            zmin=zmin,
            zmax=zmax,
            colorscale=colorscale,
            colorbar=dict(
                title="Δ margin",
            ),
            hovertemplate=(
                "Varied first-demo label L'=%{x}<br>"
                "True query label L=%{y}<br>"
                "δM_L(L')=%{z:.4f}"
                "<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title=dict(
            text=(
                f"Causal DPS-C raw Δmargin — {spec['display_name']}"
                f"<br><sup>"
                f"feature={feature_idx}, "
                f"alpha={float(alpha):g}, "
                f"position={int(demo_position)}, "
                f"steering={steering_mode}, "
                f"split={split}"
                f"</sup>"
            ),
            x=0.0,
            xanchor="left",
        ),
        xaxis=dict(
            title="Varied demonstration label L'",
            tickangle=-35,
        ),
        yaxis=dict(
            title="True query label L",
        ),
        font=dict(
            family=PLOT_FONT,
            size=14,
        ),
        width=int(width),
        height=int(height),
        margin=dict(
            l=150,
            r=80,
            t=110,
            b=150,
        ),
    )

    fig.show()

    return (
        prompt_result_df,
        causal_equation_df,
        matrix_df,
        fig,
    )


# ============================================================
# Example
# ============================================================
#
# feature_prompt_df, feature_equation_df, feature_matrix, feature_fig = (
#     plot_single_feature_causal_heatmap(
#         feature_idx=8687,
#         task_name="Yahoo",
#         alpha=10.0,
#         steering_mode=CAUSAL_STEERING_MODE,
#         color_min=-1.5,   # optional
#         color_max=1.5,    # optional
#     )
# )

## 17. Camera-ready Top-20 causal-efficacy heatmap

We first select the **top-20 activation-level DPS-C features** and then test each feature independently by amplifying its decoder direction at the **verified pre-label intervention site**.

For feature \(f\) and query label \(L\), the recommended main-table cell is

\[
\delta M_{f,L}(L)
=
M_{f,L}(L;\alpha)-M_{f,L}(L;0),
\]

where the varied first-demo label is also \(L\). Thus each cell directly reports whether amplifying feature \(f\) increases the correct-label decision margin under the expected matched condition.

- **Blue**: positive causal efficacy.
- **White**: approximately zero.
- **Red**: negative causal efficacy.

For the stricter matching-specificity control, the same stored results can instead plot

\[
\delta M_{f,L}(L)
-
\max_{L'\neq L}\delta M_{f,L}(L').
\]

No model rerun is needed to switch between these two heatmaps.

In [28]:
import contextlib
import io


def _causal_heatmap_column_order(
    task_keys: Sequence[str] = ("agnews", "trec", "yahoo"),
) -> pd.DataFrame:
    rows = []
    for task_key in task_keys:
        spec = TASK_SPECS[task_key]
        for label_id in spec["label_ids"]:
            rows.append({
                "task_key": task_key,
                "task": spec["display_name"],
                "query_label_id": int(label_id),
                "query_label": spec["label_to_word"][int(label_id)],
                "column_key": f"{task_key}::{int(label_id)}",
                "tick_label": (
                    f"{spec['display_name']}<br>"
                    f"{spec['label_to_word'][int(label_id)]}"
                ),
            })
    return pd.DataFrame(rows)


def plot_top_feature_causal_heatmap(
    all_feature_equation_df: pd.DataFrame,
    *,
    metric: str = "matched_delta_margin",
    feature_order: Optional[Sequence[int]] = None,
    task_keys: Sequence[str] = ("agnews", "trec", "yahoo"),
    title: Optional[str] = None,
    yaxis_title: str = "Activation-level DPS-C features",
    decimals: int = 2,
    width: int = 1800,
    height: Optional[int] = None,
    save_html: Optional[str] = None,
):
    """
    Plot a feature × task-label causal heatmap.

    metric="matched_delta_margin":
        cell = δM_L(L), direct causal efficacy in the MATCHED condition.

    metric="causal_dps_gap":
        cell = min_{L' != L} δM_L(L') - δM_L(L),
        DPS-C causal matching-specificity.

        Positive means the matched condition is the smallest causal effect.
    """
    metric_info = {
        "matched_delta_margin": {
            "title": "Matched-condition causal efficacy: δM_L(L)",
            "colorbar": "δ correct-label margin",
        },
        "causal_dps_gap": {
            "title": "Causal DPS-C matched-as-minimum specificity gap",
            "colorbar": "Min mismatched − matched",
        },
    }

    if metric not in metric_info:
        raise ValueError(
            f"metric must be one of {list(metric_info)}, got {metric!r}"
        )

    required = {
        "feature_idx",
        "task_key",
        "query_label_id",
        "matched_delta_margin",
        "causal_dps_gap",
        "causal_dps_satisfied",
    }
    missing = required - set(all_feature_equation_df.columns)
    if missing:
        raise ValueError(
            f"Missing columns in all_feature_equation_df: {sorted(missing)}"
        )

    order_df = _causal_heatmap_column_order(task_keys)

    df = all_feature_equation_df.copy()
    df["feature_idx"] = df["feature_idx"].astype(int)
    df["column_key"] = (
        df["task_key"].astype(str)
        + "::"
        + df["query_label_id"].astype(int).astype(str)
    )

    if feature_order is None:
        feature_order = list(dict.fromkeys(df["feature_idx"].tolist()))
    else:
        feature_order = [int(f) for f in feature_order]

    column_keys = order_df["column_key"].tolist()
    tick_labels = order_df["tick_label"].tolist()

    matrix_df = (
        df.pivot_table(
            index="feature_idx",
            columns="column_key",
            values=metric,
            aggfunc="first",
        )
        .reindex(
            index=feature_order,
            columns=column_keys,
        )
    )

    gap_df = (
        df.pivot_table(
            index="feature_idx",
            columns="column_key",
            values="causal_dps_gap",
            aggfunc="first",
        )
        .reindex(
            index=feature_order,
            columns=column_keys,
        )
    )

    success_df = (
        df.pivot_table(
            index="feature_idx",
            columns="column_key",
            values="causal_dps_satisfied",
            aggfunc="first",
        )
        .reindex(
            index=feature_order,
            columns=column_keys,
        )
    )

    z = matrix_df.to_numpy(dtype=float)
    absmax = float(np.nanmax(np.abs(z)))
    if not np.isfinite(absmax) or absmax < 1e-12:
        absmax = 1.0

    text = np.empty(z.shape, dtype=object)
    for i in range(z.shape[0]):
        for j in range(z.shape[1]):
            value = z[i, j]
            text[i, j] = (
                ""
                if not np.isfinite(value)
                else f"{value:.{int(decimals)}f}"
            )

    gap_values = gap_df.to_numpy(dtype=float)
    success_values = success_df.to_numpy()
    custom = np.empty(z.shape + (2,), dtype=object)

    for i in range(z.shape[0]):
        for j in range(z.shape[1]):
            custom[i, j, 0] = (
                gap_values[i, j]
                if np.isfinite(gap_values[i, j])
                else np.nan
            )
            custom[i, j, 1] = bool(success_values[i, j])

    if height is None:
        height = max(650, 34 * len(feature_order) + 260)

    if title is None:
        title = metric_info[metric]["title"]

    fig = go.Figure(
        data=go.Heatmap(
            z=z,
            x=tick_labels,
            y=[f"Feature {f}" for f in feature_order],
            zmin=-absmax,
            zmax=absmax,
            zmid=0.0,
            colorscale=CAUSAL_HEATMAP_COLORSCALE,
            text=text,
            texttemplate="%{text}",
            textfont=dict(size=11),
            customdata=custom,
            colorbar=dict(
                title=metric_info[metric]["colorbar"],
                thickness=18,
            ),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "%{x}<br>"
                f"{metric}=%{{z:.4f}}<br>"
                "Causal DPS-C gap=%{customdata[0]:.4f}<br>"
                "Causal DPS satisfied=%{customdata[1]}"
                "<extra></extra>"
            ),
        )
    )

    cumulative = 0
    for task_key in task_keys[:-1]:
        cumulative += TASK_SPECS[task_key]["num_labels"]
        fig.add_vline(
            x=cumulative - 0.5,
            line_width=2,
            line_color="#555555",
        )

    fig.update_layout(
        title=dict(text=title, x=0.0, xanchor="left"),
        xaxis=dict(
            title="Task / query label",
            tickangle=-55,
        ),
        yaxis=dict(
            title=str(yaxis_title),
            autorange="reversed",
        ),
        font=dict(
            family=PLOT_FONT,
            size=13,
        ),
        width=int(width),
        height=int(height),
        margin=dict(
            l=130,
            r=110,
            t=100,
            b=230,
        ),
    )

    fig.show()

    if save_html is not None:
        fig.write_html(str(save_html))
        print("Saved heatmap HTML:", save_html)

    return matrix_df, fig


def run_top_features_causal_heatmap(
    top_features: Sequence[int],
    *,
    alpha: float = 10.0,
    demo_position: int = 1,
    split: str = "holdout",
    n_queries_per_label: Optional[int] = None,
    task_keys: Sequence[str] = ("agnews", "trec", "yahoo"),
    batch_size: int = 4,
    combine_mode: str = "sum",
    causal_tie_tol: float = 1e-9,
    metric: str = "matched_delta_margin",
    save_csv: bool = True,
    verify_prefix_alignment: bool = True,
    plot_every: Optional[int] = 5,
    steering_mode: str = CAUSAL_STEERING_MODE,
    feature_set_label: str = "activation-level DPS-C features",
    yaxis_title: str = "Activation-level DPS-C features",
    output_tag: str = "top",
):
    """
    Run each feature INDIVIDUALLY and build the paper-level feature×label heatmap.

    Parameters
    ----------
    plot_every:
        Plot the current partial feature×label heatmap after every
        `plot_every` completed features. With top-20 and plot_every=5,
        intermediate heatmaps are shown after 5, 10, and 15 features;
        the ordinary final heatmap is shown after feature 20.
        Set to None to disable intermediate plots.
    """
    top_features = [int(f) for f in top_features]
    if len(top_features) == 0:
        raise ValueError("top_features cannot be empty.")

    if plot_every is not None:
        plot_every = int(plot_every)
        if plot_every <= 0:
            raise ValueError("plot_every must be a positive integer or None.")

    print("=" * 100)
    print("TOP-FEATURE CAUSAL DPS-C RUN")
    print("features:", top_features)
    print("alpha:", float(alpha))
    print(
        "intervention:",
        "ADD direction"
        if float(alpha) > 0
        else (
            "SUBTRACT direction"
            if float(alpha) < 0
            else "NO SHIFT"
        ),
    )
    print("heatmap metric:", metric)
    print("steering_mode:", steering_mode)
    print(
        "progress heatmap cadence:",
        "disabled" if plot_every is None else f"every {plot_every} features",
    )
    print("=" * 100)

    all_results = []

    for rank, feat in enumerate(top_features, start=1):
        print(f"[{rank:02d}/{len(top_features):02d}] Feature {feat}")

        # Keep the top-20 loop readable.
        sink = io.StringIO()
        with contextlib.redirect_stdout(sink):
            _, eq_df = run_causal_dps_amplification(
                feature_indices=[feat],
                alpha=float(alpha),
                demo_position=int(demo_position),
                split=split,
                n_queries_per_label=n_queries_per_label,
                task_keys=task_keys,
                batch_size=int(batch_size),
                combine_mode=combine_mode,
                causal_tie_tol=float(causal_tie_tol),
                save_csv=False,
                show_plots=False,
                show_tables=False,
                verify_prefix_alignment=(
                    bool(verify_prefix_alignment)
                    and rank == 1
                ),
                steering_mode=steering_mode,
            )

        eq_df = eq_df.copy()
        eq_df["feature_idx"] = int(feat)
        eq_df["activation_rank"] = int(rank)
        all_results.append(eq_df)

        should_plot_progress = (
            plot_every is not None
            and rank % plot_every == 0
            and rank < len(top_features)
        )

        if should_plot_progress:
            partial_equation_df = pd.concat(
                all_results,
                ignore_index=True,
            )

            completed_features = top_features[:rank]

            print(
                f"\n--- Progress heatmap: "
                f"{rank}/{len(top_features)} features completed ---"
            )

            plot_top_feature_causal_heatmap(
                partial_equation_df,
                metric=metric,
                feature_order=completed_features,
                task_keys=task_keys,
                title=(
                    f"Progress: {rank}/{len(top_features)} "
                    f"{feature_set_label} "
                    f"(α={alpha}, steering={steering_mode})"
                ),
                yaxis_title=yaxis_title,
            )

    all_feature_equation_df = pd.concat(
        all_results,
        ignore_index=True,
    )

    feature_summary = (
        all_feature_equation_df
        .groupby(
            ["activation_rank", "feature_idx"],
            as_index=False,
        )
        .agg(
            n_causal_dps_satisfied=(
                "causal_dps_satisfied",
                "sum",
            ),
            mean_matched_delta_margin=(
                "matched_delta_margin",
                "mean",
            ),
            mean_causal_dps_gap=(
                "causal_dps_gap",
                "mean",
            ),
        )
        .sort_values("activation_rank")
    )

    print("\nPer-feature causal summary:")
    display(feature_summary)

    matrix_df, fig = plot_top_feature_causal_heatmap(
        all_feature_equation_df,
        metric=metric,
        feature_order=top_features,
        task_keys=task_keys,
        title=(
            f"{len(top_features)} {feature_set_label}: "
            f"causal efficacy "
            f"(α={alpha}, steering={steering_mode})"
        ),
        yaxis_title=yaxis_title,
    )

    if save_csv:
        alpha_slug = str(alpha).replace(".", "p").replace("-", "m")
        safe_output_tag = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            str(
                output_tag
            ),
        ).strip("_")

        prefix = (
            f"{safe_output_tag}{len(top_features)}_a{alpha_slug}_"
            f"steer-{steering_mode}"
        )

        eq_path = (
            CAUSAL_OUTPUT_DIR
            / f"{prefix}_individual_causal_equations.csv"
        )
        summary_path = (
            CAUSAL_OUTPUT_DIR
            / f"{prefix}_causal_summary.csv"
        )
        matrix_path = (
            CAUSAL_OUTPUT_DIR
            / f"{prefix}_{metric}_heatmap_matrix.csv"
        )

        all_feature_equation_df.to_csv(eq_path, index=False)
        feature_summary.to_csv(summary_path, index=False)
        matrix_df.to_csv(matrix_path)

        print("Saved:", eq_path)
        print("Saved:", summary_path)
        print("Saved:", matrix_path)

    return (
        all_feature_equation_df,
        feature_summary,
        matrix_df,
        fig,
    )

# test_query, alpha = 50

In [ ]:
# ============================================================
# Camera-ready Top-20 example
# ============================================================

top_features = list(
    selected_feature_df["feature_idx"].head(20)
)

(
    top20_causal_equations,
    top20_causal_summary,
    top20_heatmap_matrix,
    top20_heatmap_fig,
) = run_top_features_causal_heatmap(
    top_features=top_features,
    alpha=50,  # +10 and -10 are both supported
    demo_position=CAUSAL_DEMO_POSITION,
    split=CAUSAL_SPLIT,
    n_queries_per_label=CAUSAL_N_QUERIES_PER_LABEL,
    task_keys=CAUSAL_TASK_KEYS,
    batch_size=CAUSAL_BATCH_SIZE,
    combine_mode=CAUSAL_COMBINE_MODE,
    causal_tie_tol=CAUSAL_TIE_TOL,
    metric="causal_dps_gap",  # causal_dps_gap, matched_delta_margin
    plot_every=1,                   # progress plots after 5, 10, 15; final at 20
    steering_mode=CAUSAL_STEERING_MODE,
)
#
# # Optional stricter matching-specificity heatmap WITHOUT rerunning:
top20_gap_matrix, top20_gap_fig = plot_top_feature_causal_heatmap(
    top20_causal_equations,
    metric="matched_delta_margin",
    feature_order=top_features,
    task_keys=CAUSAL_TASK_KEYS,
    title="Top-20 DPS-C features: matched-condition raw causal effect",
)

## Random non-DPS-C causal control

Run the same causal-specificity experiment on the 10 reproducibly sampled
near-chance control features. This uses exactly the same held-out prompts,
\(\alpha\), steering span, and causal metric as the main DPS-C run. A genuine
DPS-specific causal signature should therefore be substantially weaker or less
consistent in this control population.

In [ ]:
# ============================================================
# Random chance-level non-DPS-C causal control
# ============================================================

(
    random_control_causal_equations,
    random_control_causal_summary,
    random_control_heatmap_matrix,
    random_control_heatmap_fig,
) = run_top_features_causal_heatmap(
    top_features=
        RANDOM_CONTROL_FEATURES,

    alpha=
        100,  # keep identical to the main Top-20 run above

    demo_position=
        CAUSAL_DEMO_POSITION,

    split=
        CAUSAL_SPLIT,

    n_queries_per_label=
        CAUSAL_N_QUERIES_PER_LABEL,

    task_keys=
        CAUSAL_TASK_KEYS,

    batch_size=
        CAUSAL_BATCH_SIZE,

    combine_mode=
        CAUSAL_COMBINE_MODE,

    causal_tie_tol=
        CAUSAL_TIE_TOL,

    metric=
        "causal_dps_gap",

    # Ten controls: plot only the final heatmap by default.
    plot_every=
        None,

    steering_mode=
        CAUSAL_STEERING_MODE,

    feature_set_label=
        "chance-level non-DPS-C control features",

    yaxis_title=
        "Chance-level control features",

    output_tag=
        "random_control",
)

# Direct matched-effect view from the SAME stored control results.
random_control_matched_matrix, random_control_matched_fig = (
    plot_top_feature_causal_heatmap(
        random_control_causal_equations,

        metric=
            "matched_delta_margin",

        feature_order=
            RANDOM_CONTROL_FEATURES,

        task_keys=
            CAUSAL_TASK_KEYS,

        title=(
            "Random chance-level non-DPS-C controls: "
            "matched-condition causal effect"
        ),

        yaxis_title=
            "Chance-level control features",
    )
)

In [ ]:
# ============================================================
# Recommended camera-ready ordering:
# 1. More positive/blue cells first
# 2. Break ties by larger mean causal effect
# ============================================================

feature_order_df = (
    top20_causal_equations
    .groupby("feature_idx")
    .agg(
        n_positive=(
            "causal_dps_gap",
            lambda x: (x > 0).sum(),
        ),
        mean_effect=(
            "causal_dps_gap",
            "mean",
        ),
    )
    .sort_values(
        ["n_positive", "mean_effect"],
        ascending=[False, False],
    )
)

blue_sorted_features = (
    feature_order_df.index
    .astype(int)
    .tolist()
)

display(feature_order_df.reset_index())

top20_blue_sorted_matrix, top20_blue_sorted_fig = (
    plot_top_feature_causal_heatmap(
        top20_causal_equations,
        metric="causal_dps_gap", # causal_dps_gap, matched_delta_margin
        feature_order=blue_sorted_features,
        task_keys=CAUSAL_TASK_KEYS,
        title="Top-20 DPS-C Features: Causal Efficacy",
    )
)

# Final Camera Ready Figure

In [ ]:
# ============================================================
# Camera-ready causal heatmap with USER-CONTROLLED color bounds
#
# Example:
#   color_min = -1.0  -> every value <= -1.0 is darkest red
#   color_max = +1.0  -> every value >= +1.0 is darkest blue
#   zero             -> always white
#
# The NUMBERS shown in cells remain the true values.
# Only the COLORS are clipped/saturated at the chosen bounds.
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from typing import Optional, Sequence


def _causal_heatmap_column_order(
    task_keys: Sequence[str] = ("agnews", "trec", "yahoo"),
) -> pd.DataFrame:
    """
    Canonical ordering of task/label columns.
    """
    rows = []

    for task_key in task_keys:
        spec = TASK_SPECS[task_key]

        for label_id in spec["label_ids"]:
            label = spec["label_to_word"][int(label_id)]

            rows.append({
                "task_key": task_key,
                "task": spec["display_name"],
                "query_label_id": int(label_id),
                "query_label": label,

                "column_key": (
                    f"{task_key}::{int(label_id)}"
                ),

                "tick_label": (
                    f"{spec['display_name']}<br>{label}"
                ),
            })

    return pd.DataFrame(rows)


def plot_top_feature_causal_heatmap(
    all_feature_equation_df: pd.DataFrame,
    *,
    metric: str = "matched_delta_margin",
    feature_order: Optional[Sequence[int]] = None,
    task_keys: Sequence[str] = ("agnews", "trec", "yahoo"),
    title: Optional[str] = None,

    # --------------------------------------------------------
    # USER-CONTROLLED COLOR BOUNDS
    #
    # <= color_min  -> darkest red
    # 0             -> white
    # >= color_max  -> darkest blue
    # --------------------------------------------------------
    color_min: float = -1.0,
    color_max: float = 1.0,

    decimals: int = 2,
    width: int = 1800,
    height: Optional[int] = None,
    save_html: Optional[str] = None,
):
    """
    Plot a feature × task-label causal heatmap.

    metric="matched_delta_margin"
        Causal EFFICACY:
            δM_L(L)

        Positive:
            Steering increases the correct-label margin
            when first-demo label == query label.

    metric="causal_dps_gap"
        Causal DPS-C SPECIFICITY:
            min_{L' != L} δM_L(L')
            -
            δM_L(L)

        Positive:
            The matching first-demo condition has the SMALLEST
            causal effect on the correct-label margin.

    Color behavior
    --------------
    color_min:
        Values <= this are darkest red.

    0:
        Always white.

    color_max:
        Values >= this are darkest blue.

    Importantly, cell text displays the TRUE value even if
    its color is clipped at color_min/color_max.
    """

    # ========================================================
    # Metric definitions
    # ========================================================
    metric_info = {
        "matched_delta_margin": {
            "title": (
                "Matched-Condition Causal Efficacy"
            ),
            "colorbar": (
                "δ correct-label margin"
            ),
        },

        "causal_dps_gap": {
            "title": (
                "Causal DPS-C Matching Specificity"
            ),
            "colorbar": (
                "Min mismatched − matched"
            ),
        },
    }

    if metric not in metric_info:
        raise ValueError(
            f"metric must be one of {list(metric_info)}, "
            f"got {metric!r}"
        )

    # ========================================================
    # Validate color bounds
    # ========================================================
    color_min = float(color_min)
    color_max = float(color_max)

    if not np.isfinite(color_min):
        raise ValueError(
            "color_min must be finite."
        )

    if not np.isfinite(color_max):
        raise ValueError(
            "color_max must be finite."
        )

    if not color_min < 0 < color_max:
        raise ValueError(
            "Require color_min < 0 < color_max. "
            "This ensures that zero can be mapped exactly to white."
        )

    # ========================================================
    # Required columns
    # ========================================================
    required = {
        "feature_idx",
        "task_key",
        "query_label_id",
        "matched_delta_margin",
        "causal_dps_gap",
        "causal_dps_satisfied",
    }

    missing = (
        required
        - set(all_feature_equation_df.columns)
    )

    if missing:
        raise ValueError(
            "Missing columns in all_feature_equation_df: "
            f"{sorted(missing)}"
        )

    # ========================================================
    # Build canonical 20-column order
    # ========================================================
    order_df = _causal_heatmap_column_order(
        task_keys
    )

    df = all_feature_equation_df.copy()

    df["feature_idx"] = (
        df["feature_idx"]
        .astype(int)
    )

    df["column_key"] = (
        df["task_key"].astype(str)
        + "::"
        + df["query_label_id"]
        .astype(int)
        .astype(str)
    )

    # ========================================================
    # Feature ordering
    # ========================================================
    if feature_order is None:
        feature_order = list(
            dict.fromkeys(
                df["feature_idx"].tolist()
            )
        )

    else:
        feature_order = [
            int(f)
            for f in feature_order
        ]

    column_keys = (
        order_df["column_key"]
        .tolist()
    )

    tick_labels = (
        order_df["tick_label"]
        .tolist()
    )

    # ========================================================
    # Main heatmap matrix
    # ========================================================
    matrix_df = (
        df.pivot_table(
            index="feature_idx",
            columns="column_key",
            values=metric,
            aggfunc="first",
        )
        .reindex(
            index=feature_order,
            columns=column_keys,
        )
    )

    # ========================================================
    # Also retain causal-DPS gap for hover information
    # ========================================================
    gap_df = (
        df.pivot_table(
            index="feature_idx",
            columns="column_key",
            values="causal_dps_gap",
            aggfunc="first",
        )
        .reindex(
            index=feature_order,
            columns=column_keys,
        )
    )

    # ========================================================
    # Causal-DPS success for hover
    # ========================================================
    success_df = (
        df.pivot_table(
            index="feature_idx",
            columns="column_key",
            values="causal_dps_satisfied",
            aggfunc="first",
        )
        .reindex(
            index=feature_order,
            columns=column_keys,
        )
    )

    z = matrix_df.to_numpy(
        dtype=float
    )

    # ========================================================
    # CUSTOM COLOR SCALE
    #
    # Plotly colorscale positions must be in [0,1].
    #
    # For symmetric [-1,+1]:
    #     red   -> position 0.0
    #     white -> position 0.5
    #     blue  -> position 1.0
    #
    # For asymmetric [-0.5,+2]:
    #     zero is automatically shifted to the correct position.
    # ========================================================
    zero_position = (
        (0.0 - color_min)
        /
        (color_max - color_min)
    )

    custom_colorscale = [
        # Strong negative
        [0.00, "#B2182B"],

        # Optional softer negative transition
        [
            max(
                0.0,
                zero_position * 0.55
            ),
            "#EF8A8A",
        ],

        # Exactly zero = white
        [
            zero_position,
            "#FFFFFF",
        ],

        # Optional softer positive transition
        [
            min(
                1.0,
                zero_position
                + (1.0 - zero_position) * 0.45
            ),
            "#8CB9E8",
        ],

        # Strong positive
        [1.00, "#2166AC"],
    ]

    # ========================================================
    # Cell text
    #
    # IMPORTANT:
    # These are actual numerical values, NOT clipped values.
    # ========================================================
    text = np.empty(
        z.shape,
        dtype=object,
    )

    for i in range(z.shape[0]):
        for j in range(z.shape[1]):

            value = z[i, j]

            if np.isfinite(value):
                text[i, j] = (
                    f"{value:.{int(decimals)}f}"
                )
            else:
                text[i, j] = ""

    # ========================================================
    # Hover information
    # ========================================================
    gap_values = gap_df.to_numpy(
        dtype=float
    )

    success_values = (
        success_df.to_numpy()
    )

    custom = np.empty(
        z.shape + (2,),
        dtype=object,
    )

    for i in range(z.shape[0]):
        for j in range(z.shape[1]):

            if np.isfinite(
                gap_values[i, j]
            ):
                custom[i, j, 0] = (
                    gap_values[i, j]
                )
            else:
                custom[i, j, 0] = np.nan

            custom[i, j, 1] = bool(
                success_values[i, j]
            )

    # ========================================================
    # Figure height
    # ========================================================
    if height is None:
        height = max(
            650,
            34 * len(feature_order)
            + 260,
        )

    if title is None:
        title = (
            metric_info[metric]["title"]
        )

    # ========================================================
    # Plot
    # ========================================================
    fig = go.Figure(
        data=go.Heatmap(

            # TRUE numerical values
            z=z,

            x=tick_labels,

            y=[
                f"Feature {f}"
                for f in feature_order
            ],

            # =================================================
            # USER-CONTROLLED COLOR SATURATION
            #
            # e.g. [-1,+1]:
            #
            # value <= -1 -> darkest red
            # value = 0   -> white
            # value >= +1 -> darkest blue
            # =================================================
            zmin=color_min,
            zmax=color_max,

            colorscale=custom_colorscale,

            # =================================================
            # Cell numbers
            # =================================================
            text=text,
            texttemplate="%{text}",
            textfont=dict(
                size=11,
            ),

            customdata=custom,

            colorbar=dict(
                title=(
                    metric_info[metric][
                        "colorbar"
                    ]
                ),
                thickness=18,

                # Explicitly show chosen limits + zero
                tickmode="array",
                tickvals=[
                    color_min,
                    0.0,
                    color_max,
                ],
                ticktext=[
                    f"≤ {color_min:g}",
                    "0",
                    f"≥ {color_max:g}",
                ],
            ),

            hovertemplate=(
                "<b>%{y}</b><br>"
                "%{x}<br>"
                f"{metric}=%{{z:.4f}}<br>"
                "Causal DPS gap="
                "%{customdata[0]:.4f}<br>"
                "Causal DPS satisfied="
                "%{customdata[1]}"
                "<extra></extra>"
            ),
        )
    )

    # ========================================================
    # Vertical separators:
    #
    # AGNews | TREC | Yahoo
    # ========================================================
    cumulative = 0

    for task_key in task_keys[:-1]:

        cumulative += (
            TASK_SPECS[task_key][
                "num_labels"
            ]
        )

        fig.add_vline(
            x=cumulative - 0.5,
            line_width=2,
            line_color="#555555",
        )

    # ========================================================
    # Layout
    # ========================================================
    fig.update_layout(

        title=dict(
            text=title,
            x=0.0,
            xanchor="left",
        ),

        xaxis=dict(
            title="Task / Query Label",
            tickangle=-55,
        ),

        yaxis=dict(
            title=(
                "Activation-Level "
                "DPS-C Features"
            ),

            # Feature 1st in list = TOP row
            autorange="reversed",
        ),

        font=dict(
            family=PLOT_FONT,
            size=13,
        ),

        width=int(width),
        height=int(height),

        margin=dict(
            l=130,
            r=120,
            t=100,
            b=230,
        ),
    )

    fig.show()

    # ========================================================
    # Optional HTML output
    # ========================================================
    if save_html is not None:

        fig.write_html(
            str(save_html)
        )

        print(
            "Saved heatmap HTML:",
            save_html,
        )

    return matrix_df, fig


# ============================================================
# EXAMPLE 1:
# CAUSAL EFFICACY HEATMAP
#
# Darkest blue : >= +1.0
# Darkest red  : <= -1.0
# ============================================================

top20_efficacy_matrix, top20_efficacy_fig = (
    plot_top_feature_causal_heatmap(

        top20_causal_equations,

        metric="matched_delta_margin",

        # Use whichever row ordering you prefer:
        feature_order=blue_sorted_features,

        task_keys=CAUSAL_TASK_KEYS,

        title=(
            "Top-20 DPS-C Features: "
            "Causal Efficacy"
        ),

        # --------------------------------------------
        # USER CHOOSES THESE
        # --------------------------------------------
        color_min=-1.0,
        color_max=1.0,
    )
)


# ============================================================
# EXAMPLE 2:
# CAUSAL MATCHING-SPECIFICITY HEATMAP
#
# Can use the SAME scale so the two figures
# are visually directly comparable.
# ============================================================

top20_gap_matrix, top20_gap_fig = (
    plot_top_feature_causal_heatmap(

        top20_causal_equations,

        metric="causal_dps_gap",

        feature_order=blue_sorted_features,

        task_keys=CAUSAL_TASK_KEYS,

        title=(
            f"Causal Intervention Results of Top-20 DPS-C Features Across All Classification Tasks<br>{MODEL_SHORT_NAME}"
#             "Causal Matching Specificity"
        ),

        # --------------------------------------------
        # USER CHOOSES THESE
        # --------------------------------------------
        color_min=-1.0,
        color_max=1.0,
    )
)